# 📘 BỘ CODE (KHÔNG OOP – 100% HÀM `def` THUẦN) – LẬP TRÌNH PHÂN TÍCH DỮ LIỆU (Dữ liệu: Superstore)

**Cách dùng notebook này khi làm bài kiểm tra**
- **Toàn bộ notebook chỉ có hàm `def` thuần**: KHÔNG có `class`, KHÔNG `__init__`/`self`/getter/setter, KHÔNG biến toàn cục bị hàm âm thầm dùng. Dữ liệu và cấu hình đi VÀO hàm bằng **tham số**, kết quả đi RA bằng **`return`** (trang 6–7 của Word: đây là kiểu *lập trình thủ tục*).
- Mỗi **thuật toán = 1 nhóm hàm**. Mỗi nhóm hàm có phần ghi chú đầu tiên gồm:
  - 📖 **TRANG**: trang tương ứng trong file Word *NOTES_LẬP_TRÌNH_PHÂN_TÍCH_DỮ_LIỆU.docx* (25 trang)
  - 🎯 **DÙNG KHI**: trường hợp sử dụng thuật toán
- Code được viết **cơ bản nhất có thể** (vòng lặp / công thức tay + so sánh với thư viện) để dễ đọc, dễ chép.
- Quy tắc vàng của notes (trang 11): **KHÔNG CHỈNH SỬA DỮ LIỆU THÔ** → mọi hàm đều làm việc trên `.copy()`.

> ⚠️ Số trang được đếm theo bản PDF xuất từ file Word (25 trang). Mở bằng Word có thể lệch ±1 trang.

**Mục lục**
| Phần | Nội dung | Trang notes |
|---|---|---|
| 0 | Cài đặt, nạp dữ liệu | – |
| 1 | Cấu trúc dữ liệu, Vector, Ma trận, Gauss, LU, QR | 1–3, 24–25 |
| 2 | Tập tin text, Regex, Series, DataFrame | 3–6 |
| 3 | Quy ước viết hàm thuần & hàm tiện ích dùng chung | 6–7, 11 |
| 4 | Tiền xử lý & EDA (lướt dữ liệu, lấy mẫu, đơn biến, đa biến) | 7–10 |
| 5 | Dữ liệu bị thiếu (7 biện pháp + MICE) | 11–12 |
| 6 | Dữ liệu bất thường (outliers) | 13–14 |
| 7 | Kiểm định giả thuyết | 15, 17 |
| 8 | Tích hợp dữ liệu | 10, 16–17 |
| 9 | Biến đổi dữ liệu (chuẩn hóa, rời rạc hóa, làm trơn…) | 17–19, 21 |
| 10 | Mã hóa dữ liệu phân loại | 19 |
| 11 | Thu gọn dữ liệu (lấy mẫu, chọn thuộc tính, PCA, SVD) | 20–21 |
| 12 | Thống kê mô tả | 21–23 |
| 13 | **Pipeline ĐẦY ĐỦ** (các hàm `step_*` + `run_pipeline`) | 11, 15–16 |

---
### 🔖 Cách đọc phần ghi chú đã bổ sung trong từng nhóm hàm
- **📚 LÝ THUYẾT [Word]**: lý thuyết lấy **nguyên từ file Word** (ưu tiên file Word).
- **🔎 CHỌN KHI NÀO**: *dữ liệu thế nào / kết quả nhìn thấy thế nào thì chọn công cụ nào* – phần quan trọng nhất khi làm bài.
- **[+bổ sung]**: Word không có, mình bổ sung từ kiến thức chuẩn (ví dụ ngưỡng 0.8 cho tương quan, ngưỡng 80–90% cho PCA) – nên đối chiếu lại với slide thầy.
- **(đã chạy)**: số liệu thật lấy từ output của chính notebook này (Superstore) để làm ví dụ minh họa cho dấu hiệu.
- Mọi hàm đều **nhận tham số tường minh** (tên cột có giá trị mặc định của Superstore, đổi được khi gọi). Cuối notebook có **BẢNG TRA NHANH: DẤU HIỆU → CÔNG CỤ**.


## PHẦN 0 – CÀI ĐẶT & NẠP DỮ LIỆU

In [ ]:
# ============================================================
# 0.1 IMPORT THƯ VIỆN (chỉ dùng thư viện phổ biến)
# ============================================================
import re                      # Regular Expression (trang 4-5)
import math                    # căn bậc 2, log ... cho công thức tay
import hashlib                 # hàm băm cho Feature Hashing (trang 19)
import numpy as np             # mảng, ma trận, đại số tuyến tính (trang 1-3)
import pandas as pd            # Series, DataFrame (trang 5-6)
import matplotlib.pyplot as plt  # vẽ biểu đồ (histogram, boxplot...)
from scipy import stats        # kiểm định giả thuyết, boxcox, yeojohnson (trang 14, 15, 17)
import scipy.linalg as sla     # lu, lu_factor (trang 24)

from IPython.display import display   # hàm display() hiện bảng đẹp trong notebook
import warnings
warnings.filterwarnings("ignore")   # tắt cảnh báo cho gọn output

pd.set_option("display.width", 200)          # in bảng rộng hơn
pd.set_option("display.max_columns", 30)     # hiện nhiều cột hơn

In [ ]:
# ============================================================
# 0.2 CÁC HÀM NẠP DỮ LIỆU: load_data / get_copy
# ------------------------------------------------------------
# 📖 TRANG 5: "Xử lý tập tin Excel (.XLS, .XLSX), CSV, TSV – dùng pandas"
# 🎯 DÙNG KHI: đầu mọi bài làm – đọc file rồi GIỮ BẢN GỐC (raw) không sửa
#              (trang 11: tiền xử lý KHÔNG CHỈNH SỬA DỮ LIỆU THÔ).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Xử lý tập tin Excel]:
#   - Tập tin bảng tính: .XLS/.XLSX ; tập tin .CSV (Comma-Separated Values), .TSV (Tab-Separated Values).
#   - Dùng thư viện pandas: Series (1 chiều) và DataFrame (2 chiều).
#   - Mở/đóng file thực chất là HĐH chép file từ đĩa cứng lên RAM; chỉ khi SAVE mới ghi lại đĩa.
# 🔎 CHỌN KHI NÀO:
#   - File có đuôi .xlsx/.xls (nhiều sheet)      → pd.read_excel(path, sheet_name="tên sheet")
#   - File .csv (ngăn cách dấu phẩy)             → pd.read_csv(path, sep=",")
#   - File .tsv (ngăn cách Tab)                  → pd.read_csv(path, sep="\t")
#   - DẤU HIỆU ĐỌC SAI SEP: sau khi đọc, df chỉ có 1 cột dài dính hết các giá trị → đổi tham số sep.
#   - Dữ liệu nằm ở nhiều sheet (Orders/Returns/Users) → đọc từng sheet riêng rồi TÍCH HỢP (Phần 8).
# ============================================================
def load_data(path, sheet="Orders"):
    # pd.read_excel: đọc 1 sheet của file Excel thành DataFrame
    return pd.read_excel(path, sheet_name=sheet)

def get_copy(raw):
    # Luôn trả về BẢN SAO để các bước xử lý không làm hỏng dữ liệu thô
    return raw.copy()

def read_other_sheet(path, sheet):
    # Đọc các sheet khác (Returns, Users) để dùng cho phần tích hợp dữ liệu
    return pd.read_excel(path, sheet_name=sheet)

def read_csv_example(csv_path, sep=","):
    # .CSV: sep="," ; .TSV: sep="\t"
    return pd.read_csv(csv_path, sep=sep)

# --- Sử dụng ---
DATA_PATH = "Superstore.xlsx"       # 👉 ĐỔI đường dẫn nếu file nằm chỗ khác (Colab: upload rồi để nguyên tên)
raw_df = load_data(DATA_PATH, "Orders")      # dữ liệu thô (KHÔNG sửa)
df = get_copy(raw_df)                        # bản làm việc
print("Kích thước:", df.shape)
df.head()

## PHẦN 1 – CẤU TRÚC DỮ LIỆU, VECTOR, MA TRẬN (Trang 1–3, 24–25)

In [ ]:
# ============================================================
# 1.1 NHÓM HÀM: List & Dictionary
# ------------------------------------------------------------
# 📖 TRANG 1: "3. Cấu trúc dữ liệu tổng hợp: List [], Dictionary {}"
# 🎯 DÙNG KHI: lưu dãy giá trị (list) hoặc cặp key:value (dict, nền tảng của DataFrame).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Cấu trúc dữ liệu tổng hợp]:
#   - LIST  []: phần tử cách nhau dấu phẩy; là DÃY có thứ tự, chỉ số bắt đầu từ 0 do Python quy định (không tự đặt được).
#       · Kỹ thuật cho phép nhiều kiểu dữ liệu, nhưng KỶ LUẬT lập trình KHÔNG nên trộn nhiều kiểu.
#       · insert(<vị trí>, <giá trị>) chèn ; append(<giá trị>) thêm cuối ; remove(<giá trị>) xóa giá trị GẶP ĐẦU TIÊN.
#   - DICTIONARY {}: mỗi phần tử <key>:<value>, key là phân biệt (duy nhất) → cho phép TỰ ĐỊNH NGHĨA chỉ số. Nền của pandas.
# 🔎 CHỌN KHI NÀO:
#   - Dữ liệu là 1 dãy giá trị có thứ tự, truy cập theo VỊ TRÍ (0,1,2…)            → LIST
#   - Dữ liệu có TÊN/NHÃN để tra cứu (tên cột → dữ liệu cột, mã → giá trị)         → DICT
#   - Muốn tạo DataFrame từ các cột có tên: {"Region": [...], "Sales": [...]}       → dict của các list
#   - Cần xóa theo ĐIỀU KIỆN/giá trị → remove (chỉ xóa lần gặp đầu); xóa theo vị trí → del / pop(i).
# ============================================================
def demo_list():
    ds = [10, 20, 30]
    ds.append(40)          # .append(giá trị): thêm vào cuối
    ds.insert(1, 15)       # .insert(vị trí, giá trị): chèn vào vị trí 1
    ds.remove(30)          # .remove(giá trị): xóa phần tử GẶP ĐẦU TIÊN có giá trị đó
    print("List:", ds, "| phần tử đầu:", ds[0], "| phần tử cuối:", ds[-1])
    return ds

def demo_dict():
    d = {"Region": "East", "Sales": 1000}
    d["Profit"] = 250                    # thêm key mới
    print("Dict:", d, "| Sales =", d["Sales"])
    for k, v in d.items():               # duyệt từng cặp key:value
        print("  ", k, "->", v)
    return d

demo_list()
demo_dict()

In [ ]:
# ============================================================
# 1.2 NHÓM HÀM: Tích vô hướng, Chuẩn, Khoảng cách, Cosine
# ------------------------------------------------------------
# 📖 TRANG 1-2: "Đại số ma trận: tích vô hướng, chuẩn vector L1/L2/Lmax, Minkowski"
# 🎯 DÙNG KHI:
#   - Tích vô hướng: tính TỔNG TIỀN = vector số lượng · vector đơn giá (trang 1)
#   - Cosine: đo độ TƯƠNG ĐỒNG giữa 2 đối tượng (Recommender system, trang 2)
#   - Khoảng cách: so sánh độ xa/gần giữa 2 điểm (A*, kNN, outlier)
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Đại số ma trận / Vector]:
#   - Vector = đơn vị cơ sở biểu diễn 1 đối tượng thực tế hoặc 1 khái niệm trừu tượng. Số phần tử = SỐ CHIỀU của không gian;
#     thứ tự các chiều không quan trọng NHƯNG phải CỐ ĐỊNH (pixel của ảnh cũng là ma trận các vector).
#   - Tích vô hướng <u,v> = Σ ui·vi = |u|·|v|·cos(u,v).  Python: v1 @ v2  hoặc  v1.dot(v2).
#   - Chuẩn vector = độ dài vector: L1 = Σ|ui| (tổng trị tuyệt đối) ; L2 (Euclid) = căn(Σ ui²) ; Lmax = max|ui|.
#   - Khoảng cách tổng quát = MINKOWSKI: p=1 → Manhattan ; p=2 → Euclid ; p=max → Chebyshev (Chess Board).
#   - Thuật toán A*: tính khoảng cách Euclid để xếp thứ tự gần/xa, rồi tùy đề bài mà dùng Manhattan hay Chebyshev.
#   - ⚠ Ghi chú Word ở mục "Tọa độ Đề-các" mô tả Chebyshev/Manhattan hơi lẫn. Định nghĩa CHUẨN (code bên dưới dùng đúng định nghĩa này):
#       Manhattan = |Δcột| + |Δdòng| (đi theo lưới, 4 hướng) ; Chebyshev = max(|Δcột|, |Δdòng|) (đi được cả đường chéo như quân vua cờ vua);
#       Euclid = đường thẳng ngắn nhất.
# 🔎 CHỌN KHI NÀO:
#   - Có 2 dãy CÙNG CHIỀU (số lượng × đơn giá) và cần MỘT con số tổng (tổng tiền) → TÍCH VÔ HƯỚNG.
#   - Muốn biết 2 đối tượng GIỐNG NHAU về CƠ CẤU/HƯỚNG (không quan tâm quy mô lớn nhỏ) → COSINE.
#       Kết quả cos GẦN 1 → rất tương đồng (cùng cơ cấu) ; GẦN 0 → không liên quan  [Word: thang 0–1, dùng cho Recommender system]
#       (đã chạy) cos(East, West) = 0.9631 → 2 vùng có cơ cấu doanh số theo loại hàng gần như giống nhau.
#   - Cần ĐO ĐỘ XA GẦN giữa 2 điểm:
#       · các chiều CÙNG ĐƠN VỊ, đi đường thẳng            → Euclid (p=2)
#       · chỉ đi được theo lưới/cộng dồn độ lệch từng chiều → Manhattan (p=1)
#       · chi phí = chiều lệch LỚN NHẤT / đi được 8 hướng    → Chebyshev (p=max)
#   - Cần "độ dài" 1 vector: muốn cộng tuyệt đối (L1), độ dài thông thường (L2), hay chỉ lấy thành phần lớn nhất (Lmax).
#   - [+bổ sung] Khoảng cách Euclid/Manhattan nhạy với ĐƠN VỊ → các chiều khác thang đo phải CHUẨN HÓA trước (Phần 9).
# ============================================================
# ----- Tích vô hướng: <u,v> = Σ ui*vi
def dot(u, v):
    u = np.array(u, dtype=float); v = np.array(v, dtype=float)
    total = 0
    for i in range(len(u)):              # cách tay: nhân từng cặp rồi cộng
        total += u[i] * v[i]
    return total                         # (cách numpy: u @ v  hoặc  u.dot(v))

# ----- Chuẩn L1 = Σ|ui| ; L2 = căn(Σ ui²) ; Lmax = max|ui|
def norm_l1(x):   return np.sum(np.abs(np.array(x, dtype=float)))
def norm_l2(x):   return math.sqrt(np.sum(np.array(x, dtype=float) ** 2))
def norm_lmax(x): return np.max(np.abs(np.array(x, dtype=float)))

# ----- Minkowski tổng quát: (Σ|ui-vi|^p)^(1/p) ; p=1 Manhattan ; p=2 Euclid ; p=∞ Chebyshev
def minkowski(u, v, p):
    diff = np.abs(np.array(u, dtype=float) - np.array(v, dtype=float))
    return np.sum(diff ** p) ** (1 / p)

def manhattan(u, v): return minkowski(u, v, 1)                                   # p = 1
def euclid(u, v):    return minkowski(u, v, 2)                                   # p = 2
def chebyshev(u, v): return np.max(np.abs(np.array(u, dtype=float) - np.array(v, dtype=float)))   # p = max

# ----- Cosine: cos(u,v) = <u,v> / (|u|.|v|)  (gần 1 → rất giống ; gần 0 → không liên quan)
def cosine(u, v):
    return dot(u, v) / (norm_l2(u) * norm_l2(v))

# --- Sử dụng 1: TỔNG TIỀN = số lượng · đơn giá (5 đơn đầu, bỏ dòng thiếu) ---
tmp = df[["Order Quantity", "Unit Price"]].dropna().head(5)
print("Tổng tiền (tích vô hướng):", round(dot(tmp["Order Quantity"], tmp["Unit Price"]), 2),
      "| numpy:", round(np.array(tmp["Order Quantity"]) @ np.array(tmp["Unit Price"]), 2))

# --- Sử dụng 2: Cosine giữa 2 vùng theo cơ cấu doanh số (Furniture, Office, Technology) ---
cat_sales = df.pivot_table(index="Region", columns="Product Category", values="Sales", aggfunc="sum")
print("Cosine East vs West:", round(cosine(cat_sales.loc["East"], cat_sales.loc["West"]), 4))

# --- Sử dụng 3: Các chuẩn & khoảng cách ---
print("L1 =", norm_l1([3, -4]), "| L2 =", norm_l2([3, -4]), "| Lmax =", norm_lmax([3, -4]))
print("Manhattan =", manhattan([1, 2], [4, 6]), "| Euclid =", euclid([1, 2], [4, 6]), "| Chebyshev =", chebyshev([1, 2], [4, 6]))

In [ ]:
# ============================================================
# 1.3 NHÓM HÀM: Ma trận, chuyển vị, định thức, nghịch đảo, hạng, GAUSS
# ------------------------------------------------------------
# 📖 TRANG 2-3: "Ma trận (matrix), chuyển vị, Gauss, hạng, định thức"
# 🎯 DÙNG KHI:
#   - Định thức ≠ 0 → ma trận VUÔNG khả nghịch (trang 3)
#   - Gauss: đưa về ma trận tam giác trên rồi THẾ LÙI để giải hệ A·x = b (trang 2-3)
#   - Hạng = số dòng KHÔNG toàn 0 của dạng bậc thang (trang 3)
#   - Ma trận chữ nhật: dùng GIẢ NGHỊCH ĐẢO (pinv) (trang 3)
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Ma trận, Định thức]:
#   - Ma trận cấp (m,n): đếm từ 1, bắt đầu góc trên-trái, DÒNG TRƯỚC – CỘT SAU. Ma trận = tập m vector trong n chiều;
#     thể hiện MỐI TƯƠNG QUAN giữa 2 tập hợp/đại lượng (thường hình chữ nhật). Ma trận VUÔNG = tương quan NỘI TẠI giữa các đối tượng
#     (ví dụ ma trận khoảng cách).
#   - Cài đặt: mảng 2 chiều, dùng numpy.
#   - Chuyển vị: để thỏa điều kiện số cột/số dòng khi NHÂN 2 ma trận.
#   - Gauss (phương pháp khử): biến đổi dòng đưa về ma trận TAM GIÁC trên/dưới, rồi THẾ LÙI để tìm các biến.
#   - Dạng bậc thang → đếm số dòng KHÔNG toàn số 0 = HẠNG của ma trận.
#   - Định thức: chỉ cho ma trận VUÔNG; det ≠ 0 ⇔ KHẢ NGHỊCH. Ma trận nghịch đảo cũng chỉ cho ma trận vuông.
#     Ma trận chữ nhật → dùng GIẢ NGHỊCH ĐẢO. Phép chia = nhân với nghịch đảo A⁻¹ (nhóm/trường chỉ có phép +, ×).
# 🔎 CHỌN KHI NÀO:
#   - Ma trận VUÔNG và muốn biết có nghịch đảo được không → tính DETERMINANT:
#       det ≠ 0 → khả nghịch → dùng inverse / giải hệ A·x=b ;  det = 0 (hoặc ≈ 0) → KHÔNG khả nghịch → đừng gọi inv (lỗi Singular matrix).
#   - Ma trận CHỮ NHẬT (số dòng ≠ số cột, ví dụ n quan sát × k biến) → KHÔNG có det/inverse → dùng GIẢ NGHỊCH ĐẢO (pinv).
#   - Giải hệ phương trình tuyến tính A·x = b → GAUSS (khử + thế lùi) ; kiểm tra bằng np.linalg.solve.
#   - Muốn biết có bao nhiêu cột/dòng ĐỘC LẬP → HẠNG: hạng < số cột ⇒ có cột phụ thuộc tuyến tính (dư thừa).  [+bổ sung]
#   - Nhân 2 ma trận bị lỗi shape → kiểm tra (m,n)@(n,p) ; cần thì CHUYỂN VỊ.
# ============================================================
def transpose(A):
    return np.array(A, dtype=float).T                    # chuyển vị: đổi dòng ↔ cột

def multiply(A, B):
    return np.array(A, dtype=float) @ np.array(B)        # nhân ma trận: số cột A = số dòng B

def determinant(A):
    return np.linalg.det(np.array(A, dtype=float))       # chỉ cho ma trận VUÔNG

def is_invertible(A):
    return abs(determinant(A)) > 1e-9                    # det ≠ 0 → khả nghịch

def inverse(A):
    return np.linalg.inv(np.array(A, dtype=float))       # A^-1 (chỉ khi khả nghịch). Phép chia = nhân nghịch đảo

def pseudo_inverse(A):
    return np.linalg.pinv(np.array(A, dtype=float))      # giả nghịch đảo cho ma trận chữ nhật

def rank(A):
    return np.linalg.matrix_rank(np.array(A, dtype=float))   # hạng ma trận

# ----- Phương pháp GAUSS (khử) + thế lùi: giải A x = b
def gauss_solve(A, b):
    A = np.array(A, dtype=float)
    n = len(A)
    M = np.hstack([A, np.array(b, dtype=float).reshape(-1, 1)])   # ma trận mở rộng [A | b]
    # B1: KHỬ → đưa về tam giác trên
    for i in range(n):
        pivot = i + np.argmax(np.abs(M[i:, i]))   # chọn dòng có phần tử lớn nhất làm trụ (tránh chia 0)
        M[[i, pivot]] = M[[pivot, i]]             # hoán đổi dòng
        for j in range(i + 1, n):                 # khử các dòng bên dưới
            factor = M[j, i] / M[i, i]
            M[j] = M[j] - factor * M[i]
    # B2: THẾ LÙI từ dòng cuối lên
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, -1] - np.dot(M[i, i + 1:n], x[i + 1:n])) / M[i, i]
    return x

# --- Sử dụng: giải hệ 2x + y = 5 ; x + 3y = 10 ---
m = [[2, 1], [1, 3]]
print("Định thức:", round(determinant(m), 3), "| Khả nghịch:", is_invertible(m))
print("Nghịch đảo:\n", inverse(m).round(3))
print("Gauss x =", gauss_solve(m, [5, 10]), "| numpy:", np.linalg.solve(np.array(m, dtype=float), [5, 10]))
print("Hạng:", rank(m))
print("Giả nghịch đảo ma trận 2x3:\n", pseudo_inverse([[1, 2, 3], [4, 5, 6]]).round(3))

In [ ]:
# ============================================================
# 1.4 NHÓM HÀM: Phân rã LU
# ------------------------------------------------------------
# 📖 TRANG 24: "LU Decomposition: A = P · L · U"
#   - L: tam giác dưới, U: tam giác trên
#   - Điều kiện: mọi ma trận con cấp 1..n có định thức ≠ 0 → nếu không thì HOÁN VỊ dòng (ma trận P)
#   - scipy.linalg.lu(A) trả về P, L, U ;  scipy.linalg.lu_factor(A) trả (ma trận hỗn hợp, chỉ số) tiết kiệm bộ nhớ
# 🎯 DÙNG KHI: giải hệ phương trình nhiều lần, tính định thức, nghịch đảo nhanh.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – LU Decomposition]:
#   - Mục đích: biến ma trận PHỨC TẠP thành các ma trận ĐƠN GIẢN: A = L × U (L: tam giác dưới ; U: tam giác trên).
#   - Ma trận con cấp 1 = (1 dòng, 1 cột đầu); cấp n = (n dòng, n cột đầu).
#     Muốn phân rã ma trận vuông ⇒ MỌI ma trận con phải có định thức ≠ 0.
#   - Nếu vi phạm → HOÁN VỊ dòng/cột (dùng ma trận hoán vị P).
#   - Ma trận hoán vị cũng là ma trận TRỰC GIAO ⇔ chuyển vị = nghịch đảo (Pᵀ = P⁻¹).
#     (Trực giao: là tính chất của MA TRẬN. Vector trực giao: là tính chất của 2 VECTOR ⇔ tích vô hướng = 0.)
#   - scipy.linalg.lu(A) → trả P, L, U (kiểm tra bằng A = P @ L @ U) ;
#     scipy.linalg.lu_factor(A) → trả 1 ma trận HỖN HỢP (L và U gộp) + 1 dãy CHỈ SỐ hoán vị → TIẾT KIỆM bộ nhớ.
#   - Lý thuyết: chỉ ma trận VUÔNG phân rã LU; nhưng trong Python ma trận chữ nhật vẫn phân rã được.
# 🔎 CHỌN KHI NÀO:
#   - Ma trận VUÔNG và cần GIẢI nhiều hệ A·x=b với cùng A (b khác nhau), tính định thức/nghịch đảo nhanh → LU (phân rã 1 lần, dùng lại).
#   - DẤU HIỆU phải hoán vị (dùng scipy_lu có P): check_minors() trả False / phần tử trụ U[i,i] = 0 (sẽ chia cho 0) / det ma trận con = 0.
#   - Ma trận lớn, cần tiết kiệm bộ nhớ → lu_factor (ma trận hỗn hợp + chỉ số) thay vì lu (3 ma trận).
#   - Ma trận CHỮ NHẬT hoặc cần nghịch đảo dễ → ưu tiên QR (ô kế tiếp), vì Q trực giao nên Q⁻¹ = Qᵀ.
# ============================================================
def check_minors(A):
    # Kiểm tra điều kiện: định thức các ma trận con cấp 1..n phải ≠ 0
    A = np.array(A, dtype=float)
    ok = True
    for k in range(1, len(A) + 1):
        d = np.linalg.det(A[:k, :k])               # ma trận con = k dòng đầu, k cột đầu
        print(f"  Ma trận con cấp {k}: det = {d:.3f}")
        if abs(d) < 1e-9:
            ok = False
    return ok

def manual_lu(A):
    # LU thủ công (Doolittle, không hoán vị): L có đường chéo = 1
    A = np.array(A, dtype=float)
    n = len(A)
    L = np.eye(n)
    U = A.copy()
    for i in range(n):
        for j in range(i + 1, n):
            L[j, i] = U[j, i] / U[i, i]            # hệ số khử lưu vào L
            U[j] = U[j] - L[j, i] * U[i]           # khử → U tam giác trên
    return L, U

def scipy_lu(A):
    P, L, U = sla.lu(np.array(A, dtype=float))     # A = P @ L @ U
    return P, L, U

def scipy_lu_factor(A):
    lu, piv = sla.lu_factor(np.array(A, dtype=float))   # lu: ma trận hỗn hợp (L và U gộp), piv: chỉ số hoán vị
    return lu, piv

A = [[2, 1, 1], [4, 3, 3], [8, 7, 9]]
print("Điều kiện ma trận con:", check_minors(A))
L, U = manual_lu(A)
print("L =\n", L, "\nU =\n", U, "\nKiểm tra L@U = A:", np.allclose(L @ U, A))
P, L2, U2 = scipy_lu(A)
print("scipy: P@L@U = A ?", np.allclose(P @ L2 @ U2, A))
print("P là ma trận trực giao (P.T @ P = I)?", np.allclose(P.T @ P, np.eye(3)))

In [ ]:
# ============================================================
# 1.5 NHÓM HÀM: Phân rã QR – Gram-Schmidt
# ------------------------------------------------------------
# 📖 TRANG 25: "QR Decomposition: A = Q · R ; Q trực giao, R tam giác trên ; thuật toán Gram-Schmidt"
#   - Lý thuyết: QR dùng cho ma trận chữ nhật BẤT KỲ
#   - Q trực giao → Q^-1 = Q^T (dễ tính nghịch đảo hơn L)
# 🎯 DÙNG KHI: giải bình phương tối thiểu (hồi quy), tìm trị riêng.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – QR Decomposition]:
#   - Lý thuyết: QR dành cho ma trận CHỮ NHẬT BẤT KỲ (LU chỉ dành cho ma trận vuông).
#   - A = Q × R ; Q là ma trận TRỰC GIAO (hơn L ở chỗ tính được Q⁻¹ = Qᵀ rất dễ) ; R là ma trận TAM GIÁC TRÊN.
#   - Thuật toán Gram–Schmidt: lần lượt trừ phần chiếu lên các vector đã trực chuẩn rồi chuẩn hóa độ dài về 1.
# 🔎 CHỌN KHI NÀO:
#   - Ma trận CHỮ NHẬT (nhiều dòng quan sát × ít cột biến, như bảng dữ liệu) → QR (không dùng LU).
#   - Cần nghịch đảo/giải hệ ổn định, hoặc bài toán bình phương tối thiểu (hồi quy) [+bổ sung] → QR.
#   - KIỂM TRA KẾT QUẢ ĐÚNG: Q.T @ Q = I (cột trực chuẩn) và Q @ R = A ; R phải là tam giác trên.
# ============================================================
def gram_schmidt(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()                        # lấy cột thứ j
        for i in range(j):                        # trừ phần chiếu lên các vector đã trực chuẩn
            R[i, j] = Q[:, i] @ A[:, j]           # R[i,j] = <qi, aj>
            v = v - R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)               # độ dài phần còn lại
        Q[:, j] = v / R[j, j]                     # chuẩn hóa thành vector đơn vị
    return Q, R

def numpy_qr(A):
    return np.linalg.qr(np.array(A, dtype=float))

A_qr = np.array([[1, 1], [1, 0], [0, 1]], dtype=float)      # ma trận chữ nhật 3x2
Q, R = gram_schmidt(A_qr)
print("Q =\n", Q.round(3), "\nR =\n", R.round(3))
print("Q@R = A ?", np.allclose(Q @ R, A_qr), "| Q.T@Q = I (trực giao) ?", np.allclose(Q.T @ Q, np.eye(2)))

## PHẦN 2 – TẬP TIN TEXT, REGEX, SERIES, DATAFRAME (Trang 3–6)

In [ ]:
# ============================================================
# 2.1 NHÓM HÀM: Đọc/ghi file text, file log để debug
# ------------------------------------------------------------
# 📖 TRANG 3-4: "Xử lý tập tin văn bản": BOF/EOF, con trỏ đọc/ghi, đọc/ghi theo ký tự, theo dòng
#   - Debug (trang 4): ghi kết quả từng vòng lặp ra file text (file log) rồi đọc lại phân tích lỗi
# 🎯 DÙNG KHI: lưu kết quả trung gian của thuật toán để kiểm tra logic.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Xử lý tập tin văn bản]:
#   - File .txt KHÔNG có file header. Có 2 vị trí đặc biệt: BOF (bắt đầu file) và EOF (kết thúc file).
#   - Mở file = HĐH chép từ đĩa lên RAM; đang mở thì thao tác trên RAM; lưu & đóng thì ghi lại đĩa và xóa trên RAM.
#   - Cơ chế truy cập: CON TRỎ (pointer) đọc/ghi ở vị trí hiện hành; sau khi đọc/ghi xong ký tự thì nhảy tới vị trí kế tiếp.
#   - Đọc/ghi: theo KÝ TỰ hoặc theo DÒNG.
#   - Sử dụng text file khi: ghi kết quả ra để DEBUG lỗi LOGIC. Quy trình debug [Word]:
#       (1) xác định phần logic/vòng lặp/từng chặng ; (2) chặn từng chỗ trong mỗi lần lặp (càng chạy nhiều càng dễ lệch kết quả);
#       (3) xuất kết quả ra file text (file log) ; (4) đọc kỹ file log để phân tích lỗi sai.
#   - Tìm thông tin trong file text theo biểu thức chính quy (Regex – ô 2.3).
# 🔎 CHỌN KHI NÀO:
#   - Kết quả chạy SAI mà code KHÔNG báo lỗi (lỗi logic), nhất là sai lệch tăng dần qua nhiều vòng lặp → ghi log từng vòng rồi đọc lại.
#   - Mode "w" (ghi đè, con trỏ ở BOF) khi tạo file mới ; mode "a" (nối cuối file) khi ghi log nhiều vòng ; mode "r" để đọc.
#   - File mỗi dòng là 1 bản ghi → đọc THEO DÒNG ; cần cắt đúng n ký tự/định vị → đọc THEO KÝ TỰ.
# ============================================================
def write_lines(filename, lines):
    # mode "w": ghi đè (con trỏ bắt đầu ở BOF). "with" tự đóng file (RAM → đĩa cứng)
    with open(filename, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")           # ghi theo dòng

def append_line(filename, line):
    with open(filename, "a", encoding="utf-8") as f:   # mode "a": ghi nối tiếp cuối file
        f.write(line + "\n")

def read_all(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()                    # đọc toàn bộ từ BOF đến EOF

def read_lines(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return [l.strip() for l in f.readlines()]   # đọc theo dòng

def read_chars(filename, n):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read(n)                   # đọc n ký tự; con trỏ nhảy tới vị trí kế tiếp

# --- Sử dụng: ghi file log từng vòng lặp (kỹ thuật debug) ---
LOG_FILE = "debug_log.txt"
write_lines(LOG_FILE, ["=== LOG DEBUG ==="])
total = 0
for i, s in enumerate(df["Sales"].dropna().head(5)):
    total += s
    append_line(LOG_FILE, f"Vòng {i+1}: Sales={s} | Tổng tích lũy={round(total, 2)}")   # chặn từng lần lặp
print(read_all(LOG_FILE))
print("5 ký tự đầu:", read_chars(LOG_FILE, 5))

In [ ]:
# ============================================================
# 2.2 NHÓM HÀM: Unit Test – kiểm thử hộp đen
# ------------------------------------------------------------
# 📖 TRANG 4: Kiểm thử Hộp đen (đưa input → so output với giải tay), Hộp trắng, Hộp xám;
#            Mức độ: Unit Test (từng hàm/module), Customer, Alpha, Beta
# 🎯 DÙNG KHI: tự kiểm tra hàm của mình trước khi nộp bài.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Phương pháp kiểm thử]:
#   - 3 phương pháp:
#       (1) HỘP ĐEN (Black Box – đóng gói): đưa input vào, xem output có đúng với GIẢI TAY không; không quan tâm code bên trong.
#       (2) HỘP TRẮNG (White Box): đọc sâu vào code, đọc được hết để kiểm thử.
#       (3) HỘP XÁM (Grey Box): 1 đoạn xem được, 1 đoạn không kiểm soát được → kết hợp cả hai.
#   - 4 mức độ: (1) UNIT TEST: từng đơn vị lập trình (module/hàm) ; (2) CUSTOMER TEST: mức khách hàng ;
#               (3) ALPHA TEST: nội bộ ; (4) BETA TEST: cho những người sẵn sàng dùng thử để đội ngũ chỉnh sửa.
# 🔎 CHỌN KHI NÀO:
#   - Chỉ có hàm + input/output, có thể tự TÍNH TAY với số nhỏ (tích vô hướng [1,2]·[3,4] = 11) → HỘP ĐEN (cách dùng trong ô này).
#   - Có toàn bộ code, nghi sai ở bước trung gian → HỘP TRẮNG (in giá trị từng bước / ghi file log).
#   - Một phần là thư viện (numpy/sklearn) không đọc được ruột + một phần code của mình → HỘP XÁM.
#   - Kiểm tra từng hàm trước khi nộp bài → UNIT TEST. Kết quả FAIL (actual ≠ expected) → quay lại debug bằng file log.
#   - [+bổ sung] So sánh số thực dùng dung sai (tol), không dùng == vì sai số làm tròn dấu phẩy động.
# ============================================================
def check(name, actual, expected, tol=1e-6):
    ok = abs(actual - expected) < tol            # so sánh output thực với kết quả giải tay
    print(("✅ PASS" if ok else "❌ FAIL"), name, "| thực tế =", actual, "| mong đợi =", expected)

check("Tích vô hướng [1,2]·[3,4]", dot([1, 2], [3, 4]), 11)       # 1*3+2*4 = 11
check("Chuẩn L2 của [3,4]", norm_l2([3, 4]), 5)                   # căn(9+16) = 5

In [ ]:
# ============================================================
# 2.3 NHÓM HÀM: Biểu thức chính quy
# ------------------------------------------------------------
# 📖 TRANG 4-5: "Regular Expression"
#   Set [] : [abc] đúng ký tự ; [^abc] KHÔNG phải ; [a-z] từ đầu đến cuối
#   Metacharacters: .  ^  $  *  +  ?  {}
#   Special sequences (thêm r"..." phía trước): \A  \b  \d \D \w \W \s \S (chữ thường = CÓ tính chất, IN HOA = KHÔNG)
# 🎯 DÙNG KHI: tìm/trích thông tin theo dạng thức trong dữ liệu chuỗi (tên, mã, số...).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Regular Expression]: công cụ TÌM KIẾM với dữ liệu kiểu CHUỖI.
#   - SET [ ] (biết CHÍNH XÁC là ký tự gì):  [abc] đúng 1 trong các ký tự ; [^abc] KHÔNG phải các ký tự đó ; [a-z] từ ký tự đầu đến cuối.
#   - METACHARACTERS (KHÔNG biết đích xác là gì):
#       .  : ký tự gì cũng được, 1 dấu chấm = ĐÚNG 1 ký tự      ^ : bắt đầu bằng        $ : kết thúc bằng
#       *  : 0 hoặc nhiều ký tự (tham lam: ví dụ .o*e kéo dài tới chữ 'e' XA NHẤT)
#       +  : ÍT NHẤT 1 ký tự                                      ?  : 0 hoặc 1 ký tự     {n}: đúng số lượng n
#   - SPECIAL SEQUENCES: LUÔN thêm chữ r trước chuỗi: r"(...)". \A bắt đầu chuỗi ; \b ranh giới từ ;
#       CHỮ THƯỜNG = CÓ tính chất (\d chữ số, \w chữ/số, \s khoảng trắng) ; CHỮ IN HOA = KHÔNG CÓ tính chất (\D, \W, \S).
#     (⚠ Word diễn đạt \b là "kết thúc"; thực chất \b là RANH GIỚI TỪ – đầu hoặc cuối từ đều khớp.)
# 🔎 CHỌN KHI NÀO:
#   - Dữ liệu là CHUỖI (cột kiểu object/str: tên, mã, địa chỉ, SĐT, email) và cần tìm/kiểm tra theo DẠNG THỨC lặp lại.
#   - Biết CHÍNH XÁC ký tự → dùng Set [ ] ; KHÔNG biết đích xác → dùng metacharacter (. * + ? {}).
#   - Lọc theo ĐẦU/CUỐI chuỗi → ^ và $ ; bắt buộc có ≥1 ký tự → + ; có hay không đều được → * hoặc ? ; số lượng cố định → {n}.
#   - Dùng làm công cụ phát hiện DỮ LIỆU KHÔNG NHẤT QUÁN về định dạng: ví dụ Zip Code (ép về chuỗi) phải khớp ^\d{5}$ — dòng nào không khớp là bất thường.
#   - re.search: tìm ở BẤT KỲ vị trí nào ; re.findall: lấy MỌI đoạn khớp trong 1 chuỗi (ví dụ trích các số \d+).
# ============================================================
def regex_find(texts, pattern):
    # re.search: tìm pattern ở BẤT KỲ vị trí nào trong chuỗi
    return [t for t in (str(x) for x in texts) if re.search(pattern, t)]

def regex_find_all(pattern, text):
    return re.findall(pattern, text)               # trả về mọi đoạn khớp trong 1 chuỗi

names = df["Customer Name"].dropna().unique()
print("1) ^M      : bắt đầu bằng M        →", regex_find(names, r"^M")[:3])
print("2) son$    : kết thúc bằng 'son'   →", regex_find(names, r"son$")[:3])
print("3) [^A-Z]  : chứa ký tự KHÔNG phải in hoa →", regex_find(names, r"[^A-Z a-z]")[:3])
print("4) ^A.{5}  : bắt đầu A rồi 5 ký tự bất kỳ →", regex_find(names, r"^A.{5}")[:3])
print("5) o+      : có ít nhất 1 chữ o    →", regex_find(names, r"o+")[:3])
print("6) ^J\\w*   : bắt đầu J (\\w = chữ/số) →", regex_find(names, r"^J\w*")[:3])
print("7) \\bMac   : từ bắt đầu bằng Mac   →", regex_find(names, r"\bMac")[:3])
print("8) Số trong tên sản phẩm (\\d+):", regex_find_all(r"\d+", df["Product Name"].dropna().iloc[2]))

In [ ]:
# ============================================================
# 2.4 NHÓM HÀM: Series – mảng 1 chiều của pandas
# ------------------------------------------------------------
# 📖 TRANG 5: "Cấu trúc Series": chỉ số ngầm định (0,1,2..) và tường minh (labels);
#   4 cách khai thác: indexing, fancy indexing, slicing, masking ;  loc (tường minh) / iloc (ngầm định)
#   Tham số inplace: False → phải hứng kết quả ; True → sửa trực tiếp
# 🎯 DÙNG KHI: thao tác trên 1 cột dữ liệu.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Cấu trúc Series]:
#   - Series = mảng 1 CHIỀU (giống list / array numpy) nhưng có thêm CHỈ SỐ:
#       · chỉ số NGẦM ĐỊNH (0,1,2…) và chỉ số TƯỜNG MINH (labels do mình đặt).
#       · labels tạo từ list: có thể bằng nhau hoặc khác nhau đều được ; labels lấy từ dictionary (keys): cho phép trùng nhau (THẬN TRỌNG!).
#   - 4 cách khai thác: (1) INDEXING (theo chỉ số ngầm định hoặc label) ; (2) FANCY indexing (nhảy cóc, như indexing) ;
#                       (3) SLICING (cắt 1 khoảng) ; (4) MASKING (ra điều kiện True/False rồi lọc/gán).
#   - loc[]: chỉ số TƯỜNG MINH (label) ; iloc[]: chỉ số NGẦM ĐỊNH (vị trí).
#   - Tham số inplace: False → PHẢI "hứng" kết quả vào biến ; True → sửa trực tiếp (ví dụ drop).
# 🔎 CHỌN KHI NÀO:
#   - Lấy theo TÊN nhãn → loc (slice loc['a':'c'] LẤY CẢ đầu và cuối) ; lấy theo VỊ TRÍ → iloc (slice iloc[1:3] KHÔNG lấy cuối).
#   - Cần vài phần tử RỜI RẠC → fancy s[['a','d']] ; cần 1 khoảng liên tục → slicing ; cần các giá trị thỏa điều kiện → masking s[s>150].
#   - Muốn GIỮ NGUYÊN dữ liệu gốc (quy tắc "không sửa dữ liệu thô") → inplace=False và hứng vào biến mới.
#   - Series có label TRÙNG → loc trả về NHIỀU phần tử thay vì 1 (dấu hiệu kết quả lạ) → kiểm tra index.is_unique.
# ============================================================
def series_demo():
    s = pd.Series([100, 200, 300, 400], index=["a", "b", "c", "d"])  # index tường minh = labels
    print("Series:\n", s)
    print("1) Indexing  s['b'] =", s["b"], "| s.iloc[1] =", s.iloc[1])     # theo label / theo vị trí
    print("2) Fancy     s[['a','d']] →", s[["a", "d"]].tolist())           # nhảy cóc
    print("3) Slicing   s.iloc[1:3] →", s.iloc[1:3].tolist())              # cắt đoạn (iloc: không lấy cuối)
    print("   s.loc['a':'c'] →", s.loc["a":"c"].tolist())                  # loc: LẤY CẢ đầu và cuối
    print("4) Masking   s[s>150] →", s[s > 150].tolist())                  # điều kiện True/False
    s2 = s.drop("a")                  # inplace=False (mặc định): phải HỨNG vào biến mới
    s.drop("a", inplace=True)         # inplace=True: sửa trực tiếp
    print("Sau drop:", s.tolist())

series_demo()

In [ ]:
# ============================================================
# 2.5 NHÓM HÀM: DataFrame – bảng 2 chiều
# ------------------------------------------------------------
# 📖 TRANG 5-6: "DataFrame = ghép nhiều Series; index = tên dòng; columns = tên cột;
#               chiếu NHIỀU cột phải dùng 2 cặp [[ ]]"
# 🎯 DÙNG KHI: chuyển ma trận → bảng, chọn cột/dòng, ghi file.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – DataFrame]:
#   - DataFrame = bảng dữ liệu 2 CHIỀU = ghép nối nhiều Series; tên cột chính là tên của Series.
#   - Chuyển TRỰC TIẾP từ 1 ma trận: ma trận không có tên cột/tên dòng thì dòng được đánh số ngầm định 0,1,2…
#   - pd.DataFrame(data, index=<tên dòng>, columns=<tên cột>).
#   - PHÉP CHIẾU nhiều cột phải có HAI cặp dấu [[ ]] (bắt buộc).
# 🔎 CHỌN KHI NÀO:
#   - Lấy 1 cột → df["x"] (kết quả là Series) ; lấy NHIỀU cột → df[["x","y"]] (kết quả là DataFrame). Quên cặp [] ngoài → lỗi KeyError.
#   - Có ma trận numpy mà muốn đặt tên dòng/cột → pd.DataFrame(matrix, index=..., columns=...).
#   - Dữ liệu bảng nhiều kiểu (số + chữ + ngày) → DataFrame ; ma trận thuần số cần đại số (nhân, nghịch đảo) → giữ numpy.
#   - Lọc dòng theo điều kiện → masking df[df["x"]>1] ; chọn theo nhãn → loc ; theo vị trí → iloc.
# ============================================================
def dataframe_demo(data):
    matrix = np.array([[1, 2], [3, 4], [5, 6]])                      # ma trận không tên
    t = pd.DataFrame(matrix, index=["r1", "r2", "r3"], columns=["x", "y"])   # đặt tên dòng/cột
    print(t)
    print("Chiếu 1 cột   t['x'] → Series:", t["x"].tolist())
    print("Chiếu nhiều cột t[['x','y']] → DataFrame (2 cặp []):\n", t[["x", "y"]].shape)
    print("loc  (theo nhãn):", t.loc["r2", "y"], "| iloc (theo vị trí):", t.iloc[1, 1])
    print("Lọc dòng (masking) t[t['x']>1]:\n", t[t["x"] > 1])
    # Với dữ liệu thật:
    print(data[["Region", "Sales", "Profit"]].head(3))
    # Ghi file:  data.to_csv("out.csv", index=False)  |  data.to_excel("out.xlsx", index=False)

dataframe_demo(df)

## PHẦN 3 – QUY ƯỚC VIẾT HÀM THUẦN & HÀM TIỆN ÍCH (Trang 6–7, 11)

**Quy ước cho MỌI hàm trong notebook:**
1. Giá trị hàm cần → **tham số**; kết quả → **`return`** (không `global`, không đọc biến bên ngoài).
2. Hàm xử lý DataFrame luôn `d = d.copy()` rồi trả về bảng mới → **không sửa dữ liệu thô** (trang 11).
3. Cấu hình (tên cột, ngưỡng…) gom vào **dict** rồi truyền vào hàm, thay vì gắn vào đối tượng.


In [ ]:
# ============================================================
# 3.1 HÀM TIỆN ÍCH DÙNG CHUNG (thủ tục thuần – không class)
# ------------------------------------------------------------
# 📖 TRANG 11 (không sửa dữ liệu thô)
# 🎯 DÙNG KHI: mọi hàm xử lý bên dưới đều tuân theo quy ước:
#   - nhận DataFrame làm tham số, bên trong làm việc trên BẢN SAO (không đụng dữ liệu thô)
#   - TRẢ VỀ kết quả mới thay vì sửa tại chỗ
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Quy tắc tiền xử lý, mục 3]: "Tiền xử lý nhằm mục đích phân tích → KHÔNG CHỈNH SỬA DỮ LIỆU THÔ";
#   cân nhắc chất lượng và hiệu quả (so sánh kết quả TRƯỚC và SAU tiền xử lý); linh hoạt biện pháp và các công đoạn.
# 🔎 CHỌN KHI NÀO: luôn luôn – mọi hàm xử lý phải làm việc trên BẢN SAO (.copy()) và trả về kết quả mới.
#   DẤU HIỆU làm sai quy tắc: sau khi chạy mà raw_df/df gốc bị đổi shape hoặc giá trị → có chỗ sửa trực tiếp lên dữ liệu thô.
#   → snapshot_hash() bên dưới dùng để CHỨNG MINH dữ liệu thô không đổi (so sánh dấu vân tay trước/sau).
# ============================================================
def numeric_cols(d):
    # danh sách tên các cột kiểu số của DataFrame d
    return d.select_dtypes(include="number").columns.tolist()

def snapshot_hash(d):
    # "dấu vân tay" của cả bảng: nếu dữ liệu đổi dù 1 ô thì số này đổi
    return int(pd.util.hash_pandas_object(d, index=True).sum())

# --- Sử dụng: chứng minh hàm xử lý không làm hỏng dữ liệu thô ---
fp_before = snapshot_hash(raw_df)
print("Cột số của df:", numeric_cols(df)[:5], "...")
print("Dấu vân tay dữ liệu thô:", fp_before)


## PHẦN 4 – TIỀN XỬ LÝ & KHÁM PHÁ DỮ LIỆU – EDA (Trang 7–10)

In [ ]:
# ============================================================
# 4.1 NHÓM HÀM: Các cách lấy mẫu dữ liệu sơ cấp
# ------------------------------------------------------------
# 📖 TRANG 8: "Dữ liệu sơ cấp (primary data): (1) mẫu ngẫu nhiên; (2) phân tầng; (3) gom cụm;
#              (5) thuận tiện; (6) phán đoán / ý kiến chuyên gia; (7) bông tuyết"
# 🎯 DÙNG KHI: tập dữ liệu quá lớn / cần mẫu đại diện để phân tích.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Phân loại nguồn dữ liệu]:
#   - DỮ LIỆU SƠ CẤP (primary – tự thu thập): có các cách lấy mẫu:
#       (1) NGẪU NHIÊN ; (2) PHÂN TẦNG: tập lớn chia thành n tập nhỏ, rồi nhỏ hơn ; (3) GOM CỤM ;
#       (4) chia nhóm ngẫu nhiên rồi lấy ngẫu nhiên thêm 1 người (theo ghi chú Word) ;
#       (5) THUẬN TIỆN: lấy người dễ tiếp cận nhất, cần đảm bảo khách quan, không thiên lệch ;
#       (6) PHÁN ĐOÁN / ý kiến chuyên gia: thu hẹp lại rồi khảo sát ;
#       (7) BÔNG TUYẾT: khó lấy mẫu → vài người hạt nhân, rồi lan truyền ra.
#   - DỮ LIỆU THỨ CẤP (secondary – có sẵn do người khác thu thập). Superstore là dữ liệu có sẵn → thuộc loại thứ cấp.
# 🔎 CHỌN KHI NÀO:
#   - Tập đồng nhất, không có nhóm rõ rệt                              → NGẪU NHIÊN đơn giản.
#   - Có các nhóm con (Region, Segment) quy mô CHÊNH LỆCH và muốn mẫu đại diện MỌI nhóm theo đúng tỷ lệ
#       → PHÂN TẦNG (lấy MỘT PHẦN từ MỖI nhóm). DẤU HIỆU: value_counts() lệch nhiều → lấy ngẫu nhiên dễ bỏ sót nhóm nhỏ.
#   - Các nhóm tự nhiên "bên trong giống tổng thể", khảo sát từng nhóm tốn kém (bang, cửa hàng)
#       → GOM CỤM (chọn MỘT SỐ nhóm rồi lấy TOÀN BỘ phần tử của nhóm).   ⇒ Khác phân tầng: tầng = lấy MỌI nhóm ; cụm = lấy VÀI nhóm. [+bổ sung]
#   - Cần nhanh, chấp nhận kém đại diện                                  → THUẬN TIỆN (dễ thiên lệch → phải nêu hạn chế).
#   - Có chuyên gia xác định đúng đối tượng cần xét (Technology & Sales>5000) → PHÁN ĐOÁN.
#   - Tổng thể "ẩn", khó tiếp cận, không có danh sách                     → BÔNG TUYẾT.
# ============================================================
def random_sample(d, n, seed=1):
    return d.sample(n=n, random_state=seed)                   # ngẫu nhiên đơn giản

def stratified_sample(d, col, frac, seed=1):
    # phân tầng: chia theo nhóm (tầng) rồi lấy cùng tỷ lệ ở MỖI tầng
    return d.groupby(col).sample(frac=frac, random_state=seed)

def cluster_sample(d, col, n_clusters, seed=1):
    # gom cụm: chọn ngẫu nhiên vài cụm rồi lấy TOÀN BỘ phần tử của cụm đó
    chosen = pd.Series(d[col].dropna().unique()).sample(n=n_clusters, random_state=seed)
    return d[d[col].isin(chosen)]

def convenience_sample(d, n):
    return d.head(n)                                          # thuận tiện: lấy n dòng dễ lấy nhất

def judgment_sample(d, condition):
    return d.query(condition)                                 # phán đoán: lọc theo hiểu biết chuyên gia

print("Ngẫu nhiên:", len(random_sample(df, 100)))
strat = stratified_sample(df, "Region", 0.1)
print("Phân tầng 10% theo Region:\n", strat["Region"].value_counts())
print("Gom cụm 2 bang:", cluster_sample(df, "State", 2)["State"].unique())
print("Phán đoán (Technology & Sales>5000):", len(judgment_sample(df, "`Product Category`=='Technology' and Sales>5000")))

In [ ]:
# ============================================================
# 4.2 NHÓM HÀM: Lướt dữ liệu + Nhận diện biến + Đánh giá chất lượng
# ------------------------------------------------------------
# 📖 TRANG 8-10: Phân loại dữ liệu (định danh/định tính/định lượng, thời gian),
#   EDA, Phân tích ĐƠN BIẾN:
#     Bước 1: nhận diện biến – cấu trúc (kiểu, định dạng, dữ liệu thô hay tính toán)
#     Bước 2: nhận diện biến – chức năng (Input/feature vs Output/target)
#     Bước 3: đánh giá chất lượng (thiếu, bất thường, không nhất quán)
#     Bước 4: thu gọn dữ liệu
#   TRANG 15-16 (phương pháp làm bài kiểm tra): lướt dữ liệu → xác định input/output → đánh giá chất lượng.
# 🎯 DÙNG KHI: BƯỚC ĐẦU TIÊN của mọi bài kiểm tra.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Tiền xử lý & EDA]:
#   - PHÂN LOẠI DỮ LIỆU (tùy mục tiêu, quan điểm): theo loại (ảnh, audio, video…) ; theo cấu trúc (bán cấu trúc, không cấu trúc) ;
#     theo hướng (định tính / định lượng).
#   - ĐỊNH DANH (nominal, categorical): phân loại đối tượng, KHÔNG có thứ tự.
#   - ĐỊNH TÍNH (ordinal, qualitative): có quan hệ THỨ TỰ.
#   - ĐỊNH LƯỢNG (numerical): RỜI RẠC = đếm được ; LIÊN TỤC = đo được. Chia tiếp:
#       · INTERVAL: hiệu số CÓ ý nghĩa, THƯƠNG số KHÔNG có ý nghĩa; không có true zero (số 0 chỉ là điểm mốc); đơn vị bằng nhau.
#       · RATIO (ratio-scaled): có zero thật; hiệu số VÀ thương số đều có ý nghĩa.
#   - KIỂU THỜI GIAN: date-time (ngày giờ cố định) ; timestamp (số giây tính từ 01/01/1970 UTC, đổi linh hoạt theo múi giờ;
#     thầy nhắc: chuyển về string để tính/hiển thị dễ hơn).
#   - EDA: mục tiêu = ĐÁNH GIÁ và CHỌN GIẢI PHÁP cho dữ liệu; phục vụ phân tích MÔ TẢ / DỰ BÁO / ĐỀ XUẤT.
#   - TIÊU CHÍ DỮ LIỆU TỐT: nội tại (chính xác, đầy đủ, nhất quán) ; ngữ cảnh (cập thời, hữu dụng) ;
#     truy cập (đúng đắn, toàn vẹn, trực quan, an toàn).
#   - PHÂN TÍCH ĐƠN BIẾN – 4 bước: (1) nhận diện biến – CẤU TRÚC (kiểu, định dạng, thô hay tính toán) ;
#     (2) nhận diện biến – CHỨC NĂNG: Input = feature/predictor/independent ; Output = target/dependent ;
#     (3) đánh giá chất lượng: thiếu / bất thường / không nhất quán ; (4) thu gọn/giảm chiều.
#   - Thứ tự các bước KHÔNG cố định, có thể quay nhiều vòng; trình bày thì theo thứ tự.
# 🔎 CHỌN KHI NÀO – CÁCH NHẬN DIỆN LOẠI BIẾN TỪ DẤU HIỆU DỮ LIỆU:
#   - Giá trị là CHỮ/NHÃN, KHÔNG xếp hơn–kém được (Region, Ship Mode, City)                 → ĐỊNH DANH (nominal).
#   - Giá trị là chữ/nhãn nhưng XẾP được thứ tự (Low < Medium < High < Critical)            → ĐỊNH TÍNH (ordinal) – khi mã hóa phải GIỮ thứ tự.
#   - Là SỐ ĐẾM, toàn số nguyên (Order Quantity)                                            → ĐỊNH LƯỢNG RỜI RẠC.
#   - Là SỐ ĐO, có phần thập phân/nhận mọi giá trị trong khoảng (Sales, Shipping Cost)       → ĐỊNH LƯỢNG LIÊN TỤC.
#   - Có ZERO THẬT (Sales = 0 là "không bán được gì") và nói được "gấp đôi"                  → RATIO.
#   - Hiệu số có nghĩa nhưng "gấp đôi" vô nghĩa, số 0 là mốc tùy ý (nhiệt độ °C, năm)         → INTERVAL.
#   - Là SỐ nhưng chỉ làm MÃ (Row ID, Order ID, Zip Code): cộng/trừ/trung bình VÔ NGHĨA      → thực chất ĐỊNH DANH, đừng tính mean.
#   - Chuỗi ngày tháng                                                                      → date-time ; cần tính khoảng cách giây → timestamp ; cần hiển thị/ghép → string.
#   - CHỌN INPUT/OUTPUT: OUTPUT = biến mà đề bài muốn dự đoán/giải thích (ở đây Profit) ; INPUT = các biến còn lại có thể ảnh hưởng.
#       Cột TÍNH TOÁN từ cột khác (Sales, Profit) khi đặt chung làm input với các cột tạo ra nó → nghi DƯ THỪA (xem Phần 8).
#   - ĐÁNH GIÁ CHẤT LƯỢNG – nhìn vào đâu:
#       · THIẾU: missing_table() có Số thiếu > 0. (đã chạy) Product Base Margin thiếu 23% (> 20%) → ứng viên bỏ cột (Phần 5).
#       · BẤT THƯỜNG: outlier_table() (IQR). (đã chạy) Sales 1027, Profit 1406 outlier → phân phối lệch mạnh, cần xử lý (Phần 6).
#       · KHÔNG NHẤT QUÁN: vi phạm logic nghiệp vụ (giao trước khi đặt, Discount ngoài [0,1], Sales ≤ 0), dòng trùng lặp,
#         cùng 1 nhãn viết khác nhau (hoa/thường, khoảng trắng). (đã chạy) toàn 0 → dữ liệu này nhất quán.
# ============================================================
# --- Bước 1: nhận diện cấu trúc biến (do người làm khai báo dựa trên hiểu biết dữ liệu) ---
# (VAR_TYPE / COMPUTED / TARGET chỉ là DỮ LIỆU CẤU HÌNH; hàm step1_structure nhận chúng qua THAM SỐ)
VAR_TYPE = {
    "Row ID": "định danh (mã dòng)", "Order ID": "định danh (mã đơn)",
    "Order Date": "thời gian", "Ship Date": "thời gian",
    "Order Priority": "định tính (ordinal – có thứ tự)",
    "Order Quantity": "định lượng rời rạc", "Sales": "định lượng liên tục",
    "Discount": "định lượng liên tục", "Ship Mode": "định danh (nominal)",
    "Profit": "định lượng liên tục", "Unit Price": "định lượng liên tục",
    "Shipping Cost": "định lượng liên tục", "Customer Name": "định danh (nominal)",
    "City": "định danh (nominal)", "Zip Code": "định danh (mã vùng, KHÔNG phải số đo)",
    "State": "định danh (nominal)", "Region": "định danh (nominal)",
    "Customer Segment": "định danh (nominal)", "Product Category": "định danh (nominal)",
    "Product Sub-Category": "định danh (nominal)", "Product Name": "định danh (nominal)",
    "Product Container": "định danh (nominal)", "Product Base Margin": "định lượng liên tục (tỷ lệ)",
}
COMPUTED = ["Sales", "Profit"]      # Thô hay tính toán? (Sales, Profit được tính từ cột khác)
TARGET = "Profit"                   # Bước 2: chức năng. Ví dụ bài toán: "dự đoán/giải thích LỢI NHUẬN (Profit)"

def step1_structure(d, var_type, computed_cols, target):
    rows = []
    for c in d.columns:
        rows.append({
            "Cột": c,
            "Kiểu pandas": str(d[c].dtype),                                  # xác định kiểu
            "Loại dữ liệu": var_type.get(c, "?"),                            # định danh/định tính/định lượng
            "Thô/Tính toán": "tính toán" if c in computed_cols else "thô",
            "Vai trò": "OUTPUT (target)" if c == target else "INPUT (feature)",  # bước 2
            "Ví dụ": d[c].dropna().iloc[0],
        })
    return pd.DataFrame(rows)

# --- Bước 3a: dữ liệu bị thiếu ---
def missing_table(d):
    n = d.isna().sum()                                   # đếm ô thiếu mỗi cột
    pct = (n / len(d) * 100).round(2)                    # % thiếu
    t = pd.DataFrame({"Số thiếu": n, "% thiếu": pct})
    return t[t["Số thiếu"] > 0].sort_values("% thiếu", ascending=False)

# --- Bước 3b: dữ liệu bất thường (nhận diện sơ bộ bằng IQR/boxplot) ---
def outlier_table(d):
    out = {}
    for c in numeric_cols(d):
        x = d[c].dropna()
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        out[c] = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
    return pd.Series(out, name="Số outlier (IQR)")

# --- Bước 3c: dữ liệu KHÔNG nhất quán ---
def inconsistency_report(d, order_date="Order Date", ship_date="Ship Date", discount="Discount", sales="Sales",
                         qty="Order Quantity", cat_cols=("Region", "Ship Mode", "Customer Segment", "Product Category")):
    rep = {}
    rep["Ship Date < Order Date (giao trước khi đặt)"] = int((d[ship_date] < d[order_date]).sum())
    rep["Discount ngoài [0,1]"] = int(((d[discount] < 0) | (d[discount] > 1)).sum())
    rep["Sales <= 0"] = int((d[sales] <= 0).sum())
    rep["Order Quantity <= 0"] = int((d[qty] <= 0).sum())
    rep["Dòng trùng lặp hoàn toàn"] = int(d.duplicated().sum())
    for c in cat_cols:   # chữ hoa/thường, khoảng trắng
        s = d[c].dropna().astype(str)
        rep[f"{c}: giá trị khác nhau sau khi chuẩn hóa"] = int(s.nunique() - s.str.strip().str.lower().nunique())
    return pd.Series(rep)

# --- Xử lý dòng KHÔNG NHẤT QUÁN: loại dòng vi phạm logic nghiệp vụ + dòng trùng lặp ---
def drop_inconsistent_rows(d, order_date="Order Date", ship_date="Ship Date", discount="Discount", sales="Sales", qty="Order Quantity"):
    bad = ((d[ship_date] < d[order_date]) | (d[discount] < 0) | (d[discount] > 1) | (d[sales] <= 0) | (d[qty] <= 0))
    kept = d[~bad]
    clean = kept.drop_duplicates()                         # bỏ dòng trùng lặp hoàn toàn
    return clean, int(bad.sum()), len(kept) - len(clean)   # (bảng sạch, số dòng vi phạm, số dòng trùng)

# --- Dữ liệu thời gian: date-time ↔ timestamp ↔ string ---
def time_demo(d, col="Order Date"):
    t = d[col].iloc[0]
    ts = t.timestamp()                                   # số giây tính từ 01/01/1970 UTC
    return {"datetime": t, "timestamp": ts, "string": t.strftime("%d/%m/%Y"),
            "đổi ngược": pd.to_datetime(ts, unit="s")}

print("=== BƯỚC 1+2: Cấu trúc & chức năng biến ===")
display(step1_structure(df, VAR_TYPE, COMPUTED, TARGET))
print("=== BƯỚC 3a: Thiếu ===");           display(missing_table(df))
print("=== BƯỚC 3b: Bất thường ===");      display(outlier_table(df))
print("=== BƯỚC 3c: Không nhất quán ===");  display(inconsistency_report(df))
print("=== Thời gian ===", time_demo(df))

In [ ]:
# ============================================================
# 4.3 NHÓM HÀM: Phân tích ĐƠN BIẾN bằng thống kê mô tả + biểu đồ
# ------------------------------------------------------------
# 📖 TRANG 9: "Hình thức thể hiện chất lượng dữ liệu: con số (bảng đếm) hoặc biểu đồ (cột, đường, phân phối chuẩn)"
#    TRANG 23: Histogram (không có khoảng trắng giữa cột) ≠ Bar chart (có khoảng trắng)
# 🎯 DÙNG KHI: xem 1 biến phân bố thế nào, có lệch/outlier không.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Phân tích đơn biến & Hình dáng phân phối]:
#   - Chất lượng dữ liệu thể hiện bằng CON SỐ (ví dụ bảng đếm missing value) hoặc BIỂU ĐỒ (cột, đường, phân phối chuẩn…).
#   - BAR CHART: cho phép có KHOẢNG TRẮNG giữa các cột ; HISTOGRAM: KHÔNG có khoảng trắng (dữ liệu liên tục chia khoảng).
#   - 2 giai đoạn đều dùng Thống kê mô tả: KHÁM PHÁ → tập trung chất lượng (kích thước, quy mô, các biến, phân bố có ổn không) ;
#     MÔ TẢ → tập trung NGỮ NGHĨA.
# 🔎 CHỌN KHI NÀO:
#   - Biến ĐỊNH LƯỢNG (Sales, Profit)       → describe() + HISTOGRAM (xem hình dạng) + BOXPLOT (xem outlier).
#   - Biến ĐỊNH DANH/ĐỊNH TÍNH (Region)     → value_counts() + BAR CHART (có khoảng trắng). value_counts(dropna=False) để thấy cả ô thiếu.
#   - ĐỌC KẾT QUẢ describe():
#       · count < tổng số dòng                         → có DỮ LIỆU THIẾU.
#       · mean >> median (hoặc mean << median)         → LỆCH PHẢI (hoặc TRÁI). (đã chạy) Sales: mean 1778 ≫ median 448 → lệch phải.
#       · max (hoặc min) cách Q3 (Q1) rất xa           → có OUTLIER. (đã chạy) Sales max 89061 ≫ Q3 1711.
#       · min ≤ 0 ở biến phải dương                    → KHÔNG NHẤT QUÁN.
#       · std rất nhỏ so với mean (≈ hằng số)          → biến BIẾN THIÊN QUÁ THẤP → ứng viên bỏ (xem Phần 11).
# ============================================================
def describe_numeric(d, col):
    return d[col].describe()                       # count, mean, std, min, quartile, max

def count_categorical(d, col):
    return d[col].value_counts(dropna=False)       # bảng đếm (dropna=False để thấy cả NaN)

def plot_histogram(d, col, bins=30):
    plt.hist(d[col].dropna(), bins=bins, edgecolor="black")   # histogram: dữ liệu liên tục
    plt.title(f"Histogram - {col}"); plt.xlabel(col); plt.ylabel("Tần số"); plt.show()

def plot_bar(d, col):
    d[col].value_counts().plot(kind="bar")         # bar chart: dữ liệu phân loại
    plt.title(f"Bar chart - {col}"); plt.show()

def plot_boxplot(d, col):
    plt.boxplot(d[col].dropna(), vert=False)       # boxplot: nhìn outlier
    plt.title(f"Boxplot - {col}"); plt.show()

print(describe_numeric(df, "Sales"))
print(count_categorical(df, "Ship Mode"))
plot_histogram(df, "Sales")
plot_bar(df, "Region")
plot_boxplot(df, "Profit")

In [ ]:
# ============================================================
# 4.4 NHÓM HÀM: Phân tích ĐA BIẾN / 2 biến
# ------------------------------------------------------------
# 📖 TRANG 10: 4 trường hợp:
#   (1) 2 biến numerical     → hiệp phương sai, hệ số tương quan
#   (2) 2 biến categorical   → bảng chéo (two-way / cross table)
#   (3) numerical & categorical → z-test, t-test, ANOVA (xem Phần 7)
#   (4) Parallel boxplots (boxplot song song theo nhóm)
# 🎯 DÙNG KHI: tìm mối liên hệ giữa các biến (chọn input cho output, bỏ cột dư thừa).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Phân tích đa biến (2 biến)]:
#   1. 2 biến NUMERICAL: tương quan → HIỆP PHƯƠNG SAI (covariance), HỆ SỐ TƯƠNG QUAN (correlation) – phải là 2 con số.
#      Nếu không phải số → MÃ HÓA thành số rồi kiểm định giả thuyết.
#   2. 2 biến CATEGORICAL: TWO-WAY TABLE / CROSS-TABLE (ví dụ tuổi × giới tính).
#   3. NUMERICAL & CATEGORICAL: kiểm định giả thuyết z-test, t-test ; ANOVA (khi biến phân loại có NHIỀU loại giá trị).
#   4. PARALLEL BOXPLOTS (boxplot song song theo nhóm).
#   - Hiệp phương sai: xem mức CÙNG BIẾN THIÊN (xu thế); đổi ĐƠN VỊ thì giá trị đổi → KHÔNG kết luận được cặp nào mạnh hơn.
#   - Pearson: KHỬ đơn vị → biết CƯỜNG ĐỘ ; Spearman: dùng THỨ HẠNG (đồng hạng thì rải đều/lấy hạng trung bình).
# 🔎 CHỌN KHI NÀO – 2 biến thuộc loại nào thì dùng gì:
#   ┌──────────────────────────────────┬───────────────────────────────────────────────────────────────────────────┐
#   │ (numerical, numerical)           │ cov (xem chiều) + PEARSON (cường độ tuyến tính) / SPEARMAN (thứ hạng)       │
#   │ (categorical, categorical)       │ crosstab (two-way table) [+ Chi-square ở Phần 7 để kiểm định độc lập]       │
#   │ (numerical, categorical 2 nhóm)  │ t-test (nhị phân) ; z-test khi mẫu lớn / biết σ                             │
#   │ (numerical, categorical ≥3 nhóm) │ ANOVA ; trực quan bằng PARALLEL BOXPLOT                                    │
#   └──────────────────────────────────┴───────────────────────────────────────────────────────────────────────────┘
#   - ĐỌC KẾT QUẢ tương quan: |r| gần 1 → quan hệ tuyến tính mạnh ; gần 0 → không có quan hệ TUYẾN TÍNH (vẫn có thể phi tuyến) ;
#       dấu + cùng chiều, dấu − ngược chiều ; |r| ≥ ~0.8 giữa 2 INPUT → nghi DƯ THỪA (bỏ bớt 1) [+bổ sung ngưỡng].
#   - PEARSON ≠ SPEARMAN nhiều → có OUTLIER hoặc quan hệ không tuyến tính (nhưng đồng biến) → tin SPEARMAN hơn.
#       (đã chạy) Sales–Profit: Pearson 0.41, Spearman 0.33 → tương quan trung bình, dương.
#   - Covariance: chỉ dùng để xem DẤU/chiều ; muốn so mạnh–yếu phải dùng Pearson (đã khử đơn vị).
#   - Parallel boxplot: các hộp lệch nhau rõ / median khác xa → nhóm ảnh hưởng đến biến số (xác nhận bằng ANOVA/t-test).
# ============================================================
def num_num(d, c1, c2):
    x = d[[c1, c2]].dropna()                                # bỏ dòng thiếu của 2 cột
    return {"covariance": x[c1].cov(x[c2]), "pearson": x[c1].corr(x[c2]),
            "spearman": x[c1].corr(x[c2], method="spearman")}

def cat_cat(d, c1, c2):
    return pd.crosstab(d[c1], d[c2])                        # bảng chéo (two-way table)

def num_cat_summary(d, num, cat):
    return d.groupby(cat)[num].agg(["count", "mean", "median", "std"])

def parallel_boxplot(d, num, cat):
    groups = [g[num].dropna().values for _, g in d.groupby(cat)]
    plt.boxplot(groups)                                            # mỗi nhóm 1 hộp
    plt.xticks(range(1, len(groups) + 1), sorted(d[cat].dropna().unique()))   # tên nhóm dưới mỗi hộp
    plt.title(f"Parallel boxplot: {num} theo {cat}"); plt.xticks(rotation=20); plt.show()

print("Sales vs Profit:", num_num(df, "Sales", "Profit"))
display(cat_cat(df, "Region", "Customer Segment"))
display(num_cat_summary(df, "Profit", "Product Category"))
parallel_boxplot(df, "Shipping Cost", "Ship Mode")

## PHẦN 5 – LÀM SẠCH: DỮ LIỆU BỊ THIẾU (MISSING VALUE) (Trang 11–13)

Sheet `Missing_Value_Guide` trong file Excel cho biết cột nào thiếu theo cơ chế nào:
| Cột | Cơ chế | Biện pháp gợi ý |
|---|---|---|
| 5 cột (109 dòng thiếu cùng lúc) | – | Loại bỏ quan sát (dòng) |
| Product Base Margin (23%) | MCAR, vượt ngưỡng 20% | Loại bỏ thuộc tính (cột) |
| Ship Mode | MCAR | Giữ nguyên / thay mode / 'Unknown' |
| Discount | MAR (theo Customer Segment) | Thay theo nhóm |
| Unit Price | MAR (theo Product Sub-Category) | Thay theo nhóm / kNN |
| Shipping Cost | MCAR, tương quan số | Hồi quy / kNN / MICE |
| Profit | MNAR (cực cao/cực thấp mới thiếu) | Dùng mô hình / tri thức ngoài |

**🔎 Cách nhận biết cơ chế thiếu (dùng hàm `missing_rate_by_group` / `compare_observed_distribution` ở ô 5.1):**
| Dấu hiệu quan sát được | Cơ chế | Hướng xử lý (theo Word) |
|---|---|---|
| Ô thiếu rải rác, tỷ lệ thiếu **đều** giữa mọi nhóm | **MCAR** | Điền trực tiếp mean/mode… hoặc giữ nguyên nếu không liên quan |
| Tỷ lệ thiếu **khác nhau rõ** giữa các nhóm của một cột khác | **MAR** | Thay theo **nhóm/lân cận** cùng tính chất |
| Các giá trị còn lại **cụt đuôi** (thiếu cực cao/cực thấp), thiếu phụ thuộc chính giá trị | **MNAR** | Dùng mô hình tìm đối tượng tương đồng / bổ sung tri thức ngoài |


In [ ]:
# ============================================================
# 5.1 NHÓM HÀM: 7 biện pháp xử lý dữ liệu thiếu
# ------------------------------------------------------------
# 📖 TRANG 11-12:
#   Tác động: kết quả lệch (bias), vi phạm điều kiện đầu vào thuật toán.
#   BP1 Giữ nguyên | BP2 Loại bỏ dòng | BP3 Loại bỏ cột (ngưỡng ~20%, xem tương quan trước)
#   BP4 Thay giá trị đặc biệt (N/A, Unknown) hoặc thống kê mô tả (mean/median/mode)
#   BP5 Thay theo nhóm | BP6 Thay theo kNN (số → hồi quy) | BP7 MICE
#   TRANG 12: phân loại MCAR / MAR / MNAR
# 🎯 DÙNG KHI: bước "Đánh giá chất lượng dữ liệu (1) dữ liệu bị thiếu" – bước lâu nhất trong bài kiểm tra (trang 15).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Làm sạch: dữ liệu bị thiếu]:
#   - TÁC ĐỘNG: kết quả phân tích bị THIÊN LỆCH (bias) ; vi phạm điều kiện đầu vào của thuật toán
#     (thiếu làm lệch phân phối chuẩn → sai điều kiện của hồi quy tuyến tính).
#   - 3 CƠ CHẾ THIẾU:
#       MCAR – thiếu HOÀN TOÀN ngẫu nhiên (lẻ tẻ).  → điền trực tiếp mean/mode… (không thiên lệch); mô hình/MICE: hụt nhiều, có thể thiên lệch.
#       MAR  – thiếu ngẫu nhiên NHƯNG cột thiếu PHỤ THUỘC cột khác liên quan. → dùng trung bình LÂN CẬN (cùng tính chất/nhóm).
#       MNAR – thiếu KHÔNG ngẫu nhiên: thiếu phụ thuộc chính GIÁ TRỊ đó (lương quá cao/quá thấp thì không khai).
#              → dùng mô hình tìm người tương đồng ; không biết cao/thấp thì bổ sung TRI THỨC BÊN NGOÀI.
#   - Nguồn thiếu khác: mất SAU thu thập (lỗi rút trích: sai logic/ngữ nghĩa, có quy luật) ; chưa biết ; không thể thu thập.
#   - 7 BIỆN PHÁP:
#       1 Giữ nguyên (không làm sai điều kiện thuật toán; đỡ dữ liệu thưa).
#       2 Loại bỏ QUAN SÁT (dòng) – tạo mẫu cùng quy mô, công bằng (hạn chế dùng).
#       3 Loại bỏ THUỘC TÍNH (cột): thiếu nghiêm trọng quá ngưỡng hợp lý (~20%) VÀ xem tương quan (có tương quan thì cân nhắc giữ).
#       4 Thay bằng GIÁ TRỊ ĐẶC BIỆT (N/A, unknown…) hoặc THỐNG KÊ MÔ TẢ (median, mean, mode…).
#       5 Thay THEO NHÓM (phân lớp xong, lấy giá trị của nhóm/lân cận).
#       6 Thay bằng kNN (đoán ô thiếu; giá trị SỐ thì dùng mô hình hồi quy).
#       7 MICE: điều kiện số cột thiếu < số cột đầy đủ. Quy trình: điền tạm ngẫu nhiên → trả 1 biến X1 về missing → hồi quy từ các cột còn lại
#         → điền dự đoán → sang X2… (1 iteration) → lặp 5–10 vòng đến hội tụ → "Multiple": lặp độc lập m lần ra m bộ dữ liệu rồi gộp (Rubin's rules).
#   - Quy tắc quyết định giữ/bỏ: xét ô thiếu có liên quan TRỰC TIẾP đến yêu cầu phân tích không
#       (ví dụ so sánh học lực nam–nữ: ô thiếu GIỚI TÍNH → bỏ cả dòng quan sát). KHÔNG liên quan → GIỮ NGUYÊN.
#   - Hạn chế thêm dữ liệu giả cho dữ liệu THƯA (sparsity) hoặc MẤT CÂN BẰNG (imbalance).
# 🔎 CHỌN KHI NÀO – ĐI THEO TRÌNH TỰ HỎI NÀY:
#   (1) Ô thiếu có LIÊN QUAN trực tiếp đến câu hỏi phân tích không?   KHÔNG → BP1 giữ nguyên.   CÓ → hỏi tiếp.
#   (2) Nhiều cột thiếu CÙNG LÚC trên cùng các dòng (cụm dòng "rỗng")?   → BP2 loại dòng (how="all"). (đã chạy) 109 dòng thiếu đồng thời 5 cột.
#   (3) Một CỘT thiếu > ~20%?   → xem tương quan với cột khác: KHÔNG tương quan → BP3 BỎ CỘT ; CÓ tương quan đáng kể → cân nhắc giữ + BP6/BP7.
#       (đã chạy) Product Base Margin thiếu 23%, tương quan cao nhất chỉ 0.377 → bỏ.
#   (4) XÁC ĐỊNH CƠ CHẾ bằng missing_rate_by_group():
#       · tỷ lệ thiếu ĐỀU giữa các nhóm, rải rác                         → MCAR → BP4 (mean/median/mode hoặc 'Unknown').
#       · tỷ lệ thiếu KHÁC NHAU RÕ giữa các nhóm của 1 cột khác         → MAR → BP5 thay THEO NHÓM.
#           (đã chạy) Discount thiếu 33.8% ở Home Office vs ~5–7% nhóm khác → MAR theo Customer Segment.
#       · giá trị CÒN LẠI bị CỤT ĐUÔI (thiếu cực cao/cực thấp)          → MNAR → BP6/BP7 mô hình + tri thức ngoài.
#   (5) Chọn công cụ điền theo KIỂU biến:
#       · biến PHÂN LOẠI, ít thiếu, MCAR   → 'Unknown' hoặc MODE        · biến SỐ phân phối cân đối → MEAN ; LỆCH/có outlier → MEDIAN [+bổ sung]
#       · biến SỐ có tương quan với biến khác (Shipping Cost ~ Sales) → HỒI QUY / kNN
#       · biến PHÂN LOẠI cần đoán từ cột khác → kNN phân lớp (m6_knn_categorical)
#       · NHIỀU cột số thiếu và có tương quan lẫn nhau, số cột thiếu < số cột đầy đủ → MICE
#   (6) LUÔN so sánh TRƯỚC/SAU (compare_before_after): mean/std đổi nhiều ⇒ coi chừng thiên lệch.
#       (đã chạy) MICE làm mean Profit từ 82.3 → 130.9 và std 381.7 → 509.4 → thay đổi lớn, đây đúng là dấu hiệu MNAR (đuôi bị cụt)
#       → phải nêu rõ trong bài là điền bằng mô hình thì kết quả có thể thiên lệch.
# ============================================================
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer   # bật tính năng IterativeImputer (MICE của sklearn)
from sklearn.impute import IterativeImputer

# ----- Bảng thống kê thiếu -----
def mv_missing_table(d):
    n = d.isna().sum()
    t = pd.DataFrame({"Số thiếu": n, "% thiếu": (n / len(d) * 100).round(2)})
    return t[t["Số thiếu"] > 0]

# ----- Chẩn đoán cơ chế thiếu (MCAR / MAR / MNAR) -----
def missing_rate_by_group(d, col, group):
    # MAR: nếu tỷ lệ thiếu của `col` KHÁC NHAU rõ rệt giữa các nhóm của `group` → thiếu phụ thuộc cột khác
    return d[col].isna().groupby(d[group]).mean().round(3)

def compare_observed_distribution(d, col):
    # MNAR: giá trị thiếu phụ thuộc chính nó → các giá trị CÒN LẠI thường thiếu phần đuôi (cực cao/thấp)
    return d[col].describe()

# ----- BP1: GIỮ NGUYÊN (khi ô thiếu không liên quan yêu cầu phân tích) -----
def m1_keep(d):
    return d.copy()

# ----- BP2: LOẠI BỎ QUAN SÁT (dòng) -----
def m2_drop_rows(d, cols, how="all"):
    # how="all": chỉ bỏ dòng thiếu ĐỒNG THỜI tất cả cột trong cols ; how="any": thiếu bất kỳ cột nào
    return d.dropna(subset=cols, how=how)

# ----- BP3: LOẠI BỎ THUỘC TÍNH (cột) khi thiếu > ngưỡng (mặc định 20%) -----
def m3_drop_cols(d, threshold=0.20, show_corr=True):
    ratio = d.isna().mean()
    bad = ratio[ratio > threshold].index.tolist()
    if show_corr and bad:                                    # xem tương quan trước khi xóa
        num = d.select_dtypes("number")
        for c in bad:
            if c in num.columns:
                print(f"Tương quan của '{c}' với các cột số khác:\n", num.corr()[c].drop(c).round(3).to_string())
    return d.drop(columns=bad), bad

# ----- BP4a: THAY BẰNG GIÁ TRỊ ĐẶC BIỆT -----
def m4_fill_special(d, col, value="Unknown"):
    d = d.copy()
    d[col] = d[col].fillna(value)
    return d

# ----- BP4b: THAY BẰNG THỐNG KÊ MÔ TẢ (mean / median / mode) -----
def m4_fill_stat(d, col, stat="median"):
    d = d.copy()
    if stat == "mean":     v = d[col].mean()
    elif stat == "median": v = d[col].median()
    else:                  v = d[col].mode()[0]              # mode: giá trị xuất hiện nhiều nhất
    d[col] = d[col].fillna(v)
    return d

# ----- BP5: THAY THEO NHÓM (dùng cho MAR) -----
def m5_group_fill(d, col, group, stat="median"):
    d = d.copy()
    # transform: tính thống kê của TỪNG NHÓM rồi gán lại đúng dòng
    d[col] = d[col].fillna(d.groupby(group)[col].transform(stat))
    d[col] = d[col].fillna(d[col].median())                  # nhóm nào thiếu hết thì dùng median toàn cột
    return d

# ----- BP6a: kNN cho dữ liệu SỐ -----
def m6_knn_numeric(d, cols, k=5):
    d = d.copy()
    mu, sd = d[cols].mean(), d[cols].std()
    z = (d[cols] - mu) / sd                                   # chuẩn hóa để cột lớn không áp đảo khoảng cách
    z[:] = KNNImputer(n_neighbors=k).fit_transform(z)         # điền bằng trung bình k hàng xóm gần nhất
    d[cols] = z * sd + mu                                     # trả về thang đo cũ
    return d

# ----- BP6b: kNN cho dữ liệu PHÂN LOẠI (phân lớp) -----
def m6_knn_categorical(d, target, features, k=5):
    d = d.copy()
    known = d[target].notna() & d[features].notna().all(axis=1)    # dòng đủ để HỌC
    unknown = d[target].isna() & d[features].notna().all(axis=1)   # dòng cần ĐOÁN
    mu, sd = d.loc[known, features].mean(), d.loc[known, features].std()
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit((d.loc[known, features] - mu) / sd, d.loc[known, target])
    d.loc[unknown, target] = model.predict((d.loc[unknown, features] - mu) / sd)
    return d

# ----- BP6c: HỒI QUY cho dữ liệu SỐ -----
def m6_regression(d, target, features):
    d = d.copy()
    known = d[target].notna() & d[features].notna().all(axis=1)
    unknown = d[target].isna() & d[features].notna().all(axis=1)
    model = LinearRegression().fit(d.loc[known, features], d.loc[known, target])
    d.loc[unknown, target] = model.predict(d.loc[unknown, features])
    return d

# ----- BP7: MICE viết tay theo đúng các bước trong notes -----
def m7_mice_manual(d, cols, n_iter=5, m=3, seed=0):
    results = []
    for k in range(m):                                        # "Multiple": lặp độc lập m lần → m bộ dữ liệu
        rng = np.random.RandomState(seed + k)
        X = d[cols].copy()
        mask = X.isna()                                       # nhớ vị trí ô thiếu
        # B1: điền tạm các ô thiếu bằng giá trị NGẪU NHIÊN lấy từ giá trị đã quan sát
        for c in cols:
            X.loc[mask[c], c] = rng.choice(X[c].dropna().values, mask[c].sum())
        order = sorted(cols, key=lambda c: mask[c].sum())     # cột thiếu ÍT xử lý trước
        # B2-B4: mỗi iteration = lần lượt từng cột thiếu làm target
        for it in range(n_iter):                              # lặp 5–10 vòng cho đến khi hội tụ
            for c in order:
                if mask[c].sum() == 0: continue
                others = [o for o in cols if o != c]          # các cột còn lại làm predictor
                train = ~mask[c]                              # dòng có giá trị thật → để học
                model = LinearRegression().fit(X.loc[train, others], X.loc[train, c])
                X.loc[mask[c], c] = model.predict(X.loc[mask[c], others])   # điền giá trị dự đoán
        results.append(X)
    pooled = sum(results) / m                                 # gộp m bộ (đơn giản: trung bình)
    out = d.copy()
    out[cols] = pooled
    return out, results

# ----- BP7 (bản thư viện để đối chiếu) -----
def m7_mice_sklearn(d, cols, n_iter=5):
    d = d.copy()
    d[cols] = IterativeImputer(max_iter=n_iter, random_state=0).fit_transform(d[cols])
    return d

# ----- So sánh TRƯỚC / SAU xử lý (quy tắc 2) -----
def compare_before_after(before, after, cols):
    t = pd.DataFrame({"mean_trước": before[cols].mean(), "mean_sau": after[cols].mean(),
                      "std_trước": before[cols].std(), "std_sau": after[cols].std()})
    return t.round(3)

# --- 5.2 Chẩn đoán ---
print("=== Bảng thiếu ==="); display(mv_missing_table(df))
print("MAR? Tỷ lệ thiếu Discount theo Customer Segment:\n", missing_rate_by_group(df, "Discount", "Customer Segment"))
print("MAR? Tỷ lệ thiếu Unit Price theo Sub-Category:\n", missing_rate_by_group(df, "Unit Price", "Product Sub-Category").sort_values().tail(5))
print("MNAR? Profit còn lại (thiếu đuôi cực trị):\n", compare_observed_distribution(df, "Profit"))

In [ ]:
# ============================================================
# 5.3 ÁP DỤNG TUẦN TỰ 7 BIỆN PHÁP LÊN SUPERSTORE  →  df_clean
# (mỗi bước trả về DataFrame mới, df gốc không đổi)
# ------------------------------------------------------------
# 🔎 VÌ SAO CHỌN BIỆN PHÁP NÀY CHO TỪNG CỘT (tóm tắt theo dấu hiệu ở ô 5.1):
#   - 5 cột thiếu ĐỒNG THỜI trên cùng 109 dòng            → BP2 loại dòng.
#   - Product Base Margin thiếu 23% > ngưỡng 20%, ít tương quan → BP3 loại cột.
#   - Ship Mode (biến phân loại, thiếu rải rác – MCAR)      → BP4 'Unknown' (hoặc mode).
#   - Discount / Unit Price: tỷ lệ thiếu KHÁC nhau theo nhóm → MAR → BP5 thay theo nhóm (Customer Segment / Sub-Category).
#   - Shipping Cost, Profit: biến SỐ, tương quan với các cột số khác, nhiều cột cùng thiếu → BP7 MICE.
# ============================================================
step = df.copy()

# BP2 – Loại 109 dòng thiếu đồng thời 5 cột quan trọng
step = m2_drop_rows(step, ["Sales", "Order Quantity", "Customer Name", "City", "Ship Mode"], how="all")
print("BP2: số dòng", len(df), "→", len(step))

# BP3 – Loại cột thiếu > 20% (Product Base Margin ≈ 23%)
step, dropped = m3_drop_cols(step, threshold=0.20, show_corr=True)
print("BP3: đã bỏ cột", dropped)

# BP4 – Ship Mode (MCAR): thay bằng 'Unknown'  (hoặc mode: m4_fill_stat(step,'Ship Mode','mode'))
step = m4_fill_special(step, "Ship Mode", "Unknown")

# BP5 – Discount (MAR theo Customer Segment) & Unit Price (MAR theo Sub-Category): thay theo nhóm
step = m5_group_fill(step, "Discount", "Customer Segment", "median")
step = m5_group_fill(step, "Unit Price", "Product Sub-Category", "median")

# BP7 – Shipping Cost & Profit (MNAR): dùng MICE (mô hình dự đoán từ các cột số còn lại)
cols_mice = ["Order Quantity", "Sales", "Discount", "Unit Price", "Shipping Cost", "Profit"]
step_mice, mice_sets = m7_mice_manual(step, cols_mice, n_iter=5, m=3)
print("So sánh trước/sau MICE:"); display(compare_before_after(step, step_mice, ["Shipping Cost", "Profit"]))

df_clean = step_mice.copy()                          # ✅ dữ liệu đã làm sạch missing (dùng cho các phần sau)
print("Còn thiếu sau xử lý:", int(df_clean.isna().sum().sum()), "| kích thước:", df_clean.shape)

# (Tham khảo các biện pháp còn lại – chạy thử trên 1 cột)
df_try = df.dropna(subset=["Sales"])
_ = m6_knn_numeric(df_try, ["Order Quantity", "Sales", "Shipping Cost"])             # BP6a kNN số
_ = m6_knn_categorical(df_try, "Ship Mode", ["Order Quantity", "Sales"])             # BP6b kNN phân loại
_ = m6_regression(df_try, "Shipping Cost", ["Order Quantity", "Sales"])              # BP6c hồi quy
_ = m7_mice_sklearn(df_try, ["Order Quantity", "Sales", "Shipping Cost"])            # MICE sklearn
print("Đã chạy thử BP6a/6b/6c và MICE sklearn.")

## PHẦN 6 – LÀM SẠCH: DỮ LIỆU BẤT THƯỜNG – OUTLIERS (Trang 13–14, 23)

In [ ]:
# ============================================================
# 6.1 NHÓM HÀM: Phát hiện + xử lý outliers
# ------------------------------------------------------------
# 📖 TRANG 13-14:
#   3 mức bất thường: attribute (khác tổng thể) / item (khác lân cận) / feature (biến thiên quá thấp)
#   Phát hiện: 3-Sigma (z-test), Euclid (hình tròn), Mahalanobis (hình bầu dục, khác thang đo)
#   Xử lý: Log transform (số dương, lệch phải) | Box-Cox (dương) | Yeo-Johnson (âm/dương đều được)
#          Binning (làm trơn cục bộ) | Winsorization (kìm hãm theo percentile) | Dữ liệu thời gian (ngày trước, cùng kỳ năm trước, nội suy)
#   TRANG 23: phát hiện bằng z-test hoặc boxplot; bất đẳng thức Chebyshev cho tỷ lệ dữ liệu giữ lại
# 🎯 DÙNG KHI: bước "(2) dữ liệu bất thường" trong đánh giá chất lượng dữ liệu (trang 15).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Làm sạch: dữ liệu bất thường]:
#   - EDA chỉ NHẬN DIỆN bất thường – tích cực hay tiêu cực KHÔNG quan tâm (outlier ≠ dữ liệu sai).
#   - 3 MỨC bất thường: (1) ATTRIBUTE: giá trị khác biệt so với TỔNG THỂ ; (2) ITEM: không khác tổng thể nhưng khác LÂN CẬN ;
#     (3) FEATURE: thuộc tính vô dụng, dư thừa, ĐỘ BIẾN THIÊN QUÁ THẤP.
#   - TÁC ĐỘNG tiêu cực: làm lệch trung bình, luôn TĂNG phương sai, gây OVERFITTING (quá sát outlier, kém tổng quát).
#   - PHÁT HIỆN bằng thống kê:
#       · 3-SIGMA (z-test, ý nghĩa: KHỬ ĐƠN VỊ): cách trung bình quá 3 độ lệch chuẩn (3 "layers") là outlier.
#       · 2 biến: vẽ VÒNG TRÒN quanh TRỌNG TÂM (trọng tâm = trung bình các vector) với bán kính R ; điểm nào cách tâm > R là outlier.
#         EUCLID (hình tròn) chỉ ổn khi các chiều CÙNG ĐƠN VỊ/THANG ĐO ; KHÁC thang đo → MAHALANOBIS (hình BẦU DỤC).
#   - XỬ LÝ:
#       · LOG: dữ liệu DƯƠNG, LỆCH PHẢI (thu hẹp khoảng biến thiên); nhược điểm: chỉ dùng với số dương.
#       · BOX-COX: kéo về phân phối chuẩn; Word: dùng được cả số âm nếu âm HẠN CHẾ (ít) – thực tế thư viện scipy yêu cầu x > 0
#         (nếu có âm/0 thì cộng hằng số dịch lên dương trước).
#       · YEO-JOHNSON: kéo về phân phối chuẩn, phương sai gần dữ liệu thô, ổn định; ÂM hay DƯƠNG đều dùng được, không hạn chế.
#       · BINNING (làm trơn cục bộ): chia bins cùng kích thước/cùng phân phối rồi thay bằng mean/median/mode của bin.
#       · WINSORIZATION (kìm hãm): kéo outlier về NGƯỠNG xác định trước (percentile hoặc khoảng tin cậy) – thay bằng giá trị lớn nhất không phải outlier.
#       · DỮ LIỆU THỜI GIAN: thay bằng giá trị NGÀY TRƯỚC ; thay bằng CÙNG KỲ NĂM TRƯỚC ; nội suy tuyến tính giữa ngày trước và ngày sau.
#   - CHEBYSHEV [Word]: tỷ lệ dữ liệu GIỮ LẠI trong ±zσ ≥ 1 − 1/z² (z ≥ 2). Nếu 2 phía lệch KHÔNG bằng nhau → lấy MIN của 2 z.
#   - Cách phát hiện: z-test (phân phối chuẩn) hoặc BOXPLOT.
# 🔎 CHỌN KHI NÀO:
#   A) CHỌN CÁCH PHÁT HIỆN:
#     - Biến có phân phối GẦN CHUẨN (mean ≈ median, skew ≈ 0)                 → 3-SIGMA |z| > 3.
#     - Biến LỆCH MẠNH/đuôi dài (mean ≫ median)                               → IQR/BOXPLOT (không giả định chuẩn) hoặc biến đổi (log) rồi mới z-test.
#         (đã chạy) Sales skew = 5.39: 3-sigma chỉ bắt 206 dòng, IQR bắt 1027 dòng → vì mean/std bị chính outlier kéo lên nên 3-sigma "dễ dãi".
#     - Xét 2 biến CÙNG đơn vị → EUCLID (hình tròn) ; KHÁC thang đo/2 biến có tương quan → MAHALANOBIS (hình bầu dục).
#         (đã chạy) Euclid bắt 54 điểm, Mahalanobis bắt 395 điểm → hai cách cho kết quả rất khác nhau, chọn theo tính chất thang đo.
#     - Không biết phân phối nhưng cần biết "bỏ outlier thì còn lại bao nhiêu" → CHEBYSHEV (z=2 → ≥ 75% ; z=3 → ≥ 88.9%).
#     - MỨC bất thường: khác tổng thể → z/IQR ; khác lân cận → khoảng cách/kNN ; biến thiên quá thấp → loại cột (variance threshold, Phần 11).
#   B) CHỌN CÁCH XỬ LÝ:
#     - Số DƯƠNG, lệch phải mạnh, muốn nén khoảng biến thiên                  → LOG.  (đã chạy) skew Sales 5.39 → 0.08.
#     - Số DƯƠNG, muốn máy tự tìm λ để gần chuẩn nhất                          → BOX-COX (đã chạy: λ ≈ −0.013, skew → 0.0).
#     - Có số ÂM hoặc 0 (Profit có lỗ)                                        → YEO-JOHNSON.  (đã chạy) skew Profit 2.47 → −0.67.
#     - MUỐN GIỮ NGUYÊN số quan sát, chỉ kéo ngoại lai vào ngưỡng             → WINSORIZE (percentile 5–95%). Lưu ý: chỉ cắt đuôi, KHÔNG đổi dạng phân phối
#         (đã chạy) skew Sales chỉ giảm 5.39 → 1.91, kém log nhiều ⇒ nếu mục tiêu là "chuẩn hóa phân phối" thì log/Box-Cox tốt hơn.
#     - Muốn làm trơn nhiễu cục bộ từng khoảng                                → BINNING.
#     - Dữ liệu CHUỖI THỜI GIAN: outlier đơn lẻ giữa 2 điểm bình thường → NỘI SUY ; chuỗi có MÙA VỤ → CÙNG KỲ NĂM TRƯỚC ; chuỗi đều, biến động nhỏ → NGÀY/KỲ TRƯỚC.
#     - Outlier là đơn hàng LỚN THẬT (có nghĩa nghiệp vụ) → đừng xóa; ghi nhận hoặc biến đổi/winsorize và nêu lý do.
# ============================================================
# ----- Phát hiện bằng 3-SIGMA: |z| > 3 với z = (x - mean)/std -----
def zscore_outliers(d, col, k=3):
    x = d[col]
    z = (x - x.mean()) / x.std(ddof=1)            # z-test: khử đơn vị
    return d[np.abs(z) > k]

# ----- Phát hiện bằng BOXPLOT/IQR: ngoài [Q1-1.5IQR , Q3+1.5IQR] -----
def iqr_outliers(d, col):
    x = d[col]
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return d[(x < low) | (x > high)], (low, high)

# ----- Bất đẳng thức CHEBYSHEV: tỷ lệ dữ liệu GIỮ LẠI trong ±z σ ≥ 1 - 1/z² -----
def chebyshev_keep_ratio(z):
    return 1 - 1 / (z ** 2)

# ----- 2 biến bằng khoảng cách EUCLID (hình tròn) – chỉ ổn khi CÙNG THANG ĐO -----
def euclid_outliers(d, c1, c2, k=3):
    X = d[[c1, c2]].values
    center = X.mean(axis=0)                                  # trọng tâm = trung bình các vector
    dist = np.sqrt(((X - center) ** 2).sum(axis=1))          # khoảng cách từ mỗi điểm tới tâm
    R = dist.mean() + k * dist.std()                         # bán kính vòng tròn (k lớp sigma)
    return d[dist > R]

# ----- 2 biến bằng MAHALANOBIS (hình bầu dục) – dùng khi KHÁC THANG ĐO -----
def mahalanobis_outliers(d, c1, c2, k=3):
    X = d[[c1, c2]].values
    mu = X.mean(axis=0)
    S_inv = np.linalg.inv(np.cov(X.T))                       # nghịch đảo ma trận hiệp phương sai
    diff = X - mu
    dist = np.sqrt(np.einsum("ij,jk,ik->i", diff, S_inv, diff))   # d = căn((x-μ)ᵀ S⁻¹ (x-μ))
    return d[dist > k]

# ----- XỬ LÝ 1: LOG transform – chỉ số DƯƠNG, lệch phải -----
def log_transform(d, col):
    return np.log1p(d[col].clip(lower=0))                    # log(1+x) để an toàn khi x = 0

# ----- XỬ LÝ 2: BOX-COX – dữ liệu phải DƯƠNG (>0) -----
def boxcox_transform(d, col):
    x = d[col]
    y, lam = stats.boxcox(x[x > 0])                          # trả về (dữ liệu mới, lambda tối ưu)
    return pd.Series(y, index=x[x > 0].index), lam

# ----- XỬ LÝ 3: YEO-JOHNSON – âm hay dương đều dùng được -----
def yeojohnson_transform(d, col):
    y, lam = stats.yeojohnson(d[col].values)
    return pd.Series(y, index=d.index), lam

# ----- XỬ LÝ 4: BINNING – chia bins cùng tần số rồi thay bằng mean của bin -----
def binning_smooth(d, col, n_bins=10):
    bins = pd.qcut(d[col], q=n_bins, duplicates="drop")      # chia n bins cùng số phần tử
    return d.groupby(bins)[col].transform("mean")            # thay mỗi giá trị bằng mean của bin

# ----- XỬ LÝ 5: WINSORIZATION – kìm hãm về ngưỡng percentile -----
def winsorize(d, col, lower=0.05, upper=0.95):
    lo, hi = d[col].quantile(lower), d[col].quantile(upper)  # ngưỡng percentile
    return d[col].clip(lower=lo, upper=hi)                   # <lo → lo ; >hi → hi

# ----- XỬ LÝ 6: DỮ LIỆU THỜI GIAN -----
def time_replace(series, outlier_mask, method="previous"):
    s = series.copy()
    s[outlier_mask] = np.nan                                  # đánh dấu outlier là thiếu
    if method == "previous":  return s.ffill().bfill()        # thay bằng giá trị NGÀY/KỲ TRƯỚC
    if method == "last_year": return s.fillna(series.shift(12)).ffill().bfill()   # CÙNG KỲ NĂM TRƯỚC (dữ liệu theo tháng)
    if method == "linear":    return s.interpolate(method="linear", limit_direction="both")   # nội suy tuyến tính

# --- Sử dụng (trên df_clean đã làm sạch missing) ---
print("3-sigma  Sales:", len(zscore_outliers(df_clean, "Sales")), "dòng")
iqr_rows, bounds = iqr_outliers(df_clean, "Sales")
print("IQR      Sales:", len(iqr_rows), "dòng | ngưỡng:", np.round(bounds, 2))
print("Chebyshev z=3 giữ lại ≥", round(chebyshev_keep_ratio(3) * 100, 1), "% dữ liệu")
print("Euclid (Order Quantity vs Unit Price):", len(euclid_outliers(df_clean, "Order Quantity", "Unit Price")))
print("Mahalanobis (Sales vs Profit):", len(mahalanobis_outliers(df_clean, "Sales", "Profit")))

# So sánh độ lệch (skew) trước/sau các phép biến đổi cho cột Sales
print("\nSkew Sales gốc      :", round(df_clean["Sales"].skew(), 2))
print("Skew sau Log        :", round(log_transform(df_clean, "Sales").skew(), 2))
bc, lam_bc = boxcox_transform(df_clean, "Sales");     print("Skew sau Box-Cox    :", round(bc.skew(), 2), "| lambda =", round(lam_bc, 3))
yj, lam_yj = yeojohnson_transform(df_clean, "Profit"); print("Skew Profit gốc/Yeo-Johnson:", round(df_clean["Profit"].skew(), 2), "→", round(yj.skew(), 2))
print("Skew sau Winsorize  :", round(winsorize(df_clean, "Sales").skew(), 2))
print("Binning (5 giá trị đầu):", binning_smooth(df_clean, "Sales").head().round(2).tolist())

# --- Dữ liệu thời gian: doanh số theo THÁNG, thay outlier ---
monthly = df_clean.set_index("Order Date")["Sales"].resample("MS").sum()      # tổng Sales mỗi tháng
zm = (monthly - monthly.mean()) / monthly.std(ddof=1)
mask = np.abs(zm) > 2                                                        # outlier tháng (|z|>2)
print("\nSố tháng bất thường:", int(mask.sum()))
print("Thay bằng kỳ trước:", time_replace(monthly, mask, "previous")[mask].round(0).tolist())
print("Thay bằng nội suy  :", time_replace(monthly, mask, "linear")[mask].round(0).tolist())
print("Thay cùng kỳ năm trước:", time_replace(monthly, mask, "last_year")[mask].round(0).tolist())
print("Giá trị gốc của tháng bất thường:", monthly[mask].round(0).tolist())

plt.boxplot([df_clean["Sales"], winsorize(df_clean, "Sales")]); plt.xticks([1, 2], ["Gốc", "Winsorize 5-95%"]); plt.title("Sales trước/sau Winsorize"); plt.show()

## PHẦN 7 – KIỂM ĐỊNH GIẢ THUYẾT (Trang 15, 17)

In [ ]:
# ============================================================
# 7.1 NHÓM HÀM: z-test, t-test, ANOVA, Chi-square
# ------------------------------------------------------------
# 📖 TRANG 15: Kiểm định giả thuyết dựa trên lấy mẫu.
#   - H0 (null): điều cho là đúng; dấu "=" BẮT BUỘC để ở H0; giả thuyết muốn BÁC BỎ để ở H0.
#   - H1/Ha: giả thuyết muốn ỦNG HỘ.
#   - Kiểm định 1 phía (one tail) / 2 phía (two tails, dấu "=").
#   - Ma trận nhầm lẫn: Sai lầm loại I (bác bỏ H0 khi H0 đúng) – nghiêm trọng hơn;
#                       Sai lầm loại II (không bác bỏ H0 khi H0 sai).
# 📖 TRANG 10, 16-17: numerical & categorical → z-test, t-test (nhị phân), ANOVA (nhiều nhóm); 2 categorical → Chi-square.
# 🎯 DÙNG KHI: xem 2 biến có liên quan không (để giữ/bỏ cột dư thừa), so sánh trung bình giữa các nhóm.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Kiểm định giả thuyết] (dựa trên lấy mẫu):
#   - H0 (null): điều phát biểu thường được cho là đúng. Ha/H1: giả thuyết còn lại.
#   - Đặt ở đâu: (1) kiểm định dấu "=" → BẮT BUỘC ở H0 ; (2) giả thuyết muốn BÁC BỎ → để ở H0 ;
#                (3) giả thuyết muốn ỦNG HỘ → để ở H1/Ha. Cần XOAY phát biểu sao cho dấu ở H0 đồng bộ (≤ , ≥ , =).
#   - 1 PHÍA (one tail): hằng số nằm bên nào thì kiểm định bên đó ; 2 PHÍA (two tails): kiểm định dấu "=".
#   - Đúng/sai của H0 đã có sẵn trước khi kiểm định (ta không biết) → kiểm định KHÔNG chứng minh đúng/sai, chỉ xác lập NIỀM TIN: chấp nhận hay bác bỏ.
#   - MA TRẬN NHẦM LẪN:  Không bác bỏ H0 & H0 đúng → ĐÚNG ; Không bác bỏ H0 & H0 sai → SAI LẦM LOẠI II (noise) ;
#                        Bác bỏ H0 & H0 đúng → SAI LẦM LOẠI I (silence) ; Bác bỏ H0 & H0 sai → ĐÚNG.
#     Tùy hoàn cảnh mà loại I hay II nguy hiểm hơn ; NẾU BỎ QUA hoàn cảnh thì sai lầm LOẠI I nghiêm trọng hơn.
#   - Dùng để: numerical & categorical → z-test, t-test (biến phân loại NHỊ PHÂN), ANOVA (NHIỀU giá trị) ;
#     2 biến phân loại → Chi-square ; trong tích hợp: nhận diện thông tin dư thừa.
# 🔎 CHỌN KHI NÀO – CHỌN PHÉP KIỂM ĐỊNH THEO DẠNG DỮ LIỆU:
#   ┌──────────────────────────────────────────────┬─────────────────────────────────────────────────────────────┐
#   │ So 1 trung bình mẫu với 1 HẰNG SỐ            │ z-test 1 mẫu (mẫu lớn ≥30 hoặc biết σ) [+bổ sung]; mẫu nhỏ → t │
#   │ So trung bình của 2 NHÓM (cat nhị phân)      │ t-test 2 mẫu (H0: μ1 = μ2)                                  │
#   │ So trung bình của ≥3 NHÓM (cat nhiều giá trị)│ ANOVA (H0: mọi nhóm cùng trung bình)                        │
#   │ 2 biến PHÂN LOẠI có liên quan không          │ Chi-square (H0: 2 biến ĐỘC LẬP)                             │
#   └──────────────────────────────────────────────┴─────────────────────────────────────────────────────────────┘
#   - CHỌN 1 PHÍA hay 2 PHÍA: đề có dấu "=" / "khác" → 2 phía ; đề có "lớn hơn / nhỏ hơn" → 1 phía theo hướng của H1.
#   - ĐỌC KẾT QUẢ:  p-value < α (thường 0.05) → BÁC BỎ H0 (có bằng chứng ủng hộ H1) ;
#                   p-value ≥ α → KHÔNG BÁC BỎ H0 (chưa đủ bằng chứng – KHÔNG có nghĩa H0 đúng).
#   - ÁP DỤNG RA QUYẾT ĐỊNH VỚI DỮ LIỆU:
#       · ANOVA/t-test p < α  → biến phân loại CÓ ẢNH HƯỞNG đến biến số → GIỮ làm input.
#         (đã chạy) Profit theo Product Category: F = 153, p ≈ 0 → loại hàng ảnh hưởng lợi nhuận → giữ.
#       · p ≥ α → 2 nhóm không khác biệt → biến ít giá trị phân tích, có thể bỏ/gộp.
#         (đã chạy) Shipping Cost Express Air vs Regular Air: p = 0.389 → chưa thấy khác biệt.
#       · Chi-square p < α → 2 biến phân loại PHỤ THUỘC nhau (đã chạy: Region × Segment p ≈ 0) → có thể là thông tin dư thừa nếu quá liên quan.
#   - Nếu hậu quả của việc "bác bỏ nhầm" rất lớn (kết luận sai gây thiệt hại) → hạ α (0.01) để giảm sai lầm loại I. [+bổ sung]
# ============================================================
def decide(p, alpha=0.05):          # alpha = mức ý nghĩa (thường 0.05)
    # Quy tắc: p-value < alpha → BÁC BỎ H0 ; ngược lại → KHÔNG đủ bằng chứng bác bỏ H0
    return "BÁC BỎ H0" if p < alpha else "KHÔNG BÁC BỎ H0"

# ----- z-test 1 mẫu: z = (x̄ - μ0) / (s/√n) -----
def z_test_one_sample(d, col, mu0, tail="two", alpha=0.05):
    x = d[col].dropna()
    z = (x.mean() - mu0) / (x.std(ddof=1) / math.sqrt(len(x)))
    if tail == "two":     p = 2 * (1 - stats.norm.cdf(abs(z)))     # 2 phía: H1 μ ≠ μ0
    elif tail == "right": p = 1 - stats.norm.cdf(z)                # 1 phía phải: H1 μ > μ0
    else:                 p = stats.norm.cdf(z)                    # 1 phía trái: H1 μ < μ0
    return {"z": round(z, 3), "p-value": round(p, 4), "Kết luận": decide(p, alpha)}

# ----- t-test 2 nhóm (biến phân loại NHỊ PHÂN) -----
def t_test_two_groups(d, num, cat, g1, g2, alpha=0.05):
    a = d.loc[d[cat] == g1, num].dropna()
    b = d.loc[d[cat] == g2, num].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)               # H0: μ1 = μ2
    return {"t": round(t, 3), "p-value": round(p, 4), "Kết luận": decide(p, alpha)}

# ----- ANOVA (biến phân loại NHIỀU giá trị) -----
def anova(d, num, cat, alpha=0.05):
    groups = [g[num].dropna() for _, g in d.groupby(cat)]
    f, p = stats.f_oneway(*groups)                              # H0: trung bình mọi nhóm bằng nhau
    return {"F": round(f, 3), "p-value": round(p, 4), "Kết luận": decide(p, alpha)}

# ----- Chi-square: 2 biến PHÂN LOẠI -----
def chi_square(d, c1, c2, alpha=0.05):
    table = pd.crosstab(d[c1], d[c2])                           # bảng chéo
    chi2, p, dof, _ = stats.chi2_contingency(table)             # H0: 2 biến ĐỘC LẬP
    return {"chi2": round(chi2, 3), "dof": dof, "p-value": round(p, 4), "Kết luận": decide(p, alpha)}

# ----- Bảng ma trận nhầm lẫn -----
def confusion_table():
    return pd.DataFrame({"H0 ĐÚNG (bản chất)": ["Kết luận đúng", "Sai lầm loại I (silence)"],
                         "H0 SAI (bản chất)": ["Sai lầm loại II (noise)", "Kết luận đúng"]},
                        index=["KHÔNG bác bỏ H0", "BÁC BỎ H0"])

print("z-test 2 phía  H0: mean Discount = 0.05      →", z_test_one_sample(df_clean, "Discount", 0.05))
print("z-test 1 phía  H0: mean Shipping Cost ≤ 12 ; H1: > 12 →", z_test_one_sample(df_clean, "Shipping Cost", 12, tail="right"))
print("t-test  Shipping Cost: Express Air vs Regular Air →", t_test_two_groups(df_clean, "Shipping Cost", "Ship Mode", "Express Air", "Regular Air"))
print("ANOVA   Profit theo Product Category →", anova(df_clean, "Profit", "Product Category"))
print("Chi²    Region × Customer Segment →", chi_square(df_clean, "Region", "Customer Segment"))
display(confusion_table())

## PHẦN 8 – TÍCH HỢP DỮ LIỆU (Trang 10, 15–17)

In [ ]:
# ============================================================
# 8.1 NHÓM HÀM: Tích hợp nhiều nguồn + nhận diện thông tin dư thừa
# ------------------------------------------------------------
# 📖 TRANG 16: Phương án tích hợp: Consolidation (gom về 1 nơi, nhất quán) | Propagation (sao chép xuống các đơn vị, giảm tải)
#              | Federation (interface kết nối nguồn, data virtualization) | Data Warehouse (lưu theo chủ đề).
#   Nhận diện dư thừa: (1) thuộc tính dẫn xuất (2) hiệp phương sai (3) tương quan Pearson/Spearman
#                      (4) Chi-square (5) T-test/ANOVA.
#   TRANG 15: Dạng chuẩn 3 (không phụ thuộc bắc cầu, không trùng lấp). Metadata = dữ liệu mô tả dữ liệu.
# 🎯 DÙNG KHI: ghép sheet Orders + Returns + Users; bỏ bớt cột dư thừa trước khi phân tích.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Tích hợp dữ liệu]:
#   - 4 PHƯƠNG ÁN: (1) CONSOLIDATION: sao chép nhiều nguồn về 1 nơi (server chung) → ưu điểm NHẤT QUÁN.
#                  (2) PROPAGATION: sao chép dữ liệu XUỐNG các đơn vị → ưu điểm GIẢM TẢI.
#                  (3) FEDERATION (data virtualization): dùng INTERFACES kết nối các nguồn; dữ liệu nào không cục bộ thì tự đi lấy, đóng gói lại;
#                      lưu ý đường truyền phải tốt.
#                  (4) DATA WAREHOUSE: lưu trong KHO dữ liệu, theo CHỦ ĐỀ (khác lưu theo chiều ngang mỗi site một nơi).
#   - Khi tích hợp dữ liệu NGOÀI thì phải TIỀN XỬ LÝ trước.
#   - DẠNG CHUẨN 3 khi thiết kế CSDL: không có phụ thuộc BẮC CẦU, không trùng lấp thông tin → hạn chế bất nhất quán.
#   - METADATA = dữ liệu mô tả dữ liệu (ngày tạo, kích thước…).
#   - NHẬN DIỆN THÔNG TIN DƯ THỪA: (1) thuộc tính DẪN XUẤT ; (2) phân tích HIỆP PHƯƠNG SAI ; (3) phân tích TƯƠNG QUAN: Pearson, Spearman
#       (mã hóa biến phân loại thành số ít nhiều làm đổi bản chất bài toán) ; (4) kiểm định CHI-SQUARE ;
#       (5) kiểm định T / ANOVA: biến phân loại NHỊ PHÂN → T ; NHIỀU giá trị → ANOVA.
# 🔎 CHỌN KHI NÀO:
#   - Nhiều bảng/sheet về cùng 1 thực thể, cần 1 bảng duy nhất để phân tích (Orders + Returns + Users) → CONSOLIDATION bằng merge/join.
#   - JOIN: how="left" khi muốn GIỮ MỌI dòng của bảng chính (Orders) ; sau join PHẢI kiểm tra số dòng không đổi.
#       DẤU HIỆU lỗi: số dòng TĂNG sau join → khóa ở bảng phụ bị TRÙNG (1–nhiều) → drop_duplicates trước khi join.
#       DẤU HIỆU không khớp: cột mới toàn NaN → sai tên khóa/khác kiểu dữ liệu/khác cách viết.
#   - Nguồn nằm rải rác, KHÔNG được sao chép, mạng tốt → FEDERATION ; cần phân tích lịch sử theo chủ đề → DATA WAREHOUSE.
#   - NHẬN DIỆN CỘT DƯ THỪA – nhìn vào đâu:
#       · Cột = công thức của các cột khác, tỷ số calc/Sales ≈ 1 (đã chạy: trung vị 0.944) → Sales là cột DẪN XUẤT → nghi dư thừa, cân nhắc bỏ.
#       · 2 biến SỐ có |Pearson| cao (≥ ~0.8 [+bổ sung]) → dư thừa, bỏ bớt 1. (đã chạy) Sales–Profit 0.66 (< 0.8) → chưa kết luận dư thừa.
#       · PEARSON (liên tục, quan hệ tuyến tính, ít outlier) ; SPEARMAN (lệch, nhiều outlier, ordinal, quan hệ đơn điệu không tuyến tính).
#         (đã chạy) Sales–Unit Price: Pearson 0.59 nhưng Spearman 0.86 → quan hệ đơn điệu rất mạnh, Pearson bị outlier làm yếu → dùng Spearman để kết luận.
#       · 2 biến PHÂN LOẠI → Chi-square ; biến phân loại NHỊ PHÂN vs số → t-test ; NHIỀU nhóm vs số → ANOVA.
#   - Một cột phụ thuộc "bắc cầu" (Region → Manager) → tách thành bảng riêng (Users) cho đạt dạng chuẩn 3.
# ============================================================
# ----- Consolidation: ghép Orders + Returns + Users về MỘT bảng -----
def consolidate(orders, path, returns_sheet="Returns", users_sheet="Users", order_key="Order ID", region_key="Region", status_col="Status"):
    returns = read_other_sheet(path, returns_sheet).drop_duplicates(order_key)
    users = read_other_sheet(path, users_sheet)
    users = users.drop_duplicates(region_key)             # Region→Manager: mỗi vùng giữ 1 người (tránh nhân đôi dòng)
    d = orders.merge(returns, on=order_key, how="left")   # left join: giữ mọi đơn hàng
    d[status_col] = d[status_col].fillna("Not Returned")  # đơn không có trong Returns = không bị trả
    d = d.merge(users, on=region_key, how="left")         # thêm tên quản lý vùng
    return d

# ----- Metadata: dữ liệu mô tả dữ liệu -----
def metadata(d):
    return pd.DataFrame({"kiểu": d.dtypes.astype(str), "số giá trị khác nhau": d.nunique(),
                         "thiếu": d.isna().sum()})

# ----- (1) Thuộc tính DẪN XUẤT: kiểm tra Sales ≈ Unit Price × Quantity × (1 - Discount) -----
def derived_check(d, total_col="Sales", price_col="Unit Price", qty_col="Order Quantity", disc_col="Discount"):
    x = d.dropna(subset=[total_col, price_col, qty_col, disc_col])
    calc = x[price_col] * x[qty_col] * (1 - x[disc_col])
    return (calc / x[total_col]).describe()                 # tỷ số GẦN 1 → Sales gần như là cột DẪN XUẤT (nghi dư thừa)

# ----- (2) Hiệp phương sai & (3) Tương quan Pearson / Spearman -----
def covariance_matrix(d, cols):
    return d[cols].cov()

def pearson_matrix(d, cols):
    return d[cols].corr(method="pearson")                 # tương quan tuyến tính (khử đơn vị)

def spearman_matrix(d, cols):
    return d[cols].corr(method="spearman")                # tương quan thứ hạng

def redundant_pairs(d, cols, thr=0.8):
    c = d[cols].corr().abs()
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if c.iloc[i, j] >= thr:
                pairs.append((cols[i], cols[j], round(c.iloc[i, j], 3)))   # cặp tương quan cao → nghi dư thừa
    return pairs

num_cols = ["Order Quantity", "Sales", "Discount", "Profit", "Unit Price", "Shipping Cost"]
df_full = consolidate(df_clean, DATA_PATH)                      # ✅ bảng đã tích hợp
print("Sau tích hợp:", df_full.shape, "| Status:", df_full["Status"].value_counts().to_dict())
print("Số dòng không đổi sau left-join:", len(df_full) == len(df_clean))
display(metadata(df_full).head(8))
print("Sales ≈ UnitPrice×Qty×(1-Discount)? tỷ số:\n", derived_check(df_clean).round(3).to_string())
print("Hiệp phương sai:"); display(covariance_matrix(df_clean, num_cols).round(2))
print("Pearson:");  display(pearson_matrix(df_clean, num_cols).round(2))
print("Spearman:"); display(spearman_matrix(df_clean, num_cols).round(2))
print("Cặp nghi dư thừa (|r| ≥ 0.6):", redundant_pairs(df_clean, num_cols, thr=0.6))

## PHẦN 9 – BIẾN ĐỔI DỮ LIỆU (Trang 17–19, 21)

In [ ]:
# ============================================================
# 9.1 NHÓM HÀM: Chuẩn hóa, chỉnh dạng, rời rạc hóa, làm trơn, cross-tab
# ------------------------------------------------------------
# 📖 TRANG 17: Khi nào biến đổi: (1) features KHÔNG cùng thang đo (2) phân phối lệch/biến thiên mạnh (3) đáp ứng đầu vào thuật toán.
#   CHUẨN HÓA: Min-Max (không chuẩn, ít outlier) | Decimal scaling | Z-score (gần chuẩn, ít outlier) | Robust (nhiều outlier, dùng tứ phân vị)
# 📖 TRANG 17-18: Chỉnh dạng: Log / Box-Cox / Yeo-Johnson (xem OutlierHandler), Feature construction (x, x²), Nhị phân hóa,
#   Làm tròn, Tổng hợp (Roll-up / Drill-down), Rời rạc hóa (equal-width, equal-frequency)
# 📖 TRANG 18-19: Làm trơn: SMA (trung bình trượt), Centered SMA (±7 ngày /15), WMA (trọng số)
# 📖 TRANG 19, 21: Chuyển Cross-tab ↔ Features-Values ; qcut chia theo tần số
# 🎯 DÙNG KHI: chuẩn bị dữ liệu cho thuật toán (hồi quy, kNN, PCA...) hoặc khi cần gom nhóm/làm mượt.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Biến đổi dữ liệu]:
#   - Khi nào biến đổi (KHÔNG được đổi bản chất bài toán): (1) cần xét tương quan mà features KHÔNG cùng thang đo ;
#     (2) có mức ưu tiên quá lớn: phân phối quá đặc biệt (lệch trái/phải nhiều), biến thiên quá mạnh ;
#     (3) đáp ứng yêu cầu đầu vào của thuật toán (hồi quy tuyến tính cần phân phối chuẩn để hiệu quả).
#   - CHUẨN HÓA (normalization/scaling): tạo ĐỊNH DẠNG CHUNG cho các features:
#       · MIN-MAX: biến đổi tuyến tính [L,U] → [Lnew,Unew]. ĐK: KHÔNG có phân phối chuẩn, ÍT outlier.
#       · DECIMAL scaling: f(x) = x / 10^k, k = số chữ số của phần tử lớn nhất.
#       · Z-SCORE (standardization): (x−μ)/σ, dùng để KHỬ ĐƠN VỊ. ĐK: GẦN phân phối chuẩn, ÍT outlier.
#       · ROBUST: dùng TỨ PHÂN VỊ (median, IQR). ĐK: KHÔNG chuẩn, NHIỀU outlier.
#   - CHỈNH DẠNG: biến đổi toán học (Log / Box-Cox / Yeo-Johnson – xem Phần 6) ; FEATURE CONSTRUCTION (tạo thuộc tính mới: gộp, tách,
#     (x ; x²) đưa lên không gian 2 chiều, INTERACTION) ; NHỊ PHÂN HÓA (trên ngưỡng → 1, dưới → 0 ; ứng dụng hồi quy LOGISTIC) ;
#     LÀM TRÒN ; TỔNG HỢP (aggregation, OLAP: roll-up, drill-down) ; RỜI RẠC HÓA ; LÀM TRƠN.
#   - RỜI RẠC HÓA bằng bins: UNSUPERVISED (không quan tâm target, chỉ input): equal-WIDTH (chia miền giá trị thành k đoạn BẰNG ĐỘ RỘNG;
#     dựa tri thức lĩnh vực) ; equal-FREQUENCY (k nhóm có TẦN SỐ cân bằng; qcut) ; SUPERVISED (dùng target).
#   - LÀM TRƠN chuỗi thời gian: SMA (trung bình k kỳ TRƯỚC; cửa sổ càng lớn càng "duỗi thẳng"; nhược điểm: mọi kỳ như nhau, không phân gần/xa) ;
#     CENTERED SMA (ngày t thay bằng tổng 7 ngày trước + ngày t + 7 ngày sau, chia 15) ; WMA (SMA có TRỌNG SỐ: gần lớn, xa nhỏ).
#   - CROSS-TAB ↔ FEATURES-VALUES: Cross-tab: giá trị trong bảng là SỐ (tần suất), tên cột là giá trị của biến ;
#     Features-Values: giá trị là CHỮ, tên cột là feature. Cross-tab → Features-Values (HAY DÙNG): phải đổi tên cột, thay mã 0,1,2… bằng nhãn
#     (Nam, Nữ, Không muốn trả lời…).
# 🔎 CHỌN KHI NÀO:
#   A) CHUẨN HÓA – chọn theo HÌNH DẠNG phân phối + có outlier không:
#     - Gần chuẩn + ÍT outlier, muốn khử đơn vị                    → Z-SCORE. DẤU HIỆU không hợp: có |z| rất lớn
#         (đã chạy) Unit Price: z max = 24.6 → có outlier mạnh → nên Robust.
#     - Không chuẩn + ÍT outlier, cần miền cố định [0,1]            → MIN-MAX. DẤU HIỆU không hợp: sau min-max phần lớn giá trị dồn sát 0
#         (đã chạy) Unit Price: mean sau Min-Max = 0.012 (cả cột bị nén sát 0 vì 1 vài giá trị 6783) → outlier làm hỏng Min-Max.
#     - Không chuẩn + NHIỀU outlier                                  → ROBUST (median/IQR).
#     - Chỉ cần đưa số lớn về |x| < 1 nhanh, thô                    → DECIMAL.
#     - Khi thuật toán dựa khoảng cách/PCA/kNN và các cột khác thang đo (Unit Price hàng trăm, Discount 0–1) → PHẢI chuẩn hóa trước.
#   B) CHỈNH DẠNG:
#     - Lệch phải mạnh, giá trị dương → LOG ; có âm/0 → YEO-JOHNSON (xem Phần 6).
#     - Cần quan hệ cong (phi tuyến) hoặc tác động chéo → FEATURE CONSTRUCTION (x, x²), INTERACTION (Qty×Price).
#     - 1 cột chứa nhiều thông tin (ngày) → TÁCH (năm, tháng) ; 2 cột bổ sung nhau → GỘP.
#     - Biến liên tục nhưng bài toán là có/không (lãi/lỗ), dùng hồi quy logistic → NHỊ PHÂN HÓA theo ngưỡng nghiệp vụ.
#     - Cần cái nhìn tổng quát hơn → ROLL-UP (đơn hàng → năm) ; cần xem chi tiết hơn → DRILL-DOWN (năm → năm×loại hàng).
#   C) RỜI RẠC HÓA:
#     - Phân phối gần ĐỀU hoặc khoảng có ý nghĩa nghiệp vụ (nhóm tuổi 1–10, 11–20…) → EQUAL-WIDTH.
#         DẤU HIỆU không hợp: phân phối LỆCH mạnh → hầu hết dữ liệu rơi vào 1 bin. (đã chạy) Sales: 8250/8290 dòng nằm ở bin đầu.
#     - Phân phối LỆCH, cần các nhóm CÂN BẰNG số lượng                                → EQUAL-FREQUENCY (đã chạy: mỗi nhóm ≈ 2072 dòng).
#     - Có biến target và muốn điểm cắt "biết phân biệt nhãn"                           → SUPERVISED binning (cây quyết định).
#   D) LÀM TRƠN chuỗi thời gian nhiều nhiễu:
#     - Chỉ muốn nhìn XU HƯỚNG chung          → SMA (cửa sổ lớn = mượt hơn nhưng trễ hơn).
#     - Phân tích LỊCH SỬ (có cả kỳ sau)       → CENTERED SMA (không dùng để dự báo vì dùng dữ liệu tương lai).
#     - Gần kỳ hiện tại phải quan trọng hơn    → WMA.
#   E) CROSS-TAB vs FEATURES-VALUES: cần bảng đếm/so sánh chéo → Cross-tab ; cần đưa vào thuật toán/vẽ biểu đồ theo từng feature → Features-Values.
# ============================================================
from sklearn.tree import DecisionTreeClassifier

# ======== 1. CHUẨN HÓA ========
# Min-Max: f(x) = (x - L)/(U - L) × (Unew - Lnew) + Lnew
def minmax(d, col, new_min=0, new_max=1):
    x = d[col]
    L, U = x.min(), x.max()
    return (x - L) / (U - L) * (new_max - new_min) + new_min

# Decimal scaling: f(x) = x / 10^k , k = số chữ số của phần tử lớn nhất
def decimal_scaling(d, col):
    x = d[col]
    k = len(str(int(x.abs().max())))                   # số chữ số của giá trị lớn nhất
    return x / (10 ** k)

# Z-score: f(x) = (x - μ)/σ  (khử đơn vị) – tự gán ddof để chắc chắn
def zscore(d, col, ddof=0):
    x = d[col]
    return (x - x.mean()) / x.std(ddof=ddof)

# Robust: f(x) = (x - median)/(Q3 - Q1)  (dùng tứ phân vị, ít bị outlier ảnh hưởng)
def robust(d, col):
    x = d[col]
    return (x - x.median()) / (x.quantile(0.75) - x.quantile(0.25))

# ======== 2. CHỈNH DẠNG ========
# Feature construction: tạo thuộc tính mới (x, x²) / tương tác / tách ngày
def construct_features(d, sales="Sales", qty="Order Quantity", price="Unit Price", date_col="Order Date", cat1="Region", cat2="Customer Segment"):
    d = d.copy()
    d[sales + "_sq"] = d[sales] ** 2                                 # (x ; x²) đưa vào không gian 2 chiều
    d["Revenue_per_unit"] = d[sales] / d[qty]                        # tỷ lệ giữa 2 thuộc tính
    d["Qty_x_Price"] = d[qty] * d[price]                             # interaction: liên kết 2 thuộc tính
    d["Year"] = d[date_col].dt.year                                  # TÁCH 1 cột ngày → năm, tháng
    d["Month"] = d[date_col].dt.month
    d["Region_Segment"] = d[cat1] + "_" + d[cat2]                    # GỘP 2 cột thành 1
    return d

# Nhị phân hóa: trên ngưỡng → 1, dưới ngưỡng → 0 (ứng dụng: hồi quy logistic)
def binarize(d, col, threshold):
    return (d[col] > threshold).astype(int)

# Làm tròn số
def rounding(d, col, decimals=0):
    return d[col].round(decimals)

# Tổng hợp dữ liệu (OLAP): Roll-up (gộp lên mức trừu tượng cao) / Drill-down (khoan xuống chi tiết)
def roll_up(d, date_col="Order Date", value_col="Sales"):
    year = d[date_col].dt.year
    return d.groupby(year)[value_col].sum()                                          # mức: Năm
def drill_down(d, date_col="Order Date", cat_col="Product Category", value_col="Sales"):
    year = d[date_col].dt.year
    return d.groupby([year, d[cat_col]])[value_col].sum().unstack()                  # mức: Năm → Loại hàng

# ======== 3. RỜI RẠC HÓA ========
def equal_width(d, col, k=4):
    return pd.cut(d[col], bins=k)                    # chia miền giá trị thành k đoạn ĐỘ RỘNG bằng nhau

def equal_frequency(d, col, k=4):
    return pd.qcut(d[col], q=k, duplicates="drop")   # k nhóm có SỐ LƯỢNG bằng nhau (qcut)

def supervised_binning(d, col, target_binary, max_bins=4):
    # Supervised binning: dùng NHÃN (target) để tìm điểm cắt – ở đây dùng cây quyết định 1 biến
    x = d[[col]]
    tree = DecisionTreeClassifier(max_leaf_nodes=max_bins, random_state=0).fit(x, target_binary)
    cuts = sorted(t for t in tree.tree_.threshold if t != -2)   # -2 = nút lá (không phải điểm cắt)
    return cuts

# ======== 4. LÀM TRƠN chuỗi thời gian ========
def sma(s, window=7):
    return s.rolling(window=window).mean()                       # trung bình của k kỳ TRƯỚC (gồm hiện tại)

def centered_sma(s, window=15):
    return s.rolling(window=window, center=True).mean()          # ±7 kỳ trước/sau, chia 15

def wma(s, window=7):
    w = np.arange(1, window + 1)                                  # trọng số 1..k: GẦN thì lớn, XA thì nhỏ
    return s.rolling(window).apply(lambda x: np.dot(x, w) / w.sum(), raw=True)

# ======== 5. CROSS-TAB ↔ FEATURES-VALUES ========
def to_crosstab(d, row, col):
    return pd.crosstab(d[row], d[col])                            # Features-Values → Cross-tab (giá trị = tần suất)

def to_features_values(crosstab):
    # Cross-tab → Features-Values (dạng dài): tên cột trở thành GIÁ TRỊ của 1 cột, đổi tên cột cho rõ nghĩa
    return crosstab.reset_index().melt(id_vars=crosstab.index.name, var_name="Nhóm", value_name="Số lượng")

# --- Sử dụng ---
col = "Unit Price"
cmp = pd.DataFrame({"Gốc": df_clean[col], "MinMax": minmax(df_clean, col), "MinMax[-1,1]": minmax(df_clean, col, -1, 1),
                    "Decimal": decimal_scaling(df_clean, col), "Z-score": zscore(df_clean, col), "Robust": robust(df_clean, col)})
print("=== CHUẨN HÓA", col, "==="); display(cmp.describe().loc[["min", "max", "mean", "std"]].round(3))

feat = construct_features(df_clean)
print("Cột mới tạo:", [c for c in feat.columns if c not in df_clean.columns])
print("Nhị phân hóa Profit>0 (tỷ lệ lãi):", round(binarize(df_clean, "Profit", 0).mean(), 3))
print("Làm tròn Sales 1 số:", rounding(df_clean, "Sales", 1).head(3).tolist())
print("\nRoll-up (theo năm):\n", roll_up(df_clean).round(0))
print("Drill-down (năm × loại hàng):"); display(drill_down(df_clean).round(0))
print("Equal-width Sales (4 đoạn):\n", equal_width(df_clean, "Sales", 4).value_counts().sort_index())
print("Equal-frequency Sales (4 nhóm):\n", equal_frequency(df_clean, "Sales", 4).value_counts().sort_index())
print("Supervised binning Unit Price theo (Profit>0) – điểm cắt:", np.round(supervised_binning(df_clean, "Unit Price", (df_clean["Profit"] > 0)), 2))

# Làm trơn: doanh số theo NGÀY → SMA/Centered/WMA
daily = df_clean.groupby("Order Date")["Sales"].sum().asfreq("D").fillna(0)
sm = pd.DataFrame({"Gốc": daily, "SMA7": sma(daily, 7), "Centered15": centered_sma(daily, 15), "WMA7": wma(daily, 7)}).iloc[:200]
sm.plot(figsize=(10, 3.5), title="Làm trơn doanh số theo ngày (200 ngày đầu)"); plt.show()

ct = to_crosstab(df_clean, "Region", "Customer Segment")
print("Cross-tab:"); display(ct)
print("→ Features-Values:"); display(to_features_values(ct).head(6))

## PHẦN 10 – MÃ HÓA DỮ LIỆU PHÂN LOẠI (Trang 19)

In [ ]:
# ============================================================
# 10.1 NHÓM HÀM: Label / Ordinal / Frequency / One-hot / Dummy / Target / Hashing
# ------------------------------------------------------------
# 📖 TRANG 19: Dữ liệu ĐỊNH DANH (nominal – không thứ tự) & ĐỊNH TÍNH (ordinal – có thứ tự)
#   - Label encoding: gán số cho mỗi nhãn (nominal bị ảnh hưởng nhẹ; ordinal phải GIỮ THỨ TỰ)
#   - Frequency encoding: đếm số lần xuất hiện (chỉ làm khi tần suất các nhãn KHÔNG trùng nhau)
#   - One-hot: k nhãn → k cột 0/1
#   - Dummy: k nhãn → (k-1) cột 0/1
#   - Target / Mean / Leave-one-out encoding: dựa trên giá trị trung bình của TARGET (chú ý rò rỉ nhãn)
#   - Feature Hashing: hàm băm → độ dài cố định; dùng khi feature có RẤT NHIỀU giá trị
# 🎯 DÙNG KHI: thuật toán chỉ nhận số (hồi quy, kNN, PCA...) mà dữ liệu là chữ.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Mã hóa dữ liệu phân loại]:
#   - ĐỊNH DANH (nominal): KHÔNG thứ tự. ĐỊNH TÍNH (ordinal): CÓ thứ tự (lớn/nhỏ, trước/sau).
#   - LABEL ENCODING (gán số): nominal → giá trị biến đổi khác nhau theo giá trị gốc (gán số nên ít nhiều bị ảnh hưởng → cân nhắc) ;
#       ordinal → phải ĐẢM BẢO THỨ TỰ (khoảng cách giữa 2 giá trị cần cân nhắc: số năm, số bậc…). Word ghi kèm điều kiện thuật toán HỘI TỤ
#       (kết quả độc lập điểm khởi đầu).
#   - Đếm số lần xuất hiện (FREQUENCY): chỉ làm khi tần suất của các giá trị KHÔNG TRÙNG LẮP.
#   - ONE-HOT: k nhãn → k features mới (1 = thỏa, 0 = không thỏa).
#   - DUMMY: giống one-hot nhưng chỉ tạo (k−1) cột.
#   - TARGET / MEAN / BIN-COUNTING / LIKELIHOOD / LEAVE-ONE-OUT encoding: giá trị mã hóa dựa trên TARGET của nhãn đó
#       (tạo ràng buộc giữa feature và target → coi chừng bị trùng lặp/rò rỉ nhãn).
#   - FEATURE HASHING: hàm băm biến chuỗi dài bất kỳ → chuỗi độ dài XÁC ĐỊNH ; dùng khi feature có NHIỀU giá trị.
# 🔎 CHỌN KHI NÀO – ĐẾM SỐ NHÃN VÀ XEM CÓ THỨ TỰ KHÔNG:
#   - ORDINAL (Order Priority: Low<Medium<High<Critical)           → ORDINAL/LABEL encoding GIỮ ĐÚNG THỨ TỰ (khai báo list order).
#   - NOMINAL, ÍT nhãn (Region 4, Segment 4), thuật toán tuyến tính/khoảng cách/hồi quy
#                                                                   → ONE-HOT (k cột) ; nếu mô hình CÓ HẰNG SỐ/cần tránh đa cộng tuyến hoàn toàn → DUMMY (k−1 cột).
#   - NOMINAL mà gán số 0,1,2,3 (label) → máy hiểu nhầm West(3) > Central(0)  → chỉ dùng khi thuật toán không đọc thứ tự/khoảng cách (vd cây quyết định) [+bổ sung].
#   - Có mối liên hệ giữa nhãn và 1 biến SỐ mục tiêu, NHIỀU nhãn (State ~50) → TARGET/MEAN encoding ; coi chừng rò rỉ → LEAVE-ONE-OUT.
#   - Số nhãn RẤT LỚN (Customer Name, Product Name: hàng trăm–nghìn) → FEATURE HASHING (cố định số cột; chấp nhận va chạm).
#   - FREQUENCY chỉ dùng khi KIỂM TRA được tần suất các nhãn khác nhau: số giá trị tần suất duy nhất == số nhãn.
#       (đã chạy) 4 vùng có 4 tần suất khác nhau (2670/1923/1816/1881) → dùng được ; nếu 2 vùng cùng tần suất → bị trộn → KHÔNG dùng.
# ============================================================
# Label encoding cho NOMINAL: mỗi nhãn → 1 số nguyên (theo thứ tự chữ cái)
def label_encode(d, col):
    labels = sorted(d[col].dropna().unique())
    mapping = {lab: i for i, lab in enumerate(labels)}
    return d[col].map(mapping), mapping

# Label encoding cho ORDINAL: phải đảm bảo THỨ TỰ do người dùng khai báo
def ordinal_encode(d, col, order):
    mapping = {lab: i for i, lab in enumerate(order)}      # vị trí trong danh sách = mã
    return d[col].map(mapping), mapping

# Frequency encoding: nhãn → số lần xuất hiện
def frequency_encode(d, col):
    return d[col].map(d[col].value_counts())

# One-hot: k cột
def onehot(d, col):
    return pd.get_dummies(d[col], prefix=col).astype(int)

# Dummy: k-1 cột (bỏ cột đầu để tránh cộng tuyến hoàn toàn)
def dummy(d, col):
    return pd.get_dummies(d[col], prefix=col, drop_first=True).astype(int)

# Target / Mean encoding: nhãn → trung bình target của nhãn đó
def target_encode(d, col, target):
    return d[col].map(d.groupby(col)[target].mean())

# Leave-one-out: trung bình target của nhãn nhưng LOẠI TRỪ chính dòng đang xét (giảm rò rỉ nhãn)
def leave_one_out(d, col, target):
    g = d.groupby(col)[target]
    s, n = g.transform("sum"), g.transform("count")
    return (s - d[target]) / (n - 1)

# Feature Hashing: băm chuỗi → số trong [0, n_buckets)
def hashing(d, col, n_buckets=8):
    def h(v):
        return int(hashlib.md5(str(v).encode()).hexdigest(), 16) % n_buckets   # băm md5 rồi lấy dư
    return d[col].apply(h)

lab, mp = label_encode(df_clean, "Region");          print("Label (nominal) Region:", mp)
pri_order = ["Not Specified", "Low", "Medium", "High", "Critical"]     # quy ước: Not Specified = chưa xác định (thấp nhất)
pri, mp2 = ordinal_encode(df_clean, "Order Priority", pri_order);  print("Ordinal Order Priority:", mp2)
print("Frequency Region:", frequency_encode(df_clean, "Region").unique()[:4], "(kiểm tra: tần suất 4 vùng phải khác nhau)")
print("One-hot Customer Segment (4 cột):", onehot(df_clean, "Customer Segment").columns.tolist())
print("Dummy   Customer Segment (3 cột):", dummy(df_clean, "Customer Segment").columns.tolist())
print("Target encoding Region→mean Profit:", target_encode(df_clean, "Region", "Profit").unique().round(2)[:4])
print("Leave-one-out (5 dòng đầu):", leave_one_out(df_clean, "Region", "Profit").head().round(2).tolist())
print("Hashing State (8 bucket):", hashing(df_clean, "State", 8).head().tolist())

## PHẦN 11 – THU GỌN DỮ LIỆU (Trang 10, 20–21)

In [ ]:
# ============================================================
# 11.1 NHÓM HÀM: Giảm lượng + Giảm chiều
# ------------------------------------------------------------
# 📖 TRANG 20: Data Reduction – thu gọn mà KHÔNG đổi kết quả đáng kể.
#   (1) GIẢM KHỐI LƯỢNG (số dòng): có tham số (lưu hàm hồi quy thay vì dữ liệu) |
#       phi tham số: subtotal, histogram, clustering (lưu trọng tâm), sampling
#   (2) Nén dữ liệu (optional, phục vụ lưu trữ)
#   (3) GIẢM CHIỀU (số cột): Filter (ngưỡng phương sai, tương quan, kiểm định) |
#       Wrapper (forward selection / loại dần) | Embedded (cây quyết định, Lasso, Ridge) | RFE
# 📖 TRANG 10, 21: PCA (phổ biến, dễ hiểu nhất), SVD, chéo hóa ma trận.
# 🎯 DÙNG KHI: quá nhiều dòng/cột, cột dư thừa, cần giảm chiều trước khi chạy thuật toán.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Thu gọn dữ liệu]: thu gọn mà KHÔNG làm thay đổi kết quả đáng kể.
#   (1) GIẢM KHỐI LƯỢNG (số DÒNG): đổi dạng biểu diễn.
#       · Có THAM SỐ: thay đổi hàm sinh dữ liệu (lưu hàm hồi quy, KHÔNG lưu training set). Hồi quy → dữ liệu numerical ;
#         mô hình LOG-LINEAR → bảng tương quan (contingency table) giữa các features categorical, giá trị KHÔNG ÂM, tìm tham số bằng MLE.
#       · PHI tham số: subtotal/tóm tắt ; rời rạc hóa (histogram) ; CLUSTERING (lưu trọng tâm) ; SAMPLING.
#   (2) NÉN dữ liệu (optional): phục vụ lưu trữ/truyền thông, KHÔNG đúng ý nghĩa giảm cho phân tích.
#   (3) GIẢM CHIỀU (số CỘT):
#       · CHỌN LỌC thuộc tính (loại feature kém, giữ feature mang ý nghĩa; ví dụ bắt buộc giữ số lượng, thành tiền thì bỏ đơn giá):
#           - FILTER: ngưỡng PHƯƠNG SAI, phân tích TƯƠNG QUAN, kiểm định giả thuyết. Feature phương sai quá BÉ → bỏ (vô ích).
#             Target: phương sai quá bé → bỏ ; quá LỚN → cẩn thận vì quy luật bị lệch (overfitting).
#           - WRAPPER: dựa hiệu quả của TẬP features, 2 hướng: thêm dần (FORWARD selection) hoặc loại dần (backward).
#           - EMBEDDED: chọn ngay trong lúc xây mô hình (cây quyết định, LASSO, RIDGE…).
#           - RFE: chọn theo MỨC ĐỘ QUAN TRỌNG; đòi hỏi mô hình có cách đánh giá mức độ quan trọng; mỗi vòng bỏ feature kém nhất.
#       · RÚT TRÍCH thuộc tính: PCA (phổ biến, dễ hiểu nhất), SVD, dựa trên đại số ma trận (CHÉO HÓA ma trận).
# 🔎 CHỌN KHI NÀO:
#   A) GIẢM DÒNG:
#     - Dữ liệu SỐ có quy luật rõ (Sales ~ Unit Price) và chỉ cần quy luật → HỒI QUY (lưu hệ số).
#     - Cần bảng tổng hợp theo nhóm → SUBTOTAL ; chỉ cần phân phối → HISTOGRAM ; dữ liệu tụ thành cụm → CLUSTERING (lưu trọng tâm) ;
#       cần mẫu nhỏ đại diện → SAMPLING (xem Phần 4).
#   B) GIẢM CỘT – CÁC DẤU HIỆU:
#     - Biến thiên (phương sai) của 1 cột NHỎ HƠN ngưỡng sau khi đưa các cột CÙNG THANG ĐO (min-max)  → FILTER bỏ cột gần như hằng số.
#         (đã chạy) tất cả phương sai min-max ≥ 0.0016 > ngưỡng 0.001 → chưa cột nào bị loại.
#     - 2 INPUT tương quan cao (|r| ≥ ~0.8 [+bổ sung])              → bỏ 1 cột (giữ cột có ý nghĩa nghiệp vụ hơn). (đã chạy) không cặp nào ≥ 0.8.
#     - Có TARGET, muốn chọn tập feature cho mô hình, ít feature      → WRAPPER (forward: xem R² tăng ra sao ; dừng khi R² không tăng đáng kể).
#         (đã chạy) Profit: Sales (R² 0.43) → +Shipping Cost (0.56) → +Unit Price (0.67).
#     - Mô hình đã có sẵn cơ chế (LASSO/cây)                          → EMBEDDED: hệ số LASSO = 0 ⇒ feature bị loại. (đã chạy) Order Quantity = 0 → loại.
#     - Cần xếp hạng quan trọng và loại dần                            → RFE.
#     - Nhiều cột TƯƠNG QUAN/đa cộng tuyến, chấp nhận MẤT TÊN/NGHĨA cột gốc → PCA/SVD (thành phần mới là TỔ HỢP của các cột, KHÔNG còn nghĩa nghiệp vụ).
#     - CẦN GIỮ NGHĨA cột gốc để giải thích kết quả                   → CHỌN LỌC thuộc tính (Filter/Wrapper/Embedded), KHÔNG dùng PCA.
#   C) PCA/SVD – ĐỌC KẾT QUẢ (tỷ lệ phương sai giải thích):
#     - Giữ số thành phần sao cho phương sai TÍCH LŨY đạt ~80–90% [+bổ sung ngưỡng].
#         (đã chạy) PC1 37.4%, PC2 21.5% → 2 thành phần chỉ ≈ 58.9% ; cần 3 → 78.8%, 4 → 94.2%.
#     - Phương sai chia GẦN ĐỀU giữa các thành phần (PC1 chỉ 37%) → dữ liệu ÍT dư thừa, giảm chiều KHÔNG hiệu quả (mất nhiều thông tin).
#     - PHẢI CHUẨN HÓA z-score trước PCA/SVD khi các cột KHÁC thang đo; nếu không cột số lớn (Sales) sẽ áp đảo.
# ============================================================
from sklearn.cluster import KMeans
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA

# ======== GIẢM KHỐI LƯỢNG ========
def regression_model(d, x, y):
    # Có tham số: chỉ lưu (hệ số, hằng số) của hàm hồi quy thay cho cả bảng dữ liệu
    t = d[[x, y]].dropna()
    m = LinearRegression().fit(t[[x]], t[y])
    return m.coef_[0], m.intercept_

def subtotal(d, group, col):
    return d.groupby(group)[col].agg(["count", "sum", "mean"])   # tóm tắt/subtotal

def histogram_reduce(d, col, bins=10):
    counts, edges = np.histogram(d[col].dropna(), bins=bins)     # chỉ lưu (tần số, ranh giới)
    return counts, edges

def cluster_reduce(d, cols, k=5):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(d[cols].dropna())
    return km.cluster_centers_                                   # chỉ lưu k trọng tâm thay cho cả dữ liệu

def sampling(d, n):
    return d.sample(n=n, random_state=1)                         # lấy mẫu

# ======== GIẢM CHIỀU – FILTER ========
def variance_threshold(d, cols, thr=0.01):
    # Phương sai quá bé → cột gần như hằng số → bỏ. Dùng Min-Max trước để các cột CÙNG THANG ĐO
    scaled = (d[cols] - d[cols].min()) / (d[cols].max() - d[cols].min())
    var = scaled.var()
    return var.round(4), var[var < thr].index.tolist()

def correlation_filter(d, cols, thr=0.8):
    c = d[cols].corr().abs()
    drop = set()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if c.iloc[i, j] >= thr:
                drop.add(cols[j])                                # 2 cột tương quan cao → bỏ 1 cột
    return list(drop)

# ======== GIẢM CHIỀU – WRAPPER (forward selection) ========
def forward_selection(d, cols, target, k=3, verbose=True):
    chosen = []
    for _ in range(k):                                           # mỗi vòng thêm 1 feature tốt nhất
        best, best_score = None, -1e9
        for c in cols:
            if c in chosen: continue
            score = cross_val_score(LinearRegression(), d[chosen + [c]], d[target], cv=3, scoring="r2").mean()
            if score > best_score:
                best, best_score = c, score
        chosen.append(best)
        if verbose:
            print(f"  Thêm '{best}' → R² = {best_score:.3f}")
    return chosen

# ======== RFE: loại dần feature kém quan trọng nhất ========
def rfe(d, cols, target, k=3):
    sel = RFE(LinearRegression(), n_features_to_select=k).fit(d[cols], d[target])
    return [c for c, keep in zip(cols, sel.support_) if keep]

# ======== EMBEDDED: Lasso (hệ số về 0 = bị loại) ========
def lasso_select(d, cols, target, alpha=1.0):
    X = (d[cols] - d[cols].mean()) / d[cols].std()               # chuẩn hóa trước khi Lasso
    m = Lasso(alpha=alpha).fit(X, d[target])
    return pd.Series(m.coef_, index=cols).round(3)

# ======== PCA viết tay: chéo hóa ma trận hiệp phương sai ========
def pca_manual(d, cols, n_comp=2):
    X = d[cols].values.astype(float)
    X = (X - X.mean(axis=0)) / X.std(axis=0)                       # B1: chuẩn hóa (z-score)
    C = np.cov(X.T)                                                # B2: ma trận hiệp phương sai
    vals, vecs = np.linalg.eigh(C)                                 # B3: trị riêng & vector riêng (chéo hóa)
    order = np.argsort(vals)[::-1]                                 # B4: sắp giảm dần theo trị riêng
    vals, vecs = vals[order], vecs[:, order]
    Z = X @ vecs[:, :n_comp]                                       # B5: chiếu dữ liệu lên n_comp trục chính
    return Z, vals / vals.sum()                                    # tỷ lệ phương sai giải thích

# ======== SVD ========
def svd(d, cols, n_comp=2):
    X = d[cols].values.astype(float)
    X = (X - X.mean(axis=0)) / X.std(axis=0)                       # chuẩn hóa z-score (nếu không, cột có số lớn như Sales sẽ áp đảo)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)               # X = U · Σ · Vᵗ
    return U[:, :n_comp] * S[:n_comp], (S ** 2) / (S ** 2).sum()

feat_cols = ["Order Quantity", "Sales", "Discount", "Unit Price", "Shipping Cost"]
print("Hồi quy Sales~Unit Price (hệ số, hằng số):", np.round(regression_model(df_clean, "Unit Price", "Sales"), 3))
print("Histogram Sales (10 bins):", histogram_reduce(df_clean, "Sales")[0])
print("Trọng tâm 5 cụm (Sales, Profit):\n", cluster_reduce(df_clean, ["Sales", "Profit"]).round(1))
var, low = variance_threshold(df_clean, feat_cols + ["Profit"], thr=0.001); print("Phương sai (min-max):\n", var.to_string(), "\n→ bỏ:", low)
print("Cột tương quan cao nên bỏ (≥0.8):", correlation_filter(df_clean, feat_cols, 0.8))
print("Forward selection (Profit):"); fs = forward_selection(df_clean, feat_cols, "Profit", 3); print("→", fs)
print("RFE chọn:", rfe(df_clean, feat_cols, "Profit", 3))
print("Lasso hệ số:\n", lasso_select(df_clean, feat_cols, "Profit").to_string())
Zm, ratio_m = pca_manual(df_clean, feat_cols, 2)
pca_lib = PCA(n_components=2).fit((df_clean[feat_cols] - df_clean[feat_cols].mean()) / df_clean[feat_cols].std(ddof=0))
print("PCA tay – tỷ lệ phương sai:", ratio_m.round(3), "| sklearn:", pca_lib.explained_variance_ratio_.round(3))
Zs, ratio_s = svd(df_clean, feat_cols, 2); print("SVD – tỷ lệ phương sai (2 thành phần đầu):", ratio_s[:2].round(3))
plt.scatter(Zm[:, 0], Zm[:, 1], s=3); plt.title("PCA: 5 chiều → 2 chiều"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

## PHẦN 12 – THỐNG KÊ MÔ TẢ (Trang 21–23)

In [ ]:
# ============================================================
# 12.1 NHÓM HÀM: Xu thế trung tâm, độ phân tán, hình dáng, tương quan
# ------------------------------------------------------------
# 📖 TRANG 21-22: XU THẾ TRUNG TÂM
#   - Trung vị (vị trí) | Trung bình cộng (biến thiên ÍT) | Trung bình nhân (biến thiên LỚN) |
#     Trung bình điều hòa (TỶ SỐ khác đơn vị: vận tốc) | Yếu vị/mode (KHÔNG chọn khi kiểm tra)
# 📖 TRANG 22-23: PHÂN TÁN: khoảng biến thiên (range), bách phân vị, tứ phân vị (IQR),
#   phương sai & độ lệch chuẩn (n-1 cho mẫu; ddof=0 quần thể, ddof=1 mẫu – NÊN TỰ GÁN)
# 📖 TRANG 23: HÌNH DÁNG: skewness Pearson₂ = 3(mean - median)/s ; kurtosis (chuẩn = 3);
#   Hiệp phương sai, Pearson (khử đơn vị), Spearman (thứ hạng); đường chéo cov = phương sai; đường chéo corr = 1.
# 🎯 DÙNG KHI: bước "Thống kê mô tả" của đánh giá chất lượng và phân tích mô tả (trang 21).
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Thống kê mô tả]:
#   - Phân tích mô tả bằng: (1) giá trị số / phương pháp thống kê ; (2) bảng (ngôn ngữ truy vấn CSDL) ; (3) biểu đồ, đồ họa.
#   - XU THẾ TRUNG TÂM: dùng 1 đại lượng đại diện cho tổng thể.
#       · TRUNG VỊ: trung tâm về mặt VỊ TRÍ ; TRUNG BÌNH: trung tâm về mặt GIÁ TRỊ.
#       · Trung bình CỘNG: khi dữ liệu BIẾN THIÊN ÍT. Trung bình NHÂN: hợp lý khi dữ liệu BIẾN THIÊN LỚN.
#         → Chọn cái nào XẤP XỈ TRUNG VỊ hơn. Có 2 cách tính: (1) từ các mẫu riêng lẻ ; (2) GỘP các mẫu (gộp tử chung, mẫu chung rồi mới chia).
#       · Trung bình ĐIỀU HÒA: cho TỶ SỐ giữa 2 đại lượng KHÁC ĐƠN VỊ (vận tốc, gia tốc); vì là TỶ LỆ nên phải tính trên tỷ lệ.
#       · YẾU VỊ (mode): trung tâm về mức độ TẬP TRUNG ; liên tục thì thường không có mode, rời rạc có thể nhiều mode → Word dặn KHÔNG CHỌN khi kiểm tra.
#   - PHÂN TÁN: KHOẢNG BIẾN THIÊN (range) = max − min, ước lượng ban đầu; KHÔNG phù hợp khi chỉ quan tâm 2 đầu mút (bỏ qua phần giữa) ;
#       BÁCH PHÂN VỊ P(%) ; KHOẢNG TRẢI GIỮA = tứ phân vị (IQR) ; PHƯƠNG SAI / ĐỘ LỆCH CHUẨN (khử dấu bằng lũy thừa bậc CHẴN vì trị tuyệt đối
#       không có đạo hàm bậc 2/không có cực trị).
#     · Mẫu chia (n − 1) vì BẬC TỰ DO (phần tử cuối bị ràng buộc bởi các phần tử trước) → ước lượng KHÔNG CHỆCH.
#     · ddof (delta degrees of freedom): ddof=0 cho QUẦN THỂ, ddof=1 cho MẪU → NÊN TỰ GÁN ĐỂ CHẮC CHẮN và GHI RA RÕ.
#       (Lưu ý thực tế: pandas .var()/.std() mặc định ddof=1 ; numpy np.var()/np.std() mặc định ddof=0 → dễ ra 2 kết quả khác nhau.)
#   - HÌNH DÁNG: SKEWNESS Pearson₂ = 3(mean − median)/s : mean > median → dương → LỆCH PHẢI (đuôi bên phải) ; mean < median → âm → LỆCH TRÁI.
#       KURTOSIS (độ nhọn): so với PHÂN PHỐI CHUẨN (kurtosis = 3) → nhọn hơn hay BẸT hơn (đừng nói chung chung).
#   - TƯƠNG QUAN: HIỆP PHƯƠNG SAI = xu thế cùng biến thiên, đổi đơn vị thì đổi giá trị nên không so mạnh–yếu được ;
#       PEARSON: khử đơn vị, biết CƯỜNG ĐỘ ; SPEARMAN: dùng THỨ HẠNG (dữ liệu không thô/numerical), đồng hạng → rải đều.
#       Đường chéo ma trận HIỆP PHƯƠNG SAI = phương sai của chính nó ; đường chéo ma trận TƯƠNG QUAN = 1.
#   - Chebyshev: tỷ lệ giữ lại ≥ 1 − 1/z² ; BAR CHART có khoảng trắng, HISTOGRAM không.
# 🔎 CHỌN KHI NÀO:
#   A) CHỌN ĐẠI LƯỢNG ĐẠI DIỆN – nhìn vào 3 con số mean / median / mean nhân:
#     - mean ≈ median, dữ liệu ít biến thiên, đối xứng            → TRUNG BÌNH CỘNG.
#     - mean ≫ median (lệch phải, có outlier), biến thiên LỚN     → TRUNG VỊ (bền với outlier) hoặc TRUNG BÌNH NHÂN (nếu nó gần trung vị hơn).
#         (đã chạy) Sales: cộng 1778 | nhân 491 | trung vị 448 → trung bình NHÂN xấp xỉ trung vị hơn nhiều → chọn nhân/trung vị, KHÔNG dùng trung bình cộng.
#     - Đại lượng là TỶ SỐ giữa 2 đơn vị khác nhau (km/h, giá/đơn vị) → TRUNG BÌNH ĐIỀU HÒA (đã chạy: 60 và 40 km/h → 48, không phải 50).
#         Luôn có: điều hòa ≤ nhân ≤ cộng (với số dương).
#     - Dữ liệu đã GỘP theo nhóm/tỷ lệ → dùng cách 2 (gộp tử – gộp mẫu), đừng lấy trung bình của các tỷ lệ.
#     - Biến PHÂN LOẠI/rời rạc ít giá trị → MODE (nhưng Word dặn không dùng làm đáp án chính khi kiểm tra).
#   B) CHỌN ĐỘ PHÂN TÁN:
#     - Cần ước lượng NHANH, không có outlier → RANGE (bị outlier chi phối hoàn toàn: đã chạy Sales range = 89058).
#     - Dữ liệu LỆCH/nhiều outlier → IQR (Q3−Q1) / percentile.
#     - Dữ liệu ~ đối xứng/chuẩn → PHƯƠNG SAI/ĐỘ LỆCH CHUẨN ; dữ liệu là MẪU → ddof=1 ; là TOÀN BỘ quần thể → ddof=0.
#   C) ĐỌC HÌNH DÁNG:
#     - Skew Pearson₂ > 0 (mean > median) → lệch PHẢI → cân nhắc LOG/Box-Cox (Phần 6). (đã chạy) Sales: Pearson₂ = 1.11 > 0.
#     - Skew < 0 → lệch TRÁI.      |Skew| ≈ 0 → xấp xỉ đối xứng.
#     - Kurtosis > 3 → NHỌN hơn chuẩn, ĐUÔI DÀY → nhiều outlier. (đã chạy) Sales: 63.8 ≫ 3 → đuôi cực dày, có ngoại lai rất lớn.
#       Kurtosis < 3 → BẸT hơn chuẩn.  (pandas .kurt() trả "excess kurtosis" = kurtosis − 3, đừng so với 3.) [+bổ sung]
#   D) CHỌN HỆ SỐ TƯƠNG QUAN:
#     - Chỉ xem 2 biến cùng/ngược chiều → HIỆP PHƯƠNG SAI (dấu) ; muốn nói MẠNH hay YẾU → PEARSON (|r|).
#     - Dữ liệu chỉ có THỨ HẠNG, hoặc lệch/outlier/quan hệ đơn điệu không tuyến tính → SPEARMAN.
#     - KIỂM TRA ma trận: đường chéo cov = phương sai từng biến ; đường chéo corr = 1 (sai → nhầm công thức).
#   E) BIỂU ĐỒ: biến liên tục → HISTOGRAM (không khoảng trắng) ; biến phân loại → BAR CHART (có khoảng trắng).
# ============================================================
def clean_series(x):
    return pd.Series(x).dropna().astype(float)            # 1 dãy số (Series), bỏ ô thiếu

# ======== XU THẾ TRUNG TÂM ========
def calc_mean(x):
    x = clean_series(x); return x.sum() / len(x)                      # trung bình cộng = Σx / n
def calc_geometric(x):
    x = clean_series(x); return math.exp(np.log(x[x > 0]).mean())     # trung bình nhân = exp(mean(ln x)), x > 0
def calc_harmonic(x):
    x = clean_series(x); return len(x) / (1 / x).sum()                # điều hòa = n / Σ(1/x), x ≠ 0
def calc_median(x):                                                   # trung vị: sắp xếp rồi lấy giữa
    s = np.sort(clean_series(x).values); n = len(s)
    return s[n // 2] if n % 2 == 1 else (s[n // 2 - 1] + s[n // 2]) / 2
def calc_mode(x):
    return clean_series(x).mode().tolist()                            # yếu vị (có thể nhiều mode)

# ======== PHÂN TÁN ========
def calc_range(x):
    x = clean_series(x); return x.max() - x.min()                     # khoảng biến thiên
def calc_percentile(x, p):
    return np.percentile(clean_series(x), p)                          # bách phân vị P(p%)
def calc_iqr(x):
    return calc_percentile(x, 75) - calc_percentile(x, 25)            # khoảng tứ phân vị Q3 - Q1
def calc_variance(x, ddof=1):                                         # phương sai: Σ(x-x̄)²/(n-ddof)
    x = clean_series(x)
    return ((x - calc_mean(x)) ** 2).sum() / (len(x) - ddof)
def calc_std(x, ddof=1):
    return math.sqrt(calc_variance(x, ddof))                          # độ lệch chuẩn

# ======== HÌNH DÁNG ========
def skew_pearson2(x):                                                 # 3(mean - median)/s
    return 3 * (calc_mean(x) - calc_median(x)) / calc_std(x, 1)
def skew_direction(x):
    m, md = calc_mean(x), calc_median(x)
    return "Lệch PHẢI (đuôi bên phải)" if m > md else ("Lệch TRÁI (đuôi bên trái)" if m < md else "Đối xứng")
def calc_kurtosis(x):                                                 # độ nhọn: mean(((x-x̄)/σ)⁴) ; chuẩn = 3
    x = clean_series(x)
    z = (x - calc_mean(x)) / calc_std(x, 0)
    return (z ** 4).mean()

# ======== CHEBYSHEV: tỷ lệ dữ liệu trong ±zσ ≥ 1 - 1/z² ========
def chebyshev_ratio(z):
    return 1 - 1 / z ** 2

# ======== HIỆP PHƯƠNG SAI / PEARSON / SPEARMAN viết tay cho 2 dãy x, y ========
def pair_clean(x, y):
    t = pd.DataFrame({"x": x, "y": y}).dropna()
    return t["x"].astype(float), t["y"].astype(float)

def covariance(x, y, ddof=1):                                         # cov = Σ(x-x̄)(y-ȳ)/(n-ddof)
    x, y = pair_clean(x, y)
    return ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - ddof)

def pearson(x, y):                                                    # r = cov / (σx σy) → khử đơn vị, trong [-1,1]
    x, y = pair_clean(x, y)
    return covariance(x, y) / (x.std(ddof=1) * y.std(ddof=1))

def spearman(x, y):                                                   # Pearson áp dụng trên THỨ HẠNG (đồng hạng: rank trung bình)
    x, y = pair_clean(x, y)
    return pearson(x.rank(method="average"), y.rank(method="average"))

# --- Sử dụng trên Sales ---
sales = df_clean["Sales"]
res = {"Trung bình cộng": calc_mean(sales), "Trung bình nhân": calc_geometric(sales), "Trung bình điều hòa": calc_harmonic(sales),
       "Trung vị": calc_median(sales), "Range": calc_range(sales), "P90": calc_percentile(sales, 90), "IQR": calc_iqr(sales),
       "Phương sai (n-1)": calc_variance(sales, 1), "Phương sai (n)": calc_variance(sales, 0), "Độ lệch chuẩn (n-1)": calc_std(sales, 1),
       "Skew Pearson2": skew_pearson2(sales), "Kurtosis": calc_kurtosis(sales)}
print(pd.Series(res).round(3).to_string())
print("Hướng lệch:", skew_direction(sales), "→ trung bình nhân gần trung vị hơn trung bình cộng khi biến thiên lớn")
print("Kurtosis > 3: nhọn hơn phân phối chuẩn" if calc_kurtosis(sales) > 3 else "Kurtosis < 3: bẹt hơn phân phối chuẩn")
print("Chebyshev z=2:", chebyshev_ratio(2), "| z=3:", round(chebyshev_ratio(3), 3))
print("Mode Order Quantity:", calc_mode(df_clean["Order Quantity"])[:5])

# Kiểm tra bằng numpy/pandas (hộp đen): ddof khác nhau cho kết quả khác nhau
check("Phương sai mẫu = pandas var", calc_variance(sales, 1), sales.var(ddof=1), tol=1e-3)
check("Phương sai quần thể = numpy var", calc_variance(sales, 0), np.var(sales, ddof=0), tol=1e-3)

# Ví dụ trung bình điều hòa: đi 60 km/h rồi quay về 40 km/h → vận tốc TB = 48 (KHÔNG phải 50)
print("Vận tốc TB (điều hòa):", calc_harmonic([60, 40]))

# Tương quan
print("Cov =", round(covariance(df_clean["Sales"], df_clean["Profit"]), 2), "| Pearson =", round(pearson(df_clean["Sales"], df_clean["Profit"]), 4),
      "| Spearman =", round(spearman(df_clean["Sales"], df_clean["Profit"]), 4))
check("Pearson = pandas corr", pearson(df_clean["Sales"], df_clean["Profit"]), df_clean["Sales"].corr(df_clean["Profit"]), tol=1e-6)
cm = df_clean[["Sales", "Profit", "Shipping Cost"]].cov(); cr = df_clean[["Sales", "Profit", "Shipping Cost"]].corr()
print("Đường chéo ma trận hiệp phương sai = phương sai:", np.allclose(np.diag(cm), df_clean[["Sales", "Profit", "Shipping Cost"]].var()))
print("Đường chéo ma trận tương quan = 1:", np.allclose(np.diag(cr), 1))

# Histogram (không khoảng trắng) vs Bar chart (có khoảng trắng)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(df_clean["Sales"].clip(upper=df_clean["Sales"].quantile(0.95)), bins=20, edgecolor="black"); ax[0].set_title("Histogram Sales")
df_clean["Region"].value_counts().plot(kind="bar", ax=ax[1], width=0.6); ax[1].set_title("Bar chart Region"); plt.show()

## PHẦN 13 – PIPELINE LÀM BÀI KIỂM TRA (Trang 11, 15–16)

In [ ]:
# ============================================================
# 13.1 NHÓM HÀM: Các BƯỚC của pipeline (mỗi bước = 1 hàm thuần step_*)
# ------------------------------------------------------------
# 📖 TRANG 15-16: PHƯƠNG PHÁP LÀM BÀI KIỂM TRA:
#   1. Xác định yêu cầu & mục tiêu bài toán
#   2. Lướt dữ liệu: kiểu dữ liệu, loại dữ liệu (thô / tính toán)
#   3. Xác định input và output
#   4. Đánh giá chất lượng dữ liệu (LÂU NHẤT & thử nhiều nhất): (1) thiếu (2) bất thường (3) không nhất quán
#   5. Tích hợp → Biến đổi → Mã hóa → Thu gọn → Thống kê mô tả
# 📖 TRANG 11 (quy tắc): không chỉnh sửa dữ liệu thô; so sánh kết quả trước/sau; linh hoạt biện pháp.
# 🎯 DÙNG KHI: đề bài yêu cầu "code hoàn chỉnh pipeline tiền xử lý dữ liệu". Chỉ cần sửa dict CONFIG (ô 13.2) cho hợp đề; các hàm step_* không phải sửa.
# 🧩 QUY ƯỚC CHUNG của mọi hàm bước:   step_xxx(d, cfg)  →  (d_mới, notes, report)
#       d      : DataFrame đưa vào (hàm tự .copy(), không sửa bảng gốc)      cfg : dict cấu hình (tên cột, ngưỡng…)
#       notes  : list các câu ghi log để GIẢI TRÌNH trong bài                 report : dict các bảng minh họa (trước/sau, thống kê…)
#   run_pipeline(raw, cfg) chỉ việc gọi lần lượt các hàm step_* → KHÔNG có biến toàn cục, KHÔNG có class.
# ------------------------------------------------------------
# 📚 LÝ THUYẾT [Word – Phương pháp làm bài kiểm tra & Quy trình tiền xử lý]:
#   - Cả 2 môn đều có CÙNG phần TIỀN XỬ LÝ DỮ LIỆU trong pipeline code để kiểm tra; thầy kiểm tra TOÀN BỘ pipeline (code hoàn thiện một phần).
#   - KDD process: Data → Target data → (Preprocessing/Wrangling) → Clean data → …
#   - Giai đoạn 2 của tiền xử lý: DATA INTEGRATION (tích hợp) → DATA TRANSFORMATION (biến đổi) → DATA REDUCTION (thu gọn).
#   - Các bước làm bài: 1) xác định yêu cầu & mục tiêu ; 2) lướt dữ liệu (kiểu & loại dữ liệu: thô/tính toán) ; 3) xác định input/output
#     (lúc dễ lúc không) ; 4) ĐÁNH GIÁ CHẤT LƯỢNG (LÂU NHẤT, thử nhiều nhất, tạo khác biệt): (1) thiếu (2) bất thường (3) không nhất quán ; 5)...
#   - Quy tắc: KHÔNG chỉnh sửa dữ liệu thô ; cân nhắc chất lượng & hiệu quả (so sánh TRƯỚC–SAU) ; LINH HOẠT biện pháp & công đoạn
#     (có thể quay nhiều vòng, không theo trình tự cố định).
# 🔎 CHỌN KHI NÀO – BẢN ĐỒ "KẾT QUẢ NHÌN THẤY → HÀNH ĐỘNG TIẾP THEO" TRONG PIPELINE:
#   - B2 thấy cột là số nhưng thực chất là mã (Zip Code, ID)       → coi là ĐỊNH DANH, không chuẩn hóa/không tính mean.
#   - B4a có ô thiếu                                               → chẩn đoán MCAR/MAR/MNAR → chọn 1 trong 7 biện pháp (ô 5.1).
#   - B4b mean ≫ median, max rất xa Q3                             → outlier + lệch phải → IQR/log/Yeo-Johnson/winsorize (ô 6.1).
#   - B4c ngày giao < ngày đặt / giá trị vô lý / dòng trùng        → không nhất quán → loại hoặc sửa theo logic nghiệp vụ.
#   - B5 cần thêm thuộc tính ở bảng khác                            → tích hợp (left join) rồi kiểm tra số dòng.
#   - B6 các cột khác thang đo hoặc phân phối lệch                  → chuẩn hóa/biến đổi (ô 9.1).
#   - B7 còn cột chữ mà thuật toán chỉ nhận số                      → mã hóa theo loại (ordinal/one-hot/dummy/hashing – ô 10.1).
#   - B8 nhiều cột tương quan cao / biến thiên thấp                  → thu gọn (lọc/forward/Lasso/PCA – ô 11.1).
#   - B9 mô tả kết quả cuối                                          → thống kê mô tả (ô 12.1).
#   - Mỗi bước: LUÔN so sánh TRƯỚC/SAU và ghi vào log (note) để giải trình trong bài.
# ============================================================

# ----- B1 + B2 + B3: mục tiêu, lướt dữ liệu, xác định input/output -----
def step_overview(d, cfg):
    n_num = d.select_dtypes("number").shape[1]                    # số cột kiểu số
    n_time = d.select_dtypes("datetime").shape[1]                 # số cột thời gian
    notes = [
        f"B1. Mục tiêu: phân tích/dự đoán '{cfg['target']}' từ các đặc trưng đơn hàng.",
        f"B2. Kích thước {d.shape}; {n_num} cột số, {n_time} cột thời gian, {d.shape[1] - n_num - n_time} cột chữ.",
        f"B3. OUTPUT = {cfg['target']} ; INPUT = {cfg['inputs']}",
    ]
    report = {"structure": step1_structure(d, cfg["var_type"], cfg["computed_cols"], cfg["target"]),
              "missing": missing_table(d), "outliers": outlier_table(d), "inconsistency": inconsistency_report(d)}
    return d, notes, report

# ----- B4a: dữ liệu bị thiếu (7 biện pháp) -----
def step_missing(d, cfg):
    before = d                                                    # giữ lại để so sánh TRƯỚC/SAU
    d = d.copy(); notes = []
    n0 = len(d)
    d = m2_drop_rows(d, cfg["drop_rows_cols"], cfg["drop_rows_how"])                       # BP2: loại dòng
    notes.append(f"B4a-BP2. Loại dòng thiếu '{cfg['drop_rows_how']}' của {cfg['drop_rows_cols']}: {n0} → {len(d)} dòng.")
    d, dropped = m3_drop_cols(d, cfg["drop_col_threshold"], show_corr=False)                # BP3: loại cột > ngưỡng
    notes.append(f"B4a-BP3. Bỏ cột thiếu > {cfg['drop_col_threshold']:.0%}: {dropped}.")
    for col, value in cfg["special_fill"].items():                                          # BP4: giá trị đặc biệt
        d = m4_fill_special(d, col, value)
    notes.append(f"B4a-BP4. Điền giá trị đặc biệt: {cfg['special_fill']}.")
    for col, group in cfg["group_fill"]:                                                    # BP5: thay theo nhóm (MAR)
        d = m5_group_fill(d, col, group, "median")
    notes.append(f"B4a-BP5. Thay theo nhóm (median): {cfg['group_fill']}.")
    mice_cols = [c for c in cfg["mice_cols"] if c in d.columns]                             # BP7: MICE
    d, _ = m7_mice_manual(d, mice_cols, n_iter=cfg["mice_iter"], m=cfg["mice_m"])
    notes.append(f"B4a-BP7. MICE tự viết trên {mice_cols} (n_iter={cfg['mice_iter']}, m={cfg['mice_m']}).")
    if cfg["fill_remaining"]:                                                               # lưới an toàn cho cột còn thiếu
        left = d.columns[d.isna().any()].tolist()
        for c in left:
            if pd.api.types.is_datetime64_any_dtype(d[c]):
                notes.append(f"B4a. ⚠ Cột thời gian '{c}' còn thiếu → giữ nguyên (cần xử lý riêng).")
            elif pd.api.types.is_numeric_dtype(d[c]):
                d = m4_fill_stat(d, c, "median")
            else:
                d = m4_fill_special(d, c, "Unknown")
        if left: notes.append(f"B4a. Lưới an toàn: điền nốt cột còn thiếu {left} (số → median, chữ → 'Unknown').")
    notes.append(f"B4a. Kết quả: còn {int(d.isna().sum().sum())} ô thiếu | {len(d)} dòng | {d.shape[1]} cột.")
    return d, notes, {"before_after": compare_before_after(before, d, mice_cols), "missing_after": mv_missing_table(d)}

# ----- B4b: dữ liệu bất thường (phát hiện + xử lý) -----
def apply_outlier_method(d, col, method):
    # trả về (cột mới, hậu tố tên cột) theo cách xử lý đã chọn trong cfg["outlier_plan"]
    if method == "winsorize":   return winsorize(d, col), "wins"
    if method == "log":         return log_transform(d, col), "log"
    if method == "yeojohnson":  return yeojohnson_transform(d, col)[0], "yj"
    if method == "boxcox":      return boxcox_transform(d, col)[0].reindex(d.index), "bc"
    raise ValueError("method phải là winsorize / log / yeojohnson / boxcox")

def step_outliers(d, cfg):
    d = d.copy(); notes = []; rows = []
    for col, method in cfg["outlier_plan"].items():
        n_sigma = len(zscore_outliers(d, col))                       # 3-sigma
        n_iqr = len(iqr_outliers(d, col)[0])                         # IQR / boxplot
        skew_before = d[col].skew()
        new_values, suffix = apply_outlier_method(d, col, method)
        new_col = f"{col}_{suffix}"
        d[new_col] = new_values                                      # KHÔNG ghi đè cột gốc → giữ nguyên để so sánh
        rows.append({"Cột": col, "Outlier 3-sigma": n_sigma, "Outlier IQR": n_iqr, "Skew trước": round(skew_before, 2),
                     "Cách xử lý": method, "Cột mới": new_col, "Skew sau": round(d[new_col].skew(), 2)})
        notes.append(f"B4b. {col}: {n_sigma} outlier (3-sigma), {n_iqr} (IQR); xử lý '{method}' → cột {new_col} (skew {skew_before:.2f} → {d[new_col].skew():.2f}).")
    return d, notes, {"outlier_summary": pd.DataFrame(rows)}

# ----- B4c: dữ liệu không nhất quán -----
def step_inconsistency(d, cfg):
    before = inconsistency_report(d)
    clean, n_bad, n_dup = drop_inconsistent_rows(d)
    notes = [f"B4c. Loại {n_bad} dòng vi phạm logic (giao trước đặt, Discount ngoài [0,1], Sales/Qty ≤ 0) và {n_dup} dòng trùng lặp → còn {len(clean)} dòng."]
    return clean, notes, {"before": before, "after": inconsistency_report(clean)}

# ----- B5: tích hợp dữ liệu + nhận diện thông tin dư thừa -----
def step_integrate(d, cfg):
    n0 = len(d)
    merged = consolidate(d, cfg["path"])                             # Orders + Returns + Users (left join)
    same_rows = len(merged) == n0                                    # DẤU HIỆU lỗi join: số dòng tăng
    ratio = derived_check(merged)["50%"]                             # trung vị của calc/Sales (≈1 → Sales là cột dẫn xuất)
    pairs = redundant_pairs(merged, cfg["inputs"], cfg["redundancy_threshold"])
    notes = [f"B5. Tích hợp Returns + Users: {d.shape} → {merged.shape}; số dòng {'KHÔNG đổi ✔' if same_rows else 'THAY ĐỔI ⚠ (khóa bảng phụ bị trùng)'}.",
             f"B5. Thuộc tính dẫn xuất: trung vị (UnitPrice×Qty×(1−Discount))/Sales = {ratio:.3f}.",
             f"B5. Cặp input nghi dư thừa (|r| ≥ {cfg['redundancy_threshold']}): {pairs if pairs else 'không có'}."]
    report = {"metadata": metadata(merged), "covariance": covariance_matrix(merged, cfg["inputs"]).round(2),
              "pearson": pearson_matrix(merged, cfg["inputs"]).round(2), "spearman": spearman_matrix(merged, cfg["inputs"]).round(2)}
    return merged, notes, report

# ----- B6: biến đổi dữ liệu -----
def step_transform(d, cfg):
    d = d.copy(); notes = []
    scalers = {"zscore": zscore, "minmax": minmax, "robust": robust, "decimal": decimal_scaling}
    for col, method in cfg["scale_plan"].items():                    # chuẩn hóa từng cột theo kế hoạch
        d[f"{col}_{method}"] = scalers[method](d, col)
    notes.append(f"B6. Chuẩn hóa: {cfg['scale_plan']}.")
    d[f"{cfg['target']}_positive"] = binarize(d, cfg["target"], cfg["binarize_threshold"])
    notes.append(f"B6. Nhị phân hóa {cfg['target']} > {cfg['binarize_threshold']} (tỷ lệ = 1: {d[cfg['target'] + '_positive'].mean():.3f}).")
    for col, k in cfg["discretize_plan"].items():                    # rời rạc hóa equal-frequency
        d[f"{col}_group"] = equal_frequency(d, col, k).astype(str)
    notes.append(f"B6. Rời rạc hóa equal-frequency: {cfg['discretize_plan']}.")
    d = construct_features(d)                                        # feature construction
    notes.append("B6. Tạo thuộc tính mới: Sales_sq, Revenue_per_unit, Qty_x_Price, Year, Month, Region_Segment.")
    daily = d.groupby(cfg["date_col"])["Sales"].sum().asfreq("D").fillna(0)          # làm trơn chuỗi thời gian
    smooth = pd.DataFrame({"Gốc": daily, "SMA7": sma(daily, 7), "Centered15": centered_sma(daily, 15), "WMA7": wma(daily, 7)})
    notes.append("B6. Làm trơn doanh số theo ngày: SMA7, Centered SMA15, WMA7 (xem report).")
    return d, notes, {"roll_up": roll_up(d, cfg["date_col"]), "drill_down": drill_down(d, cfg["date_col"]), "smooth": smooth}

# ----- B7: mã hóa dữ liệu phân loại -----
def step_encode(d, cfg):
    d = d.copy(); notes = []; target = cfg["target"]
    for col, order in cfg["ordinal_plan"].items():                   # ordinal: GIỮ thứ tự
        d[f"{col}_code"], _ = ordinal_encode(d, col, order)
    notes.append(f"B7. Ordinal: {list(cfg['ordinal_plan'])}.")
    for col in cfg["dummy_cols"]:                                    # dummy: k-1 cột
        d = pd.concat([d, dummy(d, col)], axis=1)
    notes.append(f"B7. Dummy (k−1 cột): {cfg['dummy_cols']}.")
    for col in cfg["freq_cols"]:                                     # frequency: chỉ khi tần suất các nhãn KHÁC nhau
        freq = frequency_encode(d, col)
        if freq.nunique() == d[col].nunique():
            d[f"{col}_freq"] = freq
            notes.append(f"B7. Frequency '{col}': tần suất các nhãn khác nhau ✔ → dùng được.")
        else:
            notes.append(f"B7. Frequency '{col}': có nhãn trùng tần suất ✘ → KHÔNG dùng.")
    for col in cfg["loo_cols"]:                                      # leave-one-out (giảm rò rỉ nhãn)
        enc = leave_one_out(d, col, target).replace([np.inf, -np.inf], np.nan)
        d[f"{col}_loo"] = enc.fillna(target_encode(d, col, target))  # nhãn chỉ có 1 dòng → dùng mean target
    notes.append(f"B7. Leave-one-out theo {target}: {cfg['loo_cols']}.")
    for col in cfg["hash_cols"]:                                     # hashing: cột có rất nhiều nhãn
        d[f"{col}_hash"] = hashing(d, col, cfg["hash_buckets"])
    notes.append(f"B7. Feature hashing ({cfg['hash_buckets']} bucket): {cfg['hash_cols']}.")
    return d, notes, {}

# ----- B8: thu gọn dữ liệu (giảm chiều) -----
def step_reduce(d, cfg):
    cols, target = cfg["inputs"], cfg["target"]
    data = d.dropna(subset=cols + [target])
    var, low_var = variance_threshold(data, cols, cfg["var_threshold"])          # Filter 1: phương sai
    kept = [c for c in cols if c not in low_var]
    corr_drop = correlation_filter(data, kept, cfg["corr_threshold"])            # Filter 2: tương quan cao
    kept = [c for c in kept if c not in corr_drop]
    chosen = forward_selection(data, kept, target, min(cfg["forward_k"], len(kept)), verbose=False)   # Wrapper
    lasso = lasso_select(data, kept, target, cfg["lasso_alpha"])                 # Embedded
    _, ratio = pca_manual(data, kept, min(cfg["pca_components"], len(kept)))     # Rút trích: PCA
    notes = [f"B8. Filter: bỏ cột phương sai thấp {low_var}; bỏ cột tương quan cao {corr_drop}.",
             f"B8. Wrapper (forward selection, k={cfg['forward_k']}) chọn: {chosen}.",
             f"B8. Embedded (Lasso) – cột có hệ số 0: {lasso[lasso == 0].index.tolist()}.",
             f"B8. PCA: tỷ lệ phương sai giải thích {np.round(ratio, 3).tolist()}."]
    report = {"variance": var, "lasso": lasso, "pca_ratio": pd.Series(ratio, name="Tỷ lệ phương sai"),
              "selected": chosen, "df_model": data[chosen + [target]].copy()}
    return d, notes, report

# ----- B9: thống kê mô tả kết quả cuối -----
def step_describe(d, cfg):
    cols, target = [cfg["target"]] + cfg["inputs"], cfg["target"]
    rows = {}
    for c in cols:
        x = d[c]
        rows[c] = {"mean": calc_mean(x), "geometric": calc_geometric(x), "median": calc_median(x), "std(n-1)": calc_std(x, 1),
                   "IQR": calc_iqr(x), "skew P2": skew_pearson2(x), "kurtosis": calc_kurtosis(x), "hướng lệch": skew_direction(x)}
    table = pd.DataFrame(rows).T.round(3)
    corr = pd.DataFrame({"pearson": {c: pearson(d[c], d[target]) for c in cfg["inputs"]},
                         "spearman": {c: spearman(d[c], d[target]) for c in cfg["inputs"]}}).round(3)
    notes = [f"B9. {target}: mean={calc_mean(d[target]):.2f}, median={calc_median(d[target]):.2f}, std={calc_std(d[target], 1):.2f}, {skew_direction(d[target])}."]
    return d, notes, {"describe": table, "corr_with_target": corr}


### 13.2 CHẠY PIPELINE: `CONFIG` (dict cấu hình) → `run_pipeline(raw, cfg)`

| Bước | Hàm | Việc làm | Trang notes |
|---|---|---|---|
| B1–B3 | `step_overview` | mục tiêu, lướt dữ liệu, input/output, bảng chất lượng ban đầu | 8–10, 15–16 |
| B4a | `step_missing` | 7 biện pháp xử lý thiếu (loại dòng/cột, đặc biệt, theo nhóm, MICE…) | 11–12 |
| B4b | `step_outliers` | phát hiện (3-sigma, IQR) + xử lý (winsorize / log / Yeo-Johnson / Box-Cox) | 13–14 |
| B4c | `step_inconsistency` | loại dòng vô lý & dòng trùng | 9 |
| B5 | `step_integrate` | merge Returns + Users, kiểm tra dẫn xuất/dư thừa | 10, 16–17 |
| B6 | `step_transform` | chuẩn hóa, nhị phân hóa, rời rạc hóa, feature construction, làm trơn | 17–19 |
| B7 | `step_encode` | ordinal, dummy, frequency, leave-one-out, hashing | 19 |
| B8 | `step_reduce` | Filter → Wrapper → Lasso → PCA | 20–21 |
| B9 | `step_describe` | thống kê mô tả + tương quan với target | 21–23 |

> Muốn dùng cho đề khác: **chỉ sửa `CONFIG`** (tên cột, ngưỡng, kế hoạch xử lý). Hàm `run_pipeline` chỉ gọi lần lượt các hàm bước, không giữ trạng thái nào bên ngoài.

In [ ]:
# ============================================================
# 13.2 CONFIG + run_pipeline  (không class, không biến toàn cục: mọi thứ đi qua tham số / return)
# ------------------------------------------------------------
# 📖 TRANG 11, 15-16: không sửa dữ liệu thô · so sánh TRƯỚC/SAU · linh hoạt công đoạn
# 🎯 DÙNG KHI: đề yêu cầu chạy trọn pipeline từ file Excel → dữ liệu sẵn sàng phân tích.
# ============================================================
CONFIG = {
    # --- B1–B3: mục tiêu, input/output ---
    "path": DATA_PATH,                       # file Excel (dùng ở bước tích hợp để đọc sheet Returns, Users)
    "target": "Profit",                      # OUTPUT
    "inputs": ["Order Quantity", "Discount", "Unit Price", "Shipping Cost", "Sales"],   # INPUT (số)
    "var_type": VAR_TYPE, "computed_cols": COMPUTED,
    "date_col": "Order Date",
    # --- B4a: dữ liệu thiếu ---
    "drop_rows_cols": ["Sales", "Order Quantity", "Customer Name", "City", "Ship Mode"], "drop_rows_how": "all",   # BP2
    "drop_col_threshold": 0.20,                                          # BP3: bỏ cột thiếu > 20%
    "special_fill": {"Ship Mode": "Unknown"},                            # BP4 (MCAR, biến phân loại)
    "group_fill": [("Discount", "Customer Segment"), ("Unit Price", "Product Sub-Category")],   # BP5 (MAR)
    "mice_cols": ["Order Quantity", "Sales", "Discount", "Unit Price", "Shipping Cost", "Profit"], "mice_iter": 3, "mice_m": 2,   # BP7
    "fill_remaining": True,                                              # lưới an toàn cho cột còn thiếu
    # --- B4b: bất thường (cột → cách xử lý: winsorize | log | yeojohnson | boxcox) ---
    "outlier_plan": {"Sales": "winsorize", "Profit": "yeojohnson"},
    # --- B5: tích hợp / dư thừa ---
    "redundancy_threshold": 0.8,
    # --- B6: biến đổi (cột → zscore | minmax | robust | decimal) ---
    "scale_plan": {"Sales_wins": "zscore", "Unit Price": "robust", "Shipping Cost": "robust", "Discount": "minmax"},
    "discretize_plan": {"Sales": 4}, "binarize_threshold": 0,
    # --- B7: mã hóa ---
    "ordinal_plan": {"Order Priority": ["Not Specified", "Low", "Medium", "High", "Critical"]},
    "dummy_cols": ["Customer Segment", "Region"], "freq_cols": ["Region"],
    "loo_cols": ["State"], "hash_cols": ["Customer Name"], "hash_buckets": 8,
    # --- B8: thu gọn ---
    "var_threshold": 0.001, "corr_threshold": 0.8, "forward_k": 3, "lasso_alpha": 1.0, "pca_components": 2,
}

def run_pipeline(raw, cfg, log_file=None, verbose=True):
    fingerprint = snapshot_hash(raw)                     # dấu vân tay dữ liệu thô TRƯỚC khi chạy
    d = raw.copy()                                       # luôn làm việc trên bản sao
    steps = [("overview", step_overview), ("missing", step_missing), ("outliers", step_outliers),
             ("inconsistency", step_inconsistency), ("integration", step_integrate), ("transform", step_transform),
             ("encoding", step_encode), ("reduction", step_reduce), ("describe", step_describe)]
    log, reports = [], {}
    for name, step in steps:
        d, notes, report = step(d, cfg)                  # mỗi bước: nhận d, trả (d mới, ghi chú, bảng minh họa)
        log.extend(notes); reports[name] = report
        if verbose:
            for m in notes: print("•", m)
    reports["compare_raw_final"] = compare_before_after(raw, d, [cfg["target"]] + cfg["inputs"])   # so sánh TRƯỚC/SAU toàn pipeline
    untouched = snapshot_hash(raw) == fingerprint
    log.append("✔ Dữ liệu thô KHÔNG bị sửa (quy tắc trang 11)." if untouched else "⚠ Dữ liệu thô BỊ SỬA – kiểm tra lại!")
    if verbose: print(log[-1])
    if log_file: write_lines(log_file, log)              # ghi file log để nộp/giải trình
    return d, log, reports

# --- Sử dụng ---
df_final, pipe_log, reports = run_pipeline(raw_df, CONFIG, log_file="pipeline_log.txt")
print("\nDữ liệu cuối:", df_final.shape, "| dữ liệu thô vẫn nguyên:", raw_df.shape)
print("Feature được chọn cho mô hình:", reports["reduction"]["selected"])
display(reports["compare_raw_final"])
display(reports["describe"]["describe"])
display(reports["describe"]["corr_with_target"])
df_model = reports["reduction"]["df_model"]              # bảng gọn: feature đã chọn + target
df_final.head()

## ✅ CHECKLIST NHANH KHI LÀM BÀI
1. Đọc đề → xác định **mục tiêu**, **target (output)**, **features (input)** (trang 15–16).
2. `step1_structure(df)` → xem kiểu dữ liệu (trang 8–9).
3. `missing_table(df)` / `missing_rate_by_group(...)` → chẩn đoán MCAR/MAR/MNAR → chọn **1 trong 7 biện pháp** (trang 11–12).
4. các hàm `zscore_outliers / iqr_outliers / mahalanobis_outliers` và `log_transform / yeojohnson_transform / winsorize` → 3-sigma / IQR / Mahalanobis → Log / Box-Cox / Yeo-Johnson / Winsorize (trang 13–14).
5. các hàm `z_test_one_sample / t_test_two_groups / anova / chi_square` khi cần kiểm tra mối liên hệ (trang 15, 17).
6. `consolidate(...)` → ghép sheet, bỏ cột dư thừa (trang 16).
7. các hàm biến đổi (`zscore`, `minmax`, `equal_frequency`…) + hàm mã hóa (`ordinal_encode`, `onehot`, `dummy`…) → chuẩn hóa/mã hóa (trang 17–19).
8. các hàm thu gọn (`correlation_filter`, `forward_selection`, `pca_manual`…) → giảm cột/dòng, PCA (trang 20–21).
9. các hàm `calc_mean / calc_median / skew_pearson2 / calc_kurtosis…` → mô tả kết quả (trang 21–23).
10. **So sánh trước/sau** mỗi bước & **không sửa dữ liệu thô** (trang 11).
11. Chạy trọn quy trình: sửa `CONFIG` rồi gọi `df_final, log, reports = run_pipeline(raw_df, CONFIG)` (Phần 13).

## 🧭 BẢNG TRA NHANH: DẤU HIỆU DỮ LIỆU / KẾT QUẢ → CHỌN CÔNG CỤ NÀO
> Tổng hợp từ các khối **🔎 CHỌN KHI NÀO** ở từng nhóm hàm. Chi tiết lý thuyết và nguồn (Word hay bổ sung) nằm trong ghi chú của từng nhóm hàm.

### 1. Nhận diện loại biến
| Dấu hiệu | Loại biến | Hệ quả |
|---|---|---|
| Nhãn chữ, không xếp hơn–kém được | Định danh (nominal) | One-hot/Dummy/Hashing ; không tính mean |
| Nhãn chữ, xếp được thứ tự | Định tính (ordinal) | Mã hóa **giữ thứ tự** |
| Số đếm nguyên | Định lượng rời rạc | – |
| Số đo liên tục, có zero thật | Định lượng ratio | Hiệu & thương đều có nghĩa |
| Số nhưng chỉ là mã (ID, Zip) | Thực chất định danh | Không cộng/trừ/mean |

### 2. Quan hệ giữa 2 biến
| Cặp biến | Công cụ |
|---|---|
| số – số | Covariance (dấu) · **Pearson** (tuyến tính) · **Spearman** (thứ hạng/lệch/outlier) |
| phân loại – phân loại | Crosstab · Chi-square |
| số – phân loại 2 nhóm | t-test |
| số – phân loại ≥3 nhóm | ANOVA · parallel boxplot |

### 3. Dữ liệu thiếu
| Dấu hiệu | Chọn |
|---|---|
| Ô thiếu không liên quan câu hỏi | Giữ nguyên |
| Nhiều cột thiếu cùng lúc trên cùng dòng | Loại dòng |
| Cột thiếu > ~20% và ít tương quan | Loại cột |
| Thiếu đều (MCAR), biến phân loại | 'Unknown' / mode |
| Thiếu đều (MCAR), biến số | mean (cân đối) / median (lệch) |
| Tỷ lệ thiếu khác nhau theo nhóm (MAR) | Thay theo nhóm |
| Cột số tương quan với cột khác | Hồi quy / kNN |
| Nhiều cột số thiếu & tương quan lẫn nhau | MICE |
| Giá trị còn lại cụt đuôi (MNAR) | Mô hình + tri thức ngoài |

### 4. Outlier & biến đổi phân phối
| Dấu hiệu | Chọn |
|---|---|
| Gần chuẩn | 3-sigma (\|z\|>3) |
| Lệch mạnh (mean ≫ median) | IQR/boxplot, hoặc log rồi z |
| 2 biến cùng thang đo | Khoảng cách Euclid |
| 2 biến khác thang đo | Mahalanobis |
| Lệch phải, **dương** | Log (hoặc Box-Cox) |
| Có **âm hoặc 0** | Yeo-Johnson |
| Muốn giữ đủ số dòng, chỉ kéo ngoại lai | Winsorize (không đổi dạng phân phối!) |
| Chuỗi thời gian | Kỳ trước / cùng kỳ năm trước (mùa vụ) / nội suy |

### 5. Chuẩn hóa
| Dấu hiệu | Chọn |
|---|---|
| Gần chuẩn, ít outlier | Z-score |
| Không chuẩn, ít outlier, cần [0,1] | Min-Max (outlier làm dồn sát 0 → đổi sang Robust) |
| Không chuẩn, nhiều outlier | Robust (median/IQR) |
| Chỉ cần đưa \|x\|<1 nhanh | Decimal scaling |

### 6. Rời rạc hóa / làm trơn
| Dấu hiệu | Chọn |
|---|---|
| Phân phối đều hoặc khoảng có nghĩa nghiệp vụ | Equal-width |
| Phân phối lệch, cần nhóm cân bằng | Equal-frequency (qcut) |
| Có target, cần điểm cắt phân biệt nhãn | Supervised binning |
| Chuỗi thời gian nhiễu, xem xu hướng | SMA ; phân tích lịch sử → Centered SMA ; gần quan trọng hơn → WMA |

### 7. Mã hóa biến phân loại
| Dấu hiệu | Chọn |
|---|---|
| Ordinal | Label/Ordinal encoding **giữ thứ tự** |
| Nominal ít nhãn | One-hot (k cột) / Dummy (k−1 cột) |
| Nominal nhiều nhãn + có target | Target / Mean / Leave-one-out |
| Số nhãn rất lớn | Feature Hashing |
| Tần suất các nhãn **không trùng nhau** | Frequency encoding |

### 8. Thu gọn
| Dấu hiệu | Chọn |
|---|---|
| Phương sai (cùng thang đo) quá nhỏ | Filter – bỏ cột |
| 2 input tương quan cao | Bỏ 1 cột |
| Có target, ít feature | Wrapper (forward) / RFE |
| Mô hình có sẵn cơ chế | Embedded (Lasso hệ số = 0 / cây) |
| Muốn giảm chiều, chấp nhận mất nghĩa cột | PCA/SVD (chuẩn hóa trước; giữ ~80–90% phương sai) |
| Cần giải thích bằng cột gốc | Chọn lọc thuộc tính, không dùng PCA |

### 9. Thống kê mô tả
| Dấu hiệu | Chọn |
|---|---|
| mean ≈ median, ít biến thiên | Trung bình cộng |
| Biến thiên lớn / lệch phải | Trung vị hoặc Trung bình nhân (cái gần trung vị hơn) |
| Tỷ số khác đơn vị | Trung bình điều hòa |
| Lệch/outlier | IQR thay cho range/std |
| Skew Pearson₂ > 0 / < 0 | Lệch phải / lệch trái |
| Kurtosis > 3 / < 3 | Nhọn (đuôi dày) / bẹt hơn chuẩn |

### 10. Kiểm định giả thuyết
| Dấu hiệu | Chọn |
|---|---|
| Dấu "=" / "khác" | 2 phía, H0 chứa dấu "=" |
| "lớn hơn / nhỏ hơn" | 1 phía theo hướng H1 |
| p < α (0.05) | Bác bỏ H0 |
| p ≥ α | Không bác bỏ H0 (≠ H0 đúng) |

---
### ⚠️ Những chỗ trong file Word / notebook có vẻ chép nhầm hoặc tự mâu thuẫn (đã xử lý như sau trong ghi chú)
1. **Manhattan vs Chebyshev** (mục Tọa độ Đề-các, A*): Word mô tả hơi lẫn. Notebook dùng định nghĩa chuẩn: Manhattan = |Δcột|+|Δdòng|, Chebyshev = max(|Δcột|,|Δdòng|). Nên đối chiếu lại slide thầy.
2. **Regex `\b`**: Word ghi là "kết thúc"; thực chất là *ranh giới từ*.
3. **Box-Cox**: Word có chỗ ghi "dùng được số âm hạn chế", có chỗ ghi "chỉ dùng cho số dương". Thư viện scipy yêu cầu x > 0 → code đang dùng đúng yêu cầu này.
4. **`scipy.linalg.lu`**: Word ghi "trả P.T, L, U"; thực tế trả P, L, U và kiểm tra bằng A = P@L@U (code đã đúng).
5. **Cơ chế thiếu Shipping Cost**: bảng ở ô 5 ghi MCAR nhưng ô 5.3 ghi MNAR → nên thống nhất lại cách gọi trong bài làm.
